# Towards a GIS system

In [ ]:
#| default_exp chunked

In [ ]:
#| export
import numpy as np
import sys
import os
import math
from math import radians, cos, sin, sqrt, atan2
import random

#data
from collections import namedtuple
from dataclasses import dataclass,  field, asdict
import json
from typing import List , Optional
from enum import Enum



In [ ]:
#| export
from dataclasses import dataclass
from typing import Optional, Iterator
from datetime import datetime
import heapq


In [ ]:
#| export
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
from fastlite import *
import fasthtml.components as fc
import httpx


In [ ]:
#| export
from importlib import resources
from pathlib import Path
import tempfile
import shutil
import sqlite3
from dataclasses import dataclass
from typing import Iterator
import numpy as np

In [ ]:
#| export
from HexMagic.geology import Plate

In [ ]:
#| export
from HexMagic.climate import TerrainPatterns, DrainageBasins, Geology, TerraDemo, Terrain, GeoBounds, ClimatePreset, TerrainFactory
from HexMagic.primitives import MapCord, MapSize, MapRect, MapPath, Hex, HexGrid, HexWrapper, HexPosition, hexBackground, HexRegion, unique_windy_edge
from HexMagic.styles import StyleCSS, SVGBuilder, SVGDef

In [ ]:
#| export
from HexMagic.geology import River, Watershed, SoilSystem
from HexMagic.water.river import find_local_minima
# Set up a proper ChunkedTerrainGenerator test
from HexMagic.database import GeoStorageDebugger, ChunkRef, HexWeather
from HexMagic.primitives import HexPosition,  HexChunk, render_chunk



In [ ]:
#| export
from HexMagic.database import LoadResult, SaveResult, HexWeather, WatershedMeta, TerrainWorld, User, GeoStorage, HexData

In [ ]:
#| export
from HexMagic.database import world_to_chunk

In [ ]:
#| export
from HexMagic.voronoi import TerrainGenerator

In [ ]:
SVGBuilder.BUILDERHIDE = False

## Testing to see if we are good

In [ ]:
with GeoStorageDebugger(keep_on_error=False) as dbg:
    #dbg.test_multiple_worlds()
    #dbg.test_multiscale_watersheds()
    #dbg.test_smallWs()
    dbg.test_save_load_terrain()
    dbg.test_query_radius()
    dbg.test_extract_region()
    dbg.test_region_mappers()
    dbg.test_save_load_weather()
    #dbg.test_chunked_terrain_generator()
    dbg.test_save_load_watersheds()
    dbg.test_chunk_spiral_mapping()
    dbg.test_chunk_halo_overlap()

## The refactored mathod

In [ ]:
#| export
class ChunkedTerrainGenerator:
    """Generate and manage large terrains via chunking with plate tracking."""
    
    def __init__(self, storage: 'GeoStorage', 
                 coarse_rings: int = 20,
                 coarse_radius: float = 50,
                 fine_radius: float = 10,
                 chunk_rings: int = 10,
                 halo_rings: int = 3):
        """
        Args:
            storage: GeoStorage for persistence
            coarse_rings: Rings in coarse overview map
            coarse_radius: Hex radius for coarse hexes
            fine_radius: Hex radius for detailed chunks
            chunk_rings: Rings of hexes per chunk (core)
            halo_rings: Extra rings for edge computation
        """
        self.storage = storage
        self.coarse_rings = coarse_rings
        self.coarse_radius = coarse_radius
        self.fine_radius = fine_radius
        self.chunk_rings = chunk_rings
        self.halo_rings = halo_rings
        
        self.coarse_world_id = None
        self._plates = None
        self._coarse_terrain = None
    
    @property
    def scale(self) -> float:
        """Resolution scale factor between coarse and fine."""
        return self.coarse_radius / self.fine_radius


# === COARSE MAP GENERATION ===

@patch
def generate_coarse(self: ChunkedTerrainGenerator, 
                    num_plates: int = 15,
                    ocean_fraction: float = 0.4,
                    oceanic_sides: list = None,
                    name: str = "world_coarse",
                    **kwargs) -> Terrain:
    """Generate coarse map using TerrainGenerator."""
    gen = TerrainGenerator(
        hexDim=self.coarse_rings,
        radius=self.coarse_radius,
        num_plates=num_plates,
        ocean_fraction=ocean_fraction,
        oceanic_sides=oceanic_sides or [],
        subdivisions=2,
        **kwargs
    )
    
    terrain, plates = gen.generate()
    self._plates = plates
    self._coarse_terrain = terrain
    
    result = self.storage.save_world(terrain, name=name)
    self.coarse_world_id = result.id
    
    return terrain


# === CHUNK GENERATION ===

@patch
def generate_chunk(self: ChunkedTerrainGenerator, chunk_ref: ChunkRef) -> HexChunk:
    """Generate chunk by sampling from coarse with plate inheritance."""
    chunk_plates = self._query_chunk_plates(chunk_ref)
    
    chunk = HexChunk.with_halo(
        position=chunk_ref.position,
        core_rings=self.chunk_rings,
        halo_rings=self.halo_rings
    )
    chunk.add_field('plate_id', default=-1)
    
    coarse_lookup = self._build_coarse_lookup(chunk_plates)
    self._populate_from_coarse(chunk, coarse_lookup)
    self._interpolate_fields_to_chunk(chunk, coarse_lookup)
    self._add_detail_to_chunk(chunk)
    
    return chunk




@patch
def _build_coarse_lookup(self: ChunkedTerrainGenerator, chunk_plates: dict) -> dict:
    """Convert chunk_plates query result to lookup dict."""
    coarse_lookup = {}
    for pid, rows in chunk_plates.items():
        for row in rows:
            key = (row['q'], row['r'], row['s'])
            coarse_lookup[key] = {
                'elevation': row['elevation'],
                'plate_id': pid,
            }
    return coarse_lookup


@patch
def _populate_from_coarse(self: ChunkedTerrainGenerator, 
                          chunk: HexChunk, 
                          coarse_lookup: dict):
    """Fill chunk elevations and plate_id by interpolating from coarse data."""
    for idx, local_pos, world_pos in chunk.iter_with_world():
        elevation, plate_id = self._interpolate_elevation(world_pos, coarse_lookup)
        chunk.elevations[idx] = elevation
        chunk.fields['plate_id'][idx] = int(plate_id)


@patch
def _interpolate_elevation(self: ChunkedTerrainGenerator, 
                           world_pos: HexPosition, 
                           coarse_lookup: dict) -> tuple[float, int]:
    """Interpolate elevation using distance-weighted average of nearby coarse hexes."""
    scale = self.scale
    
    # Fractional coarse coordinates
    frac_q = world_pos.q / scale
    frac_r = world_pos.r / scale
    frac_s = -frac_q - frac_r
    
    # Nearest coarse hex center
    near_q = int(round(frac_q))
    near_r = int(round(frac_r))
    near_s = -near_q - near_r
    center = HexPosition(near_q, near_r, near_s)
    
    # Sample center + ring-1 neighbors (7 hexes)
    candidates = [center] + list(center.ring(1))
    
    weights = []
    elevations = []
    plate_weights = {}
    
    for cpos in candidates:
        key = (cpos.q, cpos.r, cpos.s)
        if key not in coarse_lookup:
            continue
        
        dq = cpos.q - frac_q
        dr = cpos.r - frac_r
        ds = cpos.s - frac_s
        dist = (abs(dq) + abs(dr) + abs(ds)) / 2
        
        w = 1.0 / (dist + 0.1)
        weights.append(w)
        elevations.append(coarse_lookup[key]['elevation'])
        
        pid = coarse_lookup[key]['plate_id']
        plate_weights[pid] = plate_weights.get(pid, 0) + w
    
    if not weights:
        return 0.0, -1
    
    total_weight = sum(weights)
    interp_elev = sum(w * e for w, e in zip(weights, elevations)) / total_weight
    best_plate = max(plate_weights, key=plate_weights.get)
    
    return interp_elev, best_plate


@patch
def _add_detail_to_chunk(self: ChunkedTerrainGenerator, chunk: HexChunk):
    """Add deterministic noise seeded by world position."""
    coarse = self._coarse_terrain
    noise_scale = coarse.elevationDelta * 0.1
    
    for idx, local_pos, world_pos in chunk.iter_with_world():
        if chunk.elevations[idx] <= 0:
            chunk.elevations[idx] = min(chunk.elevations[idx], -50)
            continue
        
        seed = hash((world_pos.q, world_pos.r, world_pos.s)) % (2**31)
        np.random.seed(seed)
        chunk.elevations[idx] += np.random.uniform(-noise_scale, noise_scale)


# === CHUNK COORDINATE UTILITIES ===

@patch
def chunk_refs(self: ChunkedTerrainGenerator, max_ring: int) -> Iterator[ChunkRef]:
    """Yield ChunkRefs in spiral order from center outward."""
    for ring in range(max_ring + 1):
        for pos in HexPosition.origin().ring(ring) if ring > 0 else [HexPosition.origin()]:
            yield ChunkRef(pos, self.chunk_rings)


@patch
def world_to_chunk_ref(self: ChunkedTerrainGenerator, world_pos: HexPosition) -> ChunkRef:
    """Convert world position to its containing ChunkRef."""
    chunk_pos, local_pos = world_to_chunk(world_pos, self.chunk_rings)
    return ChunkRef(chunk_pos, self.chunk_rings)


@patch
def _chunks_intersecting_rect(self: ChunkedTerrainGenerator, 
                               min_q: int, max_q: int, 
                               min_r: int, max_r: int) -> list[ChunkRef]:
    """Find all chunks that might contain hexes in the given rect bounds."""
    spacing = 2 * self.chunk_rings
    
    chunk_min_q = (min_q // spacing) - 1
    chunk_max_q = (max_q // spacing) + 1
    chunk_min_r = (min_r // spacing) - 1
    chunk_max_r = (max_r // spacing) + 1
    
    chunks = []
    for cq in range(chunk_min_q, chunk_max_q + 1):
        for cr in range(chunk_min_r, chunk_max_r + 1):
            cs = -cq - cr
            chunk_pos = HexPosition(cq, cr, cs)
            chunks.append(ChunkRef(chunk_pos, self.chunk_rings))
    
    return chunks


# === OUTPUT: CHUNKS TO TERRAIN ===

@patch
def rect_region_from_chunks(self: ChunkedTerrainGenerator,
                            min_q: int, max_q: int,
                            min_r: int, max_r: int,
                            radius: float = None,
                            fill_from_coarse: bool = True) -> Terrain:
    """Extract rectangular region from chunks."""
    radius = radius or self.fine_radius
    coarse = self._coarse_terrain
    
    chunk_refs = self._chunks_intersecting_rect(min_q, max_q, min_r, max_r)
    chunks = [self.generate_chunk(ref) for ref in chunk_refs]
    
    # Build world position -> data lookup from all chunks
    world_data = {}
    for chunk in chunks:
        for idx in chunk.iter_core():
            world_pos = chunk.index_to_world(idx)
            plate_id = chunk.fields.get('plate_id', np.array([-1]))[idx]
            world_data[world_pos] = (chunk.elevations[idx], plate_id)
    
    # Create rectangular grid
    nrows = max_r - min_r + 1
    ncols = max_q - min_q + 1
    grid = HexGrid(nRows=nrows, nCols=ncols, radius=radius, style=coarse.seaLevel)
    
    # Build terrain
    terrain = Terrain.__new__(Terrain)
    terrain.hexGrid = grid
    terrain.elevations = np.zeros(len(grid.hexes))
    terrain.fields = {'plate_id': np.full(len(grid.hexes), -1, dtype=np.int32)}
    terrain.seaLevel = coarse.seaLevel
    terrain.elevationDelta = coarse.elevationDelta
    terrain.colorLevels = coarse.colorLevels
    terrain.climate = coarse.climate
    terrain.geo = coarse.geo
    terrain.path = ""
    
    origin_world = HexPosition(min_q, min_r, -min_q - min_r)
    
    for grid_idx in range(len(grid.hexes)):
        local_pos = grid.index_to_hexposition(grid_idx, origin_index=0)
        world_pos = origin_world + local_pos
        
        if world_pos in world_data:
            elev, plate_id = world_data[world_pos]
            terrain.elevations[grid_idx] = elev
            terrain.fields['plate_id'][grid_idx] = plate_id
            
        elif fill_from_coarse:
            coarse_q = int(round(world_pos.q / self.scale))
            coarse_r = int(round(world_pos.r / self.scale))
            coarse_s = -coarse_q - coarse_r
            coarse_pos = HexPosition(coarse_q, coarse_r, coarse_s)
            coarse_idx = coarse.hexGrid.hexposition_to_index(
                coarse_pos, origin_index=coarse.hexGrid.midpoint
            )
            
            if 0 <= coarse_idx < len(coarse.elevations):
                terrain.elevations[grid_idx] = coarse.elevations[coarse_idx]
                if 'plate_id' in coarse.fields:
                    terrain.fields['plate_id'][grid_idx] = coarse.fields['plate_id'][coarse_idx]
    
    # Apply coloring
    for style in terrain.colorLevels:
        grid.builder.add_style(style)
    grid.builder.add_style(terrain.seaLevel)
    terrain.colorMap()
    
    return terrain


@patch
def chunks_to_terrain(self: ChunkedTerrainGenerator, 
                      chunks: list[HexChunk],
                      radius: float = None) -> Terrain:
    """Merge multiple HexChunks into a single Terrain for rendering."""
    if not chunks:
        return None
    
    radius = radius or self.fine_radius
    coarse = self._coarse_terrain
    
    # Find world bounds across all chunks (core only)
    all_world_positions = []
    for chunk in chunks:
        for idx in chunk.iter_core():
            all_world_positions.append(chunk.index_to_world(idx))
    
    qs = [p.q for p in all_world_positions]
    rs = [p.r for p in all_world_positions]
    
    q_range = max(qs) - min(qs)
    r_range = max(rs) - min(rs)
    rings = max(q_range, r_range) // 2 + 2
    
    center_q = (max(qs) + min(qs)) // 2
    center_r = (max(rs) + min(rs)) // 2
    center_s = -center_q - center_r
    center_world = HexPosition(center_q, center_r, center_s)
    
    grid = HexGrid.centered(rings=rings, radius=radius, style=coarse.seaLevel)
    
    terrain = Terrain.__new__(Terrain)
    terrain.hexGrid = grid
    terrain.elevations = np.zeros(len(grid.hexes))
    terrain.fields = {'plate_id': np.full(len(grid.hexes), -1, dtype=np.int32)}
    terrain.seaLevel = coarse.seaLevel
    terrain.elevationDelta = coarse.elevationDelta
    terrain.colorLevels = coarse.colorLevels
    terrain.climate = coarse.climate
    terrain.geo = coarse.geo
    terrain.path = ""
    
    for chunk in chunks:
        for idx in chunk.iter_core():
            world_pos = chunk.index_to_world(idx)
            local_pos = world_pos - center_world
            
            grid_idx = grid.hexposition_to_index(local_pos, origin_index=grid.midpoint)
            if 0 <= grid_idx < len(terrain.elevations):
                terrain.elevations[grid_idx] = chunk.elevations[idx]
                if 'plate_id' in chunk.fields:
                    terrain.fields['plate_id'][grid_idx] = chunk.fields['plate_id'][idx]
    
    for style in terrain.colorLevels:
        grid.builder.add_style(style)
    grid.builder.add_style(terrain.seaLevel)
    terrain.colorMap()
    
    return terrain



@patch
def _query_chunk_plates_old(self: ChunkedTerrainGenerator, chunk: ChunkRef) -> dict:
    """Query DB for plate membership in chunk region."""
    center = chunk.center_hex
    radius = int((self.chunk_rings + self.halo_rings) / self.scale) + 2
    
    coarse_q = int(center.q / self.scale)
    coarse_r = int(center.r / self.scale)
    coarse_s = -coarse_q - coarse_r
    
    rows = self.storage.query_hexes_in_radius(
        self.coarse_world_id,
        coarse_q, coarse_r, coarse_s,
        radius
    )
    
    plates = {}
    for row in rows:
        pid = row.get('plate_id', 0)
        if pid not in plates:
            plates[pid] = []
        plates[pid].append(row)
    
    return plates

In [ ]:
@patch
def _query_chunk_plates(self: ChunkedTerrainGenerator, chunk: ChunkRef) -> dict:
    """Query DB for plate membership in chunk region."""
    center = chunk.center_hex
    radius = int((self.chunk_rings + self.halo_rings) / self.scale) + 2
    
    coarse_q = int(center.q / self.scale)
    coarse_r = int(center.r / self.scale)
    coarse_s = -coarse_q - coarse_r
    
    rows = self.storage.query_hexes_in_radius(
        self.coarse_world_id,
        coarse_q, coarse_r, coarse_s,
        radius
    )
    
    plates = {}
    for row in rows:
        # Fix: handle None from DB
        pid = row.get('plate_id')
        if pid is None:
            pid = 0
        if pid not in plates:
            plates[pid] = []
        plates[pid].append(row)
    
    return plates


In [ ]:
@patch
def _interpolate_elevation(self: ChunkedTerrainGenerator, 
                           world_pos: HexPosition, 
                           coarse_lookup: dict,
                           sigma: float = 0.8) -> tuple[float, int]:
    """Gaussian-weighted interpolation for smooth results.
    Key changes:

    19 samples (rings 0-2) instead of 7
    Gaussian falloff exp(-d²/2σ²) instead of 1/(d + 0.1)
    Hex-aware distance dq² + dr² + dq·dr accounts for hex geometry

The sigma parameter controls smoothness - larger = smoother but blurrier. Start with 0.8 and adjust if terrain looks too smooth or still jagged.
    
    """
    scale = self.scale
    
    # Fractional coarse coordinates
    frac_q = world_pos.q / scale
    frac_r = world_pos.r / scale
    
    # Nearest coarse hex center
    near_q = int(round(frac_q))
    near_r = int(round(frac_r))
    near_s = -near_q - near_r
    center = HexPosition(near_q, near_r, near_s)
    
    # Sample center + ring 1 + ring 2 (19 hexes for smoother blending)
    candidates = [center] + list(center.ring(1)) + list(center.ring(2))
    
    weights = []
    elevations = []
    plate_weights = {}
    
    for cpos in candidates:
        key = (cpos.q, cpos.r, cpos.s)
        if key not in coarse_lookup:
            continue
        
        # Squared distance in fractional coarse space
        dq = cpos.q - frac_q
        dr = cpos.r - frac_r
        dist_sq = dq**2 + dr**2 + dq*dr  # Hex metric approximation
        
        # Gaussian falloff
        w = math.exp(-dist_sq / (2 * sigma**2))
        
        weights.append(w)
        elevations.append(coarse_lookup[key]['elevation'])
        
        pid = coarse_lookup[key]['plate_id']
        plate_weights[pid] = plate_weights.get(pid, 0) + w
    
    if not weights:
        return 0.0, -1
    
    total_weight = sum(weights)
    interp_elev = sum(w * e for w, e in zip(weights, elevations)) / total_weight
    best_plate = max(plate_weights, key=plate_weights.get)
    
    return interp_elev, best_plate


In [ ]:
@patch
def _interpolate_fields_to_chunk(self: ChunkedTerrainGenerator, 
                                  chunk: HexChunk, 
                                  coarse_lookup: dict):
    """Interpolate additional fields to chunk (placeholder for future fields)."""
    pass  # plate_id already handled in _populate_from_coarse


## ChunkSet

In [ ]:
#| export

class ChunkSet:
    """Queryable view over multiple HexChunks for algorithm execution."""
    
    def __init__(self, chunks: list[HexChunk], coarse: Terrain, scale: float):
        self.chunks = chunks
        self.coarse = coarse
        self.scale = scale
        
        # Build lookup: world_pos -> (chunk, local_idx, is_core)
        self._build_index()
    
    def _build_index(self):
        """Index all positions from all chunks (core + halo)."""
        self.pos_to_chunk = {}  # HexPosition -> (chunk, idx, is_core)
        
        for chunk in self.chunks:
            for idx in chunk.iter_all():  # Core AND halo
                world_pos = chunk.index_to_world(idx)
                is_core = chunk.is_core(idx)
                
                # Prefer core over halo if multiple chunks have it
                if world_pos not in self.pos_to_chunk or is_core:
                    self.pos_to_chunk[world_pos] = (chunk, idx, is_core)
    
    def elevation_at(self, world_pos: HexPosition) -> float:
        """Get elevation, falling back to coarse if not in any chunk."""
        if world_pos in self.pos_to_chunk:
            chunk, idx, _ = self.pos_to_chunk[world_pos]
            return chunk.elevations[idx]
        
        # Fallback to coarse
        return self._query_coarse(world_pos)
    
    def neighbors_of(self, world_pos: HexPosition) -> list[HexPosition]:
        """Get neighbors that exist in chunk set or coarse."""
        return [world_pos.neighbor(i) for i in range(6)]

    
    def lowest_neighbor(self, world_pos: HexPosition) -> HexPosition:
        """Find lowest neighbor for drainage algorithms."""
        neighbors = self.neighbors_of(world_pos)
        elevs = [(n, self.elevation_at(n)) for n in neighbors]
        return min(elevs, key=lambda x: x[1])[0]
    
    def iter_positions(self) -> Iterator[HexPosition]:
        """Iterate all covered positions (cores only for modification)."""
        for pos, (chunk, idx, is_core) in self.pos_to_chunk.items():
            if is_core:
                yield pos
    
    def set_elevation(self, world_pos: HexPosition, value: float):
        """Set elevation (only works for core hexes)."""
        if world_pos in self.pos_to_chunk:
            chunk, idx, is_core = self.pos_to_chunk[world_pos]
            if is_core:
                chunk.elevations[idx] = value


In [ ]:
@patch
def _query_coarse(self: ChunkSet, world_pos: HexPosition) -> float:
    """Get elevation from coarse map for positions outside chunks."""
    coarse_q = int(round(world_pos.q / self.scale))
    coarse_r = int(round(world_pos.r / self.scale))
    coarse_s = -coarse_q - coarse_r
    coarse_pos = HexPosition(coarse_q, coarse_r, coarse_s)
    
    coarse_idx = self.coarse.hexGrid.hexposition_to_index(
        coarse_pos, origin_index=self.coarse.hexGrid.midpoint
    )
    
    if 0 <= coarse_idx < len(self.coarse.elevations):
        return self.coarse.elevations[coarse_idx]
    return 0.0


In [ ]:
#| export
@patch
def to_terrain(self: ChunkSet, fill_gaps: bool = True) -> Terrain:
    """Convert ChunkSet to Terrain for visualization or algorithms."""
    
    # Find bounds from all indexed positions
    positions = list(self.pos_to_chunk.keys())
    qs = [p.q for p in positions]
    rs = [p.r for p in positions]
    min_q, max_q = min(qs), max(qs)
    min_r, max_r = min(rs), max(rs)
    
    # Build grid
    nrows = max_r - min_r + 1
    ncols = max_q - min_q + 1
    coarse = self.coarse
    grid = HexGrid(nRows=nrows, nCols=ncols, radius=coarse.hexGrid.radius / self.scale, 
                   style=coarse.seaLevel)
    
    # Create terrain
    terrain = Terrain.__new__(Terrain)
    terrain.hexGrid = grid
    terrain.elevations = np.zeros(len(grid.hexes))
    terrain.fields = {}
    terrain.seaLevel = coarse.seaLevel
    terrain.elevationDelta = coarse.elevationDelta
    terrain.colorLevels = coarse.colorLevels
    terrain.climate = coarse.climate
    terrain.geo = coarse.geo
    terrain.path = ""
    
    origin_world = HexPosition(min_q, min_r, -min_q - min_r)
    valid_indices = set()
    
    for grid_idx in range(len(grid.hexes)):
        local_pos = grid.index_to_hexposition(grid_idx, origin_index=0)
        world_pos = origin_world + local_pos
        
        if world_pos in self.pos_to_chunk:
            chunk, idx, is_core = self.pos_to_chunk[world_pos]
            terrain.elevations[grid_idx] = chunk.elevations[idx]
            if is_core:
                valid_indices.add(grid_idx)
        elif fill_gaps:
            terrain.elevations[grid_idx] = self._query_coarse(world_pos)
            # Gap-filled hexes are valid but marked differently if needed
            valid_indices.add(grid_idx)
    
    grid.invalid_region = set(range(len(grid.hexes))) - valid_indices
    
    return terrain


In [ ]:
@patch
def coverage_at(self: ChunkSet, world_pos: HexPosition) -> list[tuple[HexChunk, int, bool]]:
    """Get all chunks covering this position (not just the preferred one)."""
    result = []
    for chunk in self.chunks:
        idx = chunk.world_to_index(world_pos)
        if idx >= 0:
            is_core = chunk.is_core(idx)
            result.append((chunk, idx, is_core))
    return result


## The big refactor

So my thoughts on hex tilting
right now with hexpostions we have the notion of rings and centers. a hextile of
 size 1 has 1 outer
 size of 2 has 6 (the six directions)
 size of 3 has 12 (the six directions and the six directions of the second ring) 
 we can think of these shapes as having corners which are the ring * a pure direction and then an interpolation between corners. 

In [ ]:
from HexMagic.primitives import MapCord, MapSize, MapRect, MapPath, Hex, HexGrid, HexWrapper, HexPosition, hexBackground, HexRegion, unique_windy_edge, PrimitiveDemo

In [ ]:


sampleGrid = PrimitiveDemo().sampleGrid(fill="yellow",hexDim = 5,makeLabels = True)
sampleGrid.update()
SVGBuilder.BUILDERHIDE = True
sampleGrid.builder.show()



Another pass:
the question becomes how do we convert these into a rect. if we already know the rect by these indexes then we just need to intersect those that fit. if the rect is always at least one radius of our pattern away from the edge then we just need to find the hexchunks that have any member and then just crop them back. The find more than we need and then throw out the hexes is the right way to build these terrains


### Mind the gap

In [ ]:
def find_coastal_chunks(gen, coarse, max_ring=2):
    """Find chunks that contain both land and ocean."""
    all_chunks = list(gen.chunk_refs(max_ring=max_ring))
    coastal = []
    
    for i, chunk_ref in enumerate(all_chunks):
        center = chunk_ref.center_hex
        coarse_q = int(round(center.q / gen.scale))
        coarse_r = int(round(center.r / gen.scale))
        coarse_s = -coarse_q - coarse_r
        
        has_land = has_ocean = False
        for ring in range(3):
            positions = [HexPosition(coarse_q, coarse_r, coarse_s)] if ring == 0 else \
                        HexPosition(coarse_q, coarse_r, coarse_s).ring(ring)
            for pos in positions:
                idx = coarse.hexGrid.hexposition_to_index(pos, origin_index=coarse.hexGrid.midpoint)
                if 0 <= idx < len(coarse.elevations):
                    if coarse.elevations[idx] > 0: has_land = True
                    else: has_ocean = True
        
        if has_land and has_ocean:
            coastal.append((i, chunk_ref))
    
    return all_chunks, coastal


def render_global_map(gen, coarse, all_chunks, coastal_chunks):
    """Render coarse map with chunk locations marked."""
    coarse.hexGrid.adjustRadius(12)
    coarse.colorMap()
    coarse.hexGrid.update()
    
    textStyle = StyleCSS("textStyle", fill="white", font_size="1.5em", 
                         filter="drop-shadow(3px 3px 2px rgba(0,0,0,.7))")
    coarse.hexGrid.builder.add_style(textStyle)
    
    chunk_overlay = ""
    coastal_indices = {ci for ci, _ in coastal_chunks}
    
    for i, chunk_ref in enumerate(all_chunks):
        center = chunk_ref.center_hex
        coarse_q = int(round(center.q / gen.scale))
        coarse_r = int(round(center.r / gen.scale))
        coarse_s = -coarse_q - coarse_r
        coarse_pos = HexPosition(coarse_q, coarse_r, coarse_s)
        coarse_idx = coarse.hexGrid.hexposition_to_index(coarse_pos, origin_index=coarse.hexGrid.midpoint)
        
        if coarse_idx >= 0:
            hex_obj = coarse.hexGrid.hexes[coarse_idx]
            is_coastal = i in coastal_indices
            color = "#00ff00" if is_coastal else "#ff0000"
            fill = "rgba(0,255,0,0.2)" if is_coastal else "none"
            chunk_overlay += f'<circle cx="{hex_obj.center.x}" cy="{hex_obj.center.y}" r="10" fill="{fill}" stroke="{color}" stroke-width="2"/>\n'
            chunk_overlay += f'<text x="{hex_obj.center.x}" y="{hex_obj.center.y + 4}" text-anchor="middle" fill="white" font-size="8" font-weight="bold">{i}</text>\n'
    
    coarse.hexGrid.builder.adjust("chunks", chunk_overlay)
    coarse.hexGrid.builder.add_centered_text("Global View - Coastal chunks in green", 
        y_offset=-coarse.hexGrid.builder.height/2 + 30, class_name=textStyle.name)
    
    return coarse.hexGrid.builder


In [ ]:
def render_detail_with_overlaps(gen, coarse, selected_chunks):
    """Render detailed view showing cores and halo overlaps using ChunkSet."""
    
    # Generate chunks
    chunks = []
    chunk_indices = {}
    for global_idx, chunk_ref in selected_chunks:
        chunk = gen.generate_chunk(chunk_ref)
        chunks.append(chunk)
        chunk_indices[id(chunk)] = global_idx
    
    # Create ChunkSet
    chunk_set = ChunkSet(chunks, coarse, gen.scale)
    
    # Convert to terrain for rendering
    detail = chunk_set.to_terrain(fill_gaps=True)
    detail.hexGrid.adjustRadius(20)
    detail.colorMap()
    detail.hexGrid.update()
    
    # Build core/halo tracking
    positions = list(chunk_set.pos_to_chunk.keys())
    min_q = min(p.q for p in positions)
    min_r = min(p.r for p in positions)
    origin_world = HexPosition(min_q, min_r, -min_q - min_r)
    
    chunk_cores = {chunk_indices[id(c)]: set() for c in chunks}
    overlap_indices = set()
    
    for world_pos in positions:
        local_pos = world_pos - origin_world
        grid_idx = detail.hexGrid.hexposition_to_index(local_pos, origin_index=0)
        if grid_idx < 0 or grid_idx >= len(detail.hexGrid.hexes):
            continue
        
        coverage = chunk_set.coverage_at(world_pos)
        cores_here = [(c, idx) for c, idx, is_core in coverage if is_core]
        halos_here = [(c, idx) for c, idx, is_core in coverage if not is_core]
        
        for c, _ in cores_here:
            chunk_cores[chunk_indices[id(c)]].add(grid_idx)
        
        if len(halos_here) >= 2 and len(cores_here) == 0:
            overlap_indices.add(grid_idx)
    
    all_core_indices = set().union(*chunk_cores.values())
    overlap_in_halo_only = overlap_indices - all_core_indices
    
    # Styles for chunks
    chunk_colors = ["#e74c3c", "#3498db", "#2ecc71", "#f39c12", "#9b59b6"]
    chunk_styles = [
        StyleCSS(f"chunk_{i}", fill="none", stroke=chunk_colors[i % len(chunk_colors)], 
                 stroke_width=3, stroke_dasharray="8,4")
        for i in range(len(chunks))
    ]
    
    textStyle = StyleCSS("textStyle", fill="white", font_size="1.5em", 
                         filter="drop-shadow(3px 3px 2px rgba(0,0,0,.7))")
    detail.hexGrid.builder.add_style(textStyle)
    
    region_overlay = ""
    
    # Draw chunk boundaries using HexRegion
    for i, (global_idx, chunk_ref) in enumerate(selected_chunks):
        style = chunk_styles[i]
        detail.hexGrid.builder.add_style(style)
        
        core_indices = chunk_cores[global_idx]
        if core_indices:
            region = HexRegion(hexes=core_indices, hexGrid=detail.hexGrid)
            paths = region.trace_perimeter(style=style)
            for path in paths:
                curved = path.make_windy(iterations=1, offset_factor=0.15)
                region_overlay += curved.svg()
    
    # Mark overlaps with dots
    dot_style = StyleCSS("overlap_dot", fill="#ff00ff", stroke="white", stroke_width=1)
    detail.hexGrid.builder.add_style(dot_style)
    
    for grid_idx in overlap_in_halo_only:
        hex_obj = detail.hexGrid.hexes[grid_idx]
        region_overlay += f'<circle cx="{hex_obj.center.x}" cy="{hex_obj.center.y}" r="5" class="overlap_dot"/>\n'
    
    detail.hexGrid.builder.adjust("regions", region_overlay)
    detail.hexGrid.builder.add_centered_text(
        f"Detail View - {len(overlap_in_halo_only)} halo overlaps", 
        y_offset=-detail.hexGrid.builder.height/2 + 30, 
        class_name=textStyle.name
    )
    
    return detail.hexGrid.builder, chunks


In [ ]:


# === MAIN DEMO ===
with GeoStorageDebugger(keep_on_error=False) as debugger:
    gen = ChunkedTerrainGenerator(
        storage=debugger.server,
        coarse_rings=15,
        coarse_radius=50,
        fine_radius=10,
        chunk_rings=8,
        halo_rings=3
    )
    
    coarse = gen.generate_coarse(
        num_plates=8,
        ocean_fraction=0.35,
        oceanic_sides=['W'],
        name="zoom_demo",
        seed=42
    )
    
    # Find chunks
    all_chunks, coastal_chunks = find_coastal_chunks(gen, coarse)
    print(f"Found {len(coastal_chunks)} coastal chunks out of {len(all_chunks)} total")
    
    # 1. Global map
    print("\n=== GLOBAL MAP ===")
    global_builder = render_global_map(gen, coarse, all_chunks, coastal_chunks)
    display(global_builder.show())
    
    # 2. Detail view with first 3 coastal chunks
    selected = coastal_chunks[:3]
    print(f"\n=== DETAIL VIEW (chunks {[i for i, _ in selected]}) ===")
    detail_builder, chunks_data = render_detail_with_overlaps(gen, coarse, selected)
    display(detail_builder.show())


## Chunk Set drawing

In [ ]:
with GeoStorageDebugger(keep_on_error=False) as debugger:
    gen = ChunkedTerrainGenerator(
        storage=debugger.server,
        coarse_rings=15,
        coarse_radius=50,
        fine_radius=10,
        chunk_rings=8,
        halo_rings=3
    )
    
    coarse = gen.generate_coarse(num_plates=8, ocean_fraction=0.35, oceanic_sides=['W'], seed=42)
    
    # Generate some chunks
    all_chunks, coastal_chunks = find_coastal_chunks(gen, coarse)
    selected = coastal_chunks[:3]
    
    chunks = [gen.generate_chunk(ref) for _, ref in selected]
    
    # Create ChunkSet
    chunk_set = ChunkSet(chunks, coarse, gen.scale)
    
    print(f"ChunkSet covers {len(list(chunk_set.iter_positions()))} core positions")
    print(f"Total indexed positions (core+halo): {len(chunk_set.pos_to_chunk)}")
    
    # Convert to terrain for visualization
    terrain = chunk_set.to_terrain(fill_gaps=True)
    terrain.colorMap()
    terrain.hexGrid.update()
    display(terrain.hexGrid.builder.show())
    
    # Test neighbor queries
    sample_pos = next(chunk_set.iter_positions())
    print(f"\nSample position: {sample_pos}")
    print(f"Elevation: {chunk_set.elevation_at(sample_pos):.1f}")
    print(f"Lowest neighbor: {chunk_set.lowest_neighbor(sample_pos)}")


In [ ]:
with GeoStorageDebugger(keep_on_error=False) as debugger:
    storage = debugger.server
    
    gen = ChunkedTerrainGenerator(
        storage=storage,
        coarse_rings=15,
        coarse_radius=50,
        fine_radius=10,
        chunk_rings=8,
        halo_rings=3
    )
    
    coarse = gen.generate_coarse(
        num_plates=8,
        ocean_fraction=0.35,
        oceanic_sides=['W'],
        name="zoom_demo",
        seed=42
    )
    
    # === PART 1: Global view with all chunks marked ===
    coarse.hexGrid.adjustRadius(12)
    coarse.colorMap()
    coarse.hexGrid.update()
    
    textStyle = StyleCSS("textStyle", fill="white", font_size="1.5em", 
                         filter="drop-shadow(3px 3px 2px rgba(0,0,0,.7))")
    coarse.hexGrid.builder.add_style(textStyle)
    
    # Find all chunks and coastal chunks
    all_chunks, coastal_chunks = find_coastal_chunks(gen, coarse)
    print(f"Found {len(coastal_chunks)} coastal chunks")
    
    # Mark chunks on global view
    global_builder = render_global_map(gen, coarse, all_chunks, coastal_chunks)
    print("=== GLOBAL MAP ===")
    display(global_builder.show())
    
    # === PART 2: Zoom into coastal region using ChunkSet ===
    if coastal_chunks:
        selected = coastal_chunks[:min(3, len(coastal_chunks))]
        
        # Generate chunks and track their global indices
        chunks = [gen.generate_chunk(ref) for _, ref in selected]
        chunk_indices = {id(c): gi for (gi, ref), c in zip(selected, chunks)}
        
        # Create ChunkSet - handles all position indexing
        chunk_set = ChunkSet(chunks, coarse, gen.scale)
        
        # Convert to terrain - bounds computed automatically
        detail = chunk_set.to_terrain(fill_gaps=True)
        detail.hexGrid.adjustRadius(10)
        detail.hexGrid.update()
        
        # Compute origin from ChunkSet's indexed positions
        positions = list(chunk_set.pos_to_chunk.keys())
        min_q = min(p.q for p in positions)
        min_r = min(p.r for p in positions)
        origin_world = HexPosition(min_q, min_r, -min_q - min_r)
        
        # Group grid indices by chunk (core hexes only)
        chunk_grid_indices = {gi: set() for gi in chunk_indices.values()}
        for world_pos, (chunk, idx, is_core) in chunk_set.pos_to_chunk.items():
            if is_core:
                local_pos = world_pos - origin_world
                grid_idx = detail.hexGrid.hexposition_to_index(local_pos, origin_index=0)
                if 0 <= grid_idx < len(detail.hexGrid.hexes):
                    gi = chunk_indices[id(chunk)]
                    chunk_grid_indices[gi].add(grid_idx)
        
        # Use seaborn palette for chunk colors
        chunk_styles = StyleCSS.seaborn("Set2", len(chunks))
        for style in chunk_styles:
            style.properties['opacity'] = '0.4'
            style.properties['stroke_width'] = '3'
            detail.hexGrid.builder.add_style(style)
        
        # Create chunk region overlays
        region_overlay = ""
        legend_styles = []
        
        for i, (global_idx, _) in enumerate(selected):
            grid_indices = chunk_grid_indices[global_idx]
            
            if grid_indices:
                region = HexRegion(hexes=grid_indices, hexGrid=detail.hexGrid)
                
                base_style = chunk_styles[i]
                region_style = StyleCSS(
                    f"Chunk_{global_idx}",
                    fill=base_style.properties['fill'],
                    stroke=base_style.properties['fill'],
                    stroke_width=3,
                    opacity=0.35
                )
                detail.hexGrid.builder.add_style(region_style)
                legend_styles.append(region_style.copy(f"Chunk_{global_idx}"))
                
                region_overlay += region.draw(style=region_style, f=unique_windy_edge(), inset=0.95)
                
                # Add chunk label at centroid
                centroid_idx = region.centroid_hex()
                if centroid_idx and 0 <= centroid_idx < len(detail.hexGrid.hexes):
                    cx = detail.hexGrid.hexes[centroid_idx].center.x
                    cy = detail.hexGrid.hexes[centroid_idx].center.y
                    region_overlay += f'<text x="{cx}" y="{cy}" text-anchor="middle" fill="white" font-size="14" font-weight="bold" style="filter: drop-shadow(2px 2px 1px rgba(0,0,0,.8))">Chunk {global_idx}</text>\n'
        
        detail.hexGrid.builder.adjust("chunk_regions", region_overlay)
        
        # Add legend
        keyLabelStyle = StyleCSS("keyLabel", fill="#333", font_size="12px")
        detail.hexGrid.builder.add_style(keyLabelStyle)
        
        for style in legend_styles:
            style.properties['opacity'] = '1.0'
            detail.hexGrid.builder.add_style(style)
        
        legend_svg = detail.hexGrid.builder.legendBelow(legend_styles, gap=15, bg_fill="rgba(255,255,255,0.9)")
        detail.hexGrid.builder.adjust("legend", legend_svg)
        
        detail.hexGrid.builder.add_style(textStyle)
        detail.hexGrid.builder.add_centered_text("Coastal Detail - Chunk boundaries shown", 
            y_offset=-detail.hexGrid.builder.height/2 + 50, class_name=textStyle.name)
        
        print(f"\n=== ZOOMED COASTAL DETAIL ===")
        print(f"Selected chunks: {[gi for gi, _ in selected]}")
        print(f"ChunkSet: {len(list(chunk_set.iter_positions()))} core, {len(chunk_set.pos_to_chunk)} total positions")
        print(f"Detail grid: {detail.hexGrid.nRows}x{detail.hexGrid.nCols} = {len(detail.hexGrid.hexes)} hexes")
        display(detail.hexGrid.builder.show())


In [ ]:
with GeoStorageDebugger(keep_on_error=False) as debugger:
    storage = debugger.server
    
    gen = ChunkedTerrainGenerator(
        storage=storage,
        coarse_rings=15,
        coarse_radius=50,
        fine_radius=10,
        chunk_rings=8,
        halo_rings=3
    )
    
    coarse = gen.generate_coarse(
        num_plates=8,
        ocean_fraction=0.35,
        oceanic_sides=['W'],
        name="zoom_demo",
        seed=42
    )
    
    # Find coastal chunks
    all_chunks, coastal_chunks = find_coastal_chunks(gen, coarse)
    selected = coastal_chunks[:3]
    
    # Generate chunks and create ChunkSet
    chunks = [gen.generate_chunk(ref) for _, ref in selected]
    chunk_indices = {id(c): gi for (gi, _), c in zip(selected, chunks)}
    
    chunk_set = ChunkSet(chunks, coarse, gen.scale)
    
    # Convert to terrain - ChunkSet handles bounds automatically
    detail = chunk_set.to_terrain(fill_gaps=True)
    detail.colorMap()  # <-- Add this!
    detail.hexGrid.adjustRadius(10)
    detail.hexGrid.update()
    
    # Compute origin from ChunkSet positions
    positions = list(chunk_set.pos_to_chunk.keys())
    min_q = min(p.q for p in positions)
    min_r = min(p.r for p in positions)
    origin_world = HexPosition(min_q, min_r, -min_q - min_r)
    
    # Group positions by chunk and core/halo status using coverage_at
    chunk_cores = {gi: set() for gi in chunk_indices.values()}
    chunk_halos = {gi: set() for gi in chunk_indices.values()}
    
    for world_pos in positions:
        local_pos = world_pos - origin_world
        grid_idx = detail.hexGrid.hexposition_to_index(local_pos, origin_index=0)
        if grid_idx < 0 or grid_idx >= len(detail.hexGrid.hexes):
            continue
        
        # Use coverage_at to see which chunks cover this position
        coverage = chunk_set.coverage_at(world_pos)
        for chunk, idx, is_core in coverage:
            gi = chunk_indices[id(chunk)]
            if is_core:
                chunk_cores[gi].add(grid_idx)
            else:
                chunk_halos[gi].add(grid_idx)
    
    # Styles
    chunk_styles = StyleCSS.seaborn("Set2", len(chunks))
    textStyle = StyleCSS("textStyle", fill="white", font_size="1.5em", 
                         filter="drop-shadow(3px 3px 2px rgba(0,0,0,.7))")
    detail.hexGrid.builder.add_style(textStyle)
    
    region_overlay = ""
    legend_styles = []
    
    for i, (global_idx, _) in enumerate(selected):
        base_color = chunk_styles[i].properties['fill']
        
        # Draw HALO first (underneath)
        halo_indices = chunk_halos[global_idx] - chunk_cores[global_idx]  # Exclude core overlap
        if halo_indices:
            halo_region = HexRegion(hexes=halo_indices, hexGrid=detail.hexGrid)
            halo_style = StyleCSS(
                f"Halo_{global_idx}",
                fill=base_color,
                stroke=base_color,
                stroke_width=1,
                stroke_dasharray="4,2",
                opacity=0.15
            )
            detail.hexGrid.builder.add_style(halo_style)
            region_overlay += halo_region.draw(style=halo_style, f=unique_windy_edge(), inset=0.95)
        
        # Draw CORE on top
        core_indices = chunk_cores[global_idx]
        if core_indices:
            core_region = HexRegion(hexes=core_indices, hexGrid=detail.hexGrid)
            core_style = StyleCSS(
                f"Core_{global_idx}",
                fill=base_color,
                stroke=base_color,
                stroke_width=3,
                opacity=0.35
            )
            detail.hexGrid.builder.add_style(core_style)
            legend_styles.append(core_style.copy(f"Chunk_{global_idx}"))
            region_overlay += core_region.draw(style=core_style, f=unique_windy_edge(), inset=0.95)
            
            # Label at centroid
            centroid_idx = core_region.centroid_hex()
            if centroid_idx and 0 <= centroid_idx < len(detail.hexGrid.hexes):
                cx = detail.hexGrid.hexes[centroid_idx].center.x
                cy = detail.hexGrid.hexes[centroid_idx].center.y
                region_overlay += f'<text x="{cx}" y="{cy}" text-anchor="middle" fill="white" font-size="14" font-weight="bold" style="filter: drop-shadow(2px 2px 1px rgba(0,0,0,.8))">Chunk {global_idx}</text>\n'
    
    detail.hexGrid.builder.adjust("chunk_regions", region_overlay)
    
    # Legend
    keyLabelStyle = StyleCSS("keyLabel", fill="#333", font_size="12px")
    detail.hexGrid.builder.add_style(keyLabelStyle)
    for style in legend_styles:
        style.properties['opacity'] = '1.0'
        detail.hexGrid.builder.add_style(style)
    legend_svg = detail.hexGrid.builder.legendBelow(legend_styles, gap=15, bg_fill="rgba(255,255,255,0.9)")
    detail.hexGrid.builder.adjust("legend", legend_svg)
    
    detail.hexGrid.builder.add_centered_text("Core (solid) vs Halo (dashed) via ChunkSet", 
        y_offset=-detail.hexGrid.builder.height/2 + 50, class_name=textStyle.name)
    
    print(f"ChunkSet stats:")
    print(f"  Core positions: {len(list(chunk_set.iter_positions()))}")
    print(f"  Total indexed: {len(chunk_set.pos_to_chunk)}")
    print(f"  Chunks: {[gi for gi, _ in selected]}")
    detail.colorMap() 
    detail.hexGrid.update()
    detail.hexGrid.builder.show()


In [ ]:
with GeoStorageDebugger(keep_on_error=False) as debugger:
    storage = debugger.server
    
    gen = ChunkedTerrainGenerator(
        storage=storage,
        coarse_rings=15,
        coarse_radius=50,
        fine_radius=10,
        chunk_rings=8,
        halo_rings=3
    )
    
    coarse = gen.generate_coarse(
        num_plates=8, ocean_fraction=0.35, oceanic_sides=['W'], seed=42
    )
    
    all_chunks, coastal_chunks = find_coastal_chunks(gen, coarse)
    selected = coastal_chunks[:3]  # Chunks 0, 3, 4
    
    # Generate chunks and create ChunkSet
    chunks = [gen.generate_chunk(ref) for _, ref in selected]
    chunk_indices = {id(c): gi for (gi, _), c in zip(selected, chunks)}
    
    chunk_set = ChunkSet(chunks, coarse, gen.scale)
    
    # Convert to terrain WITHOUT fill_gaps - only shows actual chunk data
    detail = chunk_set.to_terrain(fill_gaps=False)
    detail.colorMap()
    detail.hexGrid.adjustRadius(12)  # Slightly larger hexes for clarity
    detail.hexGrid.update()
    
    # Compute origin
    positions = list(chunk_set.pos_to_chunk.keys())
    min_q = min(p.q for p in positions)
    min_r = min(p.r for p in positions)
    origin_world = HexPosition(min_q, min_r, -min_q - min_r)
    
    # Group by chunk core/halo
    chunk_cores = {gi: set() for gi in chunk_indices.values()}
    chunk_halos = {gi: set() for gi in chunk_indices.values()}
    
    for world_pos in positions:
        local_pos = world_pos - origin_world
        grid_idx = detail.hexGrid.hexposition_to_index(local_pos, origin_index=0)
        if grid_idx < 0 or grid_idx >= len(detail.hexGrid.hexes):
            continue
        
        coverage = chunk_set.coverage_at(world_pos)
        for chunk, idx, is_core in coverage:
            gi = chunk_indices[id(chunk)]
            if is_core:
                chunk_cores[gi].add(grid_idx)
            else:
                chunk_halos[gi].add(grid_idx)
    
    # Draw chunk boundaries
    chunk_colors = ["#e74c3c", "#3498db", "#2ecc71"]  # Red, Blue, Green
    region_overlay = ""
    
    for i, (global_idx, _) in enumerate(selected):
        color = chunk_colors[i]
        
        # Core boundary only (solid line)
        core_indices = chunk_cores[global_idx]
        if core_indices:
            core_region = HexRegion(hexes=core_indices, hexGrid=detail.hexGrid)
            boundary_style = StyleCSS(
                f"Boundary_{global_idx}",
                fill="none",
                stroke=color,
                stroke_width=4
            )
            detail.hexGrid.builder.add_style(boundary_style)
            
            paths = core_region.trace_perimeter(style=boundary_style)
            for path in paths:
                curved = path.make_windy(iterations=1, offset_factor=0.1)
                region_overlay += curved.svg()
            
            # Label
            centroid_idx = core_region.centroid_hex()
            if centroid_idx and 0 <= centroid_idx < len(detail.hexGrid.hexes):
                cx = detail.hexGrid.hexes[centroid_idx].center.x
                cy = detail.hexGrid.hexes[centroid_idx].center.y
                region_overlay += f'<text x="{cx}" y="{cy}" text-anchor="middle" fill="{color}" font-size="18" font-weight="bold" style="filter: drop-shadow(2px 2px 1px rgba(0,0,0,.8))">Chunk {global_idx}</text>\n'
    
    detail.hexGrid.builder.adjust("boundaries", region_overlay)
    
    textStyle = StyleCSS("textStyle", fill="white", font_size="1.2em", 
                         filter="drop-shadow(2px 2px 1px rgba(0,0,0,.7))")
    detail.hexGrid.builder.add_style(textStyle)
    detail.hexGrid.builder.add_centered_text("Chunks 0, 3, 4 - Core boundaries", 
        y_offset=-detail.hexGrid.builder.height/2 + 30, class_name=textStyle.name)
    
    print(f"Tight view: {len(list(chunk_set.iter_positions()))} core hexes from 3 chunks")
    detail.colorMap()
    detail.hexGrid.update()
    detail.hexGrid.builder.show()


In [ ]:
with GeoStorageDebugger(keep_on_error=False) as debugger:
    storage = debugger.server
    
    gen = ChunkedTerrainGenerator(
        storage=storage,
        coarse_rings=15,
        coarse_radius=50,
        fine_radius=10,
        chunk_rings=8,
        halo_rings=3
    )
    
    coarse = gen.generate_coarse(
        num_plates=8,
        ocean_fraction=0.35,
        oceanic_sides=['W'],
        name="zoom_demo",
        seed=42
    )
    
    # Get coastal chunks (same as before)
    all_chunks = list(gen.chunk_refs(max_ring=2))
    coastal_chunks = []
    for i, chunk_ref in enumerate(all_chunks):
        center = chunk_ref.center_hex
        coarse_q = int(round(center.q / gen.scale))
        coarse_r = int(round(center.r / gen.scale))
        coarse_s = -coarse_q - coarse_r
        
        has_land = has_ocean = False
        for ring in range(3):
            positions = [HexPosition(coarse_q, coarse_r, coarse_s)] if ring == 0 else \
                        HexPosition(coarse_q, coarse_r, coarse_s).ring(ring)
            for pos in positions:
                idx = coarse.hexGrid.hexposition_to_index(pos, origin_index=coarse.hexGrid.midpoint)
                if 0 <= idx < len(coarse.elevations):
                    if coarse.elevations[idx] > 0: has_land = True
                    else: has_ocean = True
        if has_land and has_ocean:
            coastal_chunks.append((i, chunk_ref))
    
    selected = coastal_chunks[:3]
    
    # DEBUG: Check what's happening
    for global_idx, chunk_ref in selected:
        chunk = gen.generate_chunk(chunk_ref)
        print(f"Chunk {global_idx}: center_world={chunk.center_world}")
        print(f"  Core hexes: {chunk.core_size}, Halo hexes: {len(chunk) - chunk.core_size}")
        
        # Check halo world positions
        halo_positions = [chunk.index_to_world(idx) for idx in chunk.iter_halo()]
        if halo_positions:
            print(f"  Halo sample positions: {halo_positions[:3]}")
    
    # Check triangle: what's in the middle between chunk centers?
    chunk_centers = [gen.generate_chunk(cr).center_world for _, cr in selected]
    print(f"\nChunk centers: {chunk_centers}")
    
    # Find the centroid of the 3 chunk centers
    avg_q = sum(c.q for c in chunk_centers) // 3
    avg_r = sum(c.r for c in chunk_centers) // 3
    avg_s = -avg_q - avg_r
    centroid = HexPosition(avg_q, avg_r, avg_s)
    print(f"Triangle centroid: {centroid}")
    
    # Check if centroid is covered by any chunk's core or halo
    for global_idx, chunk_ref in selected:
        chunk = gen.generate_chunk(chunk_ref)
        local_idx = chunk.world_to_index(centroid)
        if local_idx >= 0:
            in_core = chunk.is_core(local_idx)
            in_halo = chunk.is_halo(local_idx)
            print(f"  Centroid in chunk {global_idx}: core={in_core}, halo={in_halo}")
        else:
            dist = centroid.distance(chunk.center_world)
            print(f"  Centroid NOT in chunk {global_idx} (distance={dist}, max halo dist={gen.chunk_rings + gen.halo_rings})")


In [ ]:
def find_coastal_chunks(gen, coarse, max_ring=2):
    """Find chunks that contain both land and ocean."""
    all_chunks = list(gen.chunk_refs(max_ring=max_ring))
    coastal = []
    
    for i, chunk_ref in enumerate(all_chunks):
        center = chunk_ref.center_hex
        coarse_q = int(round(center.q / gen.scale))
        coarse_r = int(round(center.r / gen.scale))
        coarse_s = -coarse_q - coarse_r
        
        has_land = has_ocean = False
        for ring in range(3):
            positions = [HexPosition(coarse_q, coarse_r, coarse_s)] if ring == 0 else \
                        HexPosition(coarse_q, coarse_r, coarse_s).ring(ring)
            for pos in positions:
                idx = coarse.hexGrid.hexposition_to_index(pos, origin_index=coarse.hexGrid.midpoint)
                if 0 <= idx < len(coarse.elevations):
                    if coarse.elevations[idx] > 0: has_land = True
                    else: has_ocean = True
        
        if has_land and has_ocean:
            coastal.append((i, chunk_ref))
    
    return all_chunks, coastal


def render_global_map(gen, coarse, all_chunks, coastal_chunks):
    """Render coarse map with chunk locations marked."""
    coarse.hexGrid.adjustRadius(12)
    coarse.colorMap()
    coarse.hexGrid.update()
    
    textStyle = StyleCSS("textStyle", fill="white", font_size="1.5em", 
                         filter="drop-shadow(3px 3px 2px rgba(0,0,0,.7))")
    coarse.hexGrid.builder.add_style(textStyle)
    
    chunk_overlay = ""
    coastal_indices = {ci for ci, _ in coastal_chunks}
    
    for i, chunk_ref in enumerate(all_chunks):
        center = chunk_ref.center_hex
        coarse_q = int(round(center.q / gen.scale))
        coarse_r = int(round(center.r / gen.scale))
        coarse_s = -coarse_q - coarse_r
        coarse_pos = HexPosition(coarse_q, coarse_r, coarse_s)
        coarse_idx = coarse.hexGrid.hexposition_to_index(coarse_pos, origin_index=coarse.hexGrid.midpoint)
        
        if coarse_idx >= 0:
            hex_obj = coarse.hexGrid.hexes[coarse_idx]
            is_coastal = i in coastal_indices
            color = "#00ff00" if is_coastal else "#ff0000"
            fill = "rgba(0,255,0,0.2)" if is_coastal else "none"
            chunk_overlay += f'<circle cx="{hex_obj.center.x}" cy="{hex_obj.center.y}" r="10" fill="{fill}" stroke="{color}" stroke-width="2"/>\n'
            chunk_overlay += f'<text x="{hex_obj.center.x}" y="{hex_obj.center.y + 4}" text-anchor="middle" fill="white" font-size="8" font-weight="bold">{i}</text>\n'
    
    coarse.hexGrid.builder.adjust("chunks", chunk_overlay)
    coarse.hexGrid.builder.add_centered_text("Global View - Coastal chunks in green", 
        y_offset=-coarse.hexGrid.builder.height/2 + 30, class_name=textStyle.name)
    
    return coarse.hexGrid.builder


def render_detail_with_overlaps(gen, coarse, selected_chunks):
    """Render detailed view showing cores and halo overlaps."""
    
    # Generate chunks and collect positions
    chunks_data = []
    all_core_positions = []
    all_halo_positions = []
    
    for global_idx, chunk_ref in selected_chunks:
        chunk = gen.generate_chunk(chunk_ref)
        core_positions = [chunk.index_to_world(idx) for idx in chunk.iter_core()]
        halo_positions = [chunk.index_to_world(idx) for idx in chunk.iter_halo()]
        chunks_data.append((global_idx, chunk_ref, chunk, core_positions, halo_positions))
        all_core_positions.extend(core_positions)
        all_halo_positions.extend(halo_positions)
    
    # Calculate bounds including halo
    all_pos = all_core_positions + all_halo_positions
    qs = [p.q for p in all_pos]
    rs = [p.r for p in all_pos]
    
    pad = 3
    min_q, max_q = min(qs) - pad, max(qs) + pad
    min_r, max_r = min(rs) - pad, max(rs) + pad
    
    # Generate detailed terrain
    detail = gen.rect_region_from_chunks(min_q, max_q, min_r, max_r)
    detail.hexGrid.update()
    
    origin_world = HexPosition(min_q, min_r, -min_q - min_r)
    
    # Build index sets for each chunk's core and halo
    chunk_cores = {}    # global_idx -> set of grid indices
    chunk_halos = {}    # global_idx -> set of grid indices
    
    for global_idx, chunk_ref, chunk, core_positions, halo_positions in chunks_data:
        core_indices = set()
        for world_pos in core_positions:
            local_pos = world_pos - origin_world
            grid_idx = detail.hexGrid.hexposition_to_index(local_pos, origin_index=0)
            if 0 <= grid_idx < len(detail.hexGrid.hexes):
                core_indices.add(grid_idx)
        chunk_cores[global_idx] = core_indices
        
        halo_indices = set()
        for world_pos in halo_positions:
            local_pos = world_pos - origin_world
            grid_idx = detail.hexGrid.hexposition_to_index(local_pos, origin_index=0)
            if 0 <= grid_idx < len(detail.hexGrid.hexes):
                halo_indices.add(grid_idx)
        chunk_halos[global_idx] = halo_indices
    
    # Calculate overlaps
    all_halo_indices = set()
    for indices in chunk_halos.values():
        all_halo_indices |= indices
    
    all_core_indices = set()
    for indices in chunk_cores.values():
        all_core_indices |= indices
    
    # Halo-only (not in any core)
    halo_only = all_halo_indices - all_core_indices
    
    # Find overlapping halo regions (in 2+ halos)
    from collections import Counter
    halo_counts = Counter()
    for indices in chunk_halos.values():
        for idx in indices:
            halo_counts[idx] += 1
    
    # Indices in multiple halos (overlap zone)
    overlap_indices = {idx for idx, count in halo_counts.items() if count >= 2}
    overlap_in_halo_only = overlap_indices & halo_only
    
    # Styles
    chunk_styles = StyleCSS.seaborn("Set2", len(chunks_data))
    textStyle = StyleCSS("textStyle", fill="white", font_size="1.5em", 
                         filter="drop-shadow(3px 3px 2px rgba(0,0,0,.7))")
    detail.hexGrid.builder.add_style(textStyle)
    
    region_overlay = ""
    legend_styles = []
    
    # 1. Draw overlap region first (underneath everything)
    if overlap_in_halo_only:
        overlap_region = HexRegion(hexes=overlap_in_halo_only, hexGrid=detail.hexGrid)
        overlap_style = StyleCSS(
            "Halo_Overlap",
            fill="#ff00ff",  # Magenta for visibility
            stroke="#ff00ff",
            stroke_width=2,
            opacity=0.25
        )
        detail.hexGrid.builder.add_style(overlap_style)
        region_overlay += overlap_region.draw(style=overlap_style, f=unique_windy_edge(), inset=0.9)
        legend_styles.append(overlap_style.copy("Halo_Overlap"))
    
    # 2. Draw individual halo boundaries (dashed, no fill)
    for i, (global_idx, chunk_ref, chunk, core_positions, halo_positions) in enumerate(chunks_data):
        base_color = chunk_styles[i].properties['fill']
        halo_indices = chunk_halos[global_idx]
        
        if halo_indices:
            halo_region = HexRegion(hexes=halo_indices, hexGrid=detail.hexGrid)
            halo_style = StyleCSS(
                f"Halo_{global_idx}",
                fill="none",
                stroke=base_color,
                stroke_width=3,
                stroke_dasharray="8,4",
                opacity=0.7
            )
            detail.hexGrid.builder.add_style(halo_style)
            region_overlay += halo_region.draw(style=halo_style, f=unique_windy_edge(), inset=0.98)
    
    # 3. Draw cores on top
    for i, (global_idx, chunk_ref, chunk, core_positions, halo_positions) in enumerate(chunks_data):
        base_color = chunk_styles[i].properties['fill']
        core_indices = chunk_cores[global_idx]
        
        if core_indices:
            core_region = HexRegion(hexes=core_indices, hexGrid=detail.hexGrid)
            core_style = StyleCSS(
                f"Core_{global_idx}",
                fill=base_color,
                stroke=base_color,
                stroke_width=3,
                opacity=0.35
            )
            detail.hexGrid.builder.add_style(core_style)
            region_overlay += core_region.draw(style=core_style, f=unique_windy_edge(), inset=0.95)
            legend_styles.append(core_style.copy(f"Chunk_{global_idx}"))
            
            # Label at centroid
            centroid_idx = core_region.centroid_hex()
            if centroid_idx and 0 <= centroid_idx < len(detail.hexGrid.hexes):
                cx = detail.hexGrid.hexes[centroid_idx].center.x
                cy = detail.hexGrid.hexes[centroid_idx].center.y
                region_overlay += f'<text x="{cx}" y="{cy}" text-anchor="middle" fill="white" font-size="14" font-weight="bold" style="filter: drop-shadow(2px 2px 1px rgba(0,0,0,.8))">Chunk {global_idx}</text>\n'
    
    detail.hexGrid.builder.adjust("chunk_regions", region_overlay)
    
    # Add legend
    keyLabelStyle = StyleCSS("keyLabel", fill="#333", font_size="12px")
    detail.hexGrid.builder.add_style(keyLabelStyle)
    
    for style in legend_styles:
        style.properties['opacity'] = '1.0'
        detail.hexGrid.builder.add_style(style)
    
    legend_svg = detail.hexGrid.builder.legendBelow(legend_styles, gap=15, bg_fill="rgba(255,255,255,0.9)")
    detail.hexGrid.builder.adjust("legend", legend_svg)
    
    detail.hexGrid.builder.add_style(textStyle)
    detail.hexGrid.builder.add_centered_text("Chunk Cores + Halo Overlaps (magenta)", 
        y_offset=-detail.hexGrid.builder.height/2 + 50, class_name=textStyle.name)
    
    return detail.hexGrid.builder, chunks_data


# === MAIN DEMO ===
with GeoStorageDebugger(keep_on_error=False) as debugger:
    gen = ChunkedTerrainGenerator(
        storage=debugger.server,
        coarse_rings=15,
        coarse_radius=50,
        fine_radius=10,
        chunk_rings=8,
        halo_rings=3
    )
    
    coarse = gen.generate_coarse(
        num_plates=8,
        ocean_fraction=0.35,
        oceanic_sides=['W'],
        name="zoom_demo",
        seed=42
    )
    
    # Find chunks
    all_chunks, coastal_chunks = find_coastal_chunks(gen, coarse)
    print(f"Found {len(coastal_chunks)} coastal chunks out of {len(all_chunks)} total")
    
    # 1. Global map
    print("\n=== GLOBAL MAP ===")
    global_builder = render_global_map(gen, coarse, all_chunks, coastal_chunks)
    display(global_builder.show())
    
    # 2. Detail view with first 3 coastal chunks
    selected = coastal_chunks[:3]
    print(f"\n=== DETAIL VIEW (chunks {[i for i, _ in selected]}) ===")
    detail_builder, chunks_data = render_detail_with_overlaps(gen, coarse, selected)
    display(detail_builder.show())


In [ ]:
def visualize_overlap_interpolation(gen, coarse, selected_chunks, zoom_padding=2):
    """Zoom into overlap zone and show chunk influence with gradient patterns."""
    
    # Generate chunks and create ChunkSet
    chunks = [gen.generate_chunk(ref) for _, ref in selected_chunks]
    chunk_indices = {id(c): gi for (gi, _), c in zip(selected_chunks, chunks)}
    
    chunk_set = ChunkSet(chunks, coarse, gen.scale)
    
    # Find overlap positions using ChunkSet.coverage_at
    overlap_positions = set()
    for world_pos in chunk_set.pos_to_chunk:
        coverage = chunk_set.coverage_at(world_pos)
        if len(coverage) >= 2:
            overlap_positions.add(world_pos)
    
    if not overlap_positions:
        print("No overlap found!")
        return None
    
    # Bounds around overlap
    all_positions = list(chunk_set.pos_to_chunk.keys())
    qs = [p.q for p in all_positions]
    rs = [p.r for p in all_positions]
    min_q, max_q = min(qs) - zoom_padding, max(qs) + zoom_padding
    min_r, max_r = min(rs) - zoom_padding, max(rs) + zoom_padding
    
    nrows = max_r - min_r + 1
    ncols = max_q - min_q + 1
    grid = HexGrid(nRows=nrows, nCols=ncols, radius=gen.fine_radius * 2, style=coarse.seaLevel)
    
    origin_world = HexPosition(min_q, min_r, -min_q - min_r)
    
    # Chunk colors
    chunk_colors = {}
    chunk_styles = StyleCSS.seaborn("Set2", len(chunks))
    for i, (global_idx, _) in enumerate(selected_chunks):
        chunk_colors[global_idx] = chunk_styles[i].properties['fill']
    
    # Create patterns for each chunk at different densities
    patternGen = TerrainPatterns(coarse)
    density_levels = 6
    chunk_patterns = {}  # (global_idx, level) -> pattern
    
    for global_idx in chunk_colors:
        color = chunk_colors[global_idx]
        patterns = patternGen.ballDensity(
            levels=density_levels,
            fills=[color],
            prefix=f"chunk_{global_idx}"
        )
        for level, pattern in enumerate(patterns):
            chunk_patterns[(global_idx, level)] = pattern
    
    # Build SVG
    builder = SVGBuilder()
    builder.width = grid.builder.width
    builder.height = grid.builder.height
    
    # Add terrain styles
    for style in StyleCSS.elevations():
        builder.add_style(style)
    builder.add_style(coarse.seaLevel)
    
    # Add all patterns to builder
    for pattern in chunk_patterns.values():
        builder.add_definition(pattern)
    
    # First pass: base terrain using ChunkSet.elevation_at
    terrain_svg = ""
    for grid_idx in range(len(grid.hexes)):
        local_pos = grid.index_to_hexposition(grid_idx, origin_index=0)
        world_pos = origin_world + local_pos
        hex_obj = grid.hexes[grid_idx]
        
        elev = chunk_set.elevation_at(world_pos)  # Uses ChunkSet!
        
        level = int(elev / coarse.elevationDelta)
        level = max(0, min(level, len(StyleCSS.elevations()) - 1))
        style = coarse.seaLevel if elev <= 0 else StyleCSS.elevations()[level]
        hex_obj.style = style
        terrain_svg += "\t" + hex_obj.svg() + "\n"
    
    builder.adjust("terrain", terrain_svg)
    
    # Second pass: overlay with influence patterns
    overlay_svg = ""
    
    for grid_idx in range(len(grid.hexes)):
        local_pos = grid.index_to_hexposition(grid_idx, origin_index=0)
        world_pos = origin_world + local_pos
        hex_obj = grid.hexes[grid_idx]
        
        # Use ChunkSet.coverage_at for all chunks covering this position
        coverage = chunk_set.coverage_at(world_pos)
        if not coverage:
            continue
        
        # Calculate influence (inverse distance) for each chunk
        influences = []
        for chunk, idx, is_core in coverage:
            global_idx = chunk_indices[id(chunk)]
            chunk_center = chunk.center_world
            dist = world_pos.distance(chunk_center)
            max_dist = gen.chunk_rings + gen.halo_rings
            influence = max(0, 1.0 - dist / max_dist)
            influences.append((global_idx, influence))
        
        # Draw ALL patterns (they layer/blend visually via dots)
        for global_idx, influence in influences:
            if influence > 0.05:
                level = int(influence * (density_levels - 1))
                level = min(level, density_levels - 1)
                
                pattern_key = (global_idx, level)
                if pattern_key in chunk_patterns:
                    pat_id = chunk_patterns[pattern_key].attributes['id']
                    points = " ".join(f"{v.x},{v.y}" for v in hex_obj.v)
                    overlay_svg += f'<polygon points="{points}" fill="url(#{pat_id})"/>\n'
    
    builder.adjust("influence", overlay_svg)
    
    # Add chunk center markers
    markers_svg = ""
    for chunk in chunks:
        global_idx = chunk_indices[id(chunk)]
        chunk_center = chunk.center_world
        local_center = chunk_center - origin_world
        center_idx = grid.hexposition_to_index(local_center, origin_index=0)
        
        if 0 <= center_idx < len(grid.hexes):
            hex_obj = grid.hexes[center_idx]
            color = chunk_colors[global_idx]
            markers_svg += f'<circle cx="{hex_obj.center.x}" cy="{hex_obj.center.y}" r="15" fill="{color}" stroke="white" stroke-width="3"/>\n'
            markers_svg += f'<text x="{hex_obj.center.x}" y="{hex_obj.center.y + 5}" text-anchor="middle" fill="white" font-size="12" font-weight="bold">{global_idx}</text>\n'
    
    builder.adjust("markers", markers_svg)
    
    # Title
    textStyle = StyleCSS("textStyle", fill="#333", font_size="14px")
    builder.add_style(textStyle)
    builder.add_centered_text(
        "Overlap Zone - Dot density shows chunk influence (denser = stronger)",
        y_offset=-builder.height/2 + 25,
        class_name=textStyle.name
    )
    
    # Legend
    legend_styles = [
        StyleCSS(f"Chunk_{gi}", fill=chunk_colors[gi], stroke=chunk_colors[gi], stroke_width=2)
        for gi in chunk_colors
    ]
    for style in legend_styles:
        builder.add_style(style)
    
    legend_svg = builder.legendBelow(legend_styles, gap=15, bg_fill="rgba(255,255,255,0.9)")
    builder.adjust("legend", legend_svg)
    
    return builder


In [ ]:

# === DEMO ===
with GeoStorageDebugger(keep_on_error=False) as debugger:
    gen = ChunkedTerrainGenerator(
        storage=debugger.server,
        coarse_rings=15,
        coarse_radius=50,
        fine_radius=10,
        chunk_rings=8,
        halo_rings=3
    )
    
    coarse = gen.generate_coarse(
        num_plates=8,
        ocean_fraction=0.35,
        oceanic_sides=['W'],
        name="zoom_demo",
        seed=42
    )
    
    all_chunks, coastal_chunks = find_coastal_chunks(gen, coarse)
    selected = coastal_chunks[:3]  # Chunks 0, 3, 4
    
    print(f"Visualizing overlap interpolation for chunks {[i for i, _ in selected]}")
    
    overlap_builder = visualize_overlap_interpolation(gen, coarse, selected)
    if overlap_builder:
        display(overlap_builder.show())

### More datamethods


## The Core Challenge

Hydrology is inherently **non-local** - water flows across arbitrary distances. With chunks, we need to:
1. Respect coarse-level drainage patterns
2. Add detail within chunks
3. Handle flow crossing chunk boundaries (through halos)

## Proposed Strategy

```
┌─────────────────────────────────────────────────────────────┐
│  PHASE 1: COARSE DRAINAGE                                   │
│  ─────────────────────────                                  │
│  • Run carve_to_ocean() on coarse map                       │
│  • Identify drainage paths & outlets                        │
│  • Store as "drainage constraints"                          │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│  PHASE 2: CHUNK ORDERING                                    │
│  ───────────────────────                                    │
│  • Compute chunk dependency graph from coarse drainage      │
│  • Order: downstream chunks first (outlets → headwaters)    │
│  • Why? Downstream chunks set boundary conditions           │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│  PHASE 3: CHUNK HYDROLOGY                                   │
│  ────────────────────────                                   │
│  For each chunk (in dependency order):                      │
│    1. Load halo from already-processed neighbors            │
│    2. Interpolate coarse drainage into chunk                │
│    3. Find local minima in CORE only                        │
│    4. Carve paths to: a) halo exit points, or               │
│                       b) coarse drainage paths              │
│    5. Save core; halo is read-only                          │
└─────────────────────────────────────────────────────────────┘
```

## Halo Rules

```
     Chunk A (processed)          Chunk B (processing now)
    ┌───────────────────┐        ┌───────────────────┐
    │                   │        │                   │
    │      CORE A       │◄───────│   HALO B          │  ← B reads A's core
    │                   │        │   (read-only)     │
    │          ┌────────┼────────┼────────┐         │
    │          │ HALO A │        │ CORE B │         │
    │          │overlap │        │(writable)        │
    │          └────────┼────────┼────────┘         │
    └───────────────────┘        └───────────────────┘
    
    Key insight: A's halo ∩ B's halo = both read from coarse until 
                 one of them processes and becomes authoritative
```


I want to add carve to ocean for this process

We should a first pass at the coarse level. 
then we need someway to go through the HexChunks. There are two cases. minium inside the HexChunk center and those in the flow. We need to figure out how to handle this for hexes in a halo. I guess a halo is shared among three HexChunk and we can figure out how to align these. It is a little complex so sketching on a strategy would be great. We are eventually also going to need to compute weather (where we run the model on the coarse layer and then use it as a basis for the lower layers. computing watersheds is also going to have our halo issue so working this out as well is going to be strong. It is helpful for me if we can use overlays to build the strategy. where should we start?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great question! This is a classic **multi-resolution data flow** problem. Let me sketch the strategy:

## The Core Challenge

Hydrology is inherently **non-local** - water flows across arbitrary distances. With chunks, we need to:
1. Respect coarse-level drainage patterns
2. Add detail within chunks
3. Handle flow crossing chunk boundaries (through halos)

## Proposed Strategy

```
┌─────────────────────────────────────────────────────────────┐
│  PHASE 1: COARSE DRAINAGE                                   │
│  ─────────────────────────                                  │
│  • Run carve_to_ocean() on coarse map                       │
│  • Identify drainage paths & outlets                        │
│  • Store as "drainage constraints"                          │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│  PHASE 2: CHUNK ORDERING                                    │
│  ───────────────────────                                    │
│  • Compute chunk dependency graph from coarse drainage      │
│  • Order: downstream chunks first (outlets → headwaters)    │
│  • Why? Downstream chunks set boundary conditions           │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│  PHASE 3: CHUNK HYDROLOGY                                   │
│  ────────────────────────                                   │
│  For each chunk (in dependency order):                      │
│    1. Load halo from already-processed neighbors            │
│    2. Interpolate coarse drainage into chunk                │
│    3. Find local minima in CORE only                        │
│    4. Carve paths to: a) halo exit points, or               │
│                       b) coarse drainage paths              │
│    5. Save core; halo is read-only                          │
└─────────────────────────────────────────────────────────────┘
```

## Halo Rules

```
     Chunk A (processed)          Chunk B (processing now)
    ┌───────────────────┐        ┌───────────────────┐
    │                   │        │                   │
    │      CORE A       │◄───────│   HALO B          │  ← B reads A's core
    │                   │        │   (read-only)     │
    │          ┌────────┼────────┼────────┐         │
    │          │ HALO A │        │ CORE B │         │
    │          │overlap │        │(writable)        │
    │          └────────┼────────┼────────┘         │
    └───────────────────┘        └───────────────────┘
    
    Key insight: A's halo ∩ B's halo = both read from coarse until 
                 one of them processes and becomes authoritative
```

## Where to Start?

I'd suggest **visualizing the coarse drainage first** - this will reveal:
1. Which chunks have drainage paths passing through
2. Natural entry/exit points at chunk boundaries
3. Dependency order (which chunks drain into which)

```python
def visualize_coarse_drainage(gen, coarse):
    """Show drainage paths on coarse map with chunk overlay."""
    
    # 1. Run carve_to_ocean on coarse
    drainage_rivers = coarse.carve_to_ocean(num_lakes=3, max_iters=5)
    
    # 2. Draw coarse map
    coarse.colorMap()
    coarse.hexGrid.update()
    
    # 3. Overlay drainage paths
    river_svg = ""
    for river in drainage_rivers:
        river_svg += river.svg()
    coarse.hexGrid.builder.adjust("drainage", river_svg)
    
    # 4. Mark chunk boundaries and drainage crossings
    # ... (show where rivers cross chunk boundaries)
    
    return coarse.hexGrid.builder
```

Want me to implement this visualization? It'll help us see:
- Where drainage "enters" and "exits" each chunk
- Which chunks need to be processed before others
- Where the halo synchronization is critical

In [ ]:
#| export
def visualize_coarse_drainage(gen, coarse, max_ring=2):
    """Show drainage paths on coarse map with chunk overlay."""
    
    # 1. Carve drainage paths on coarse
    drainage_rivers = coarse.carve_to_ocean(num_lakes=3, max_iters=5)
    print(f"Created {len(drainage_rivers)} drainage paths")
    
    # 2. Recolor after carving
    coarse.hexGrid.adjustRadius(12)
    coarse.colorMap()
    coarse.hexGrid.update()
    
    # 3. Add flow diagram (arrows)
    flow_overlay = coarse.flow_diagram()
    coarse.hexGrid.builder.adjust("flow_arrows", flow_overlay)
    
    # 4. Draw drainage rivers (thicker, blue)
    river_style = StyleCSS("drainage", fill="none", stroke="#1565c0", stroke_width=4)
    coarse.hexGrid.builder.add_style(river_style)
    
    river_svg = ""
    for river in drainage_rivers:
        river_svg += river.svg(styles=[river_style])
    coarse.hexGrid.builder.adjust("drainage_rivers", river_svg)
    
    # 5. Mark chunk centers and find drainage crossings
    all_chunks = list(gen.chunk_refs(max_ring=max_ring))
    
    # Collect all drainage hex positions (in coarse coords)
    drainage_hexes = set()
    for river in drainage_rivers:
        drainage_hexes.update(river.hexes)
    
    chunk_overlay = ""
    chunk_drainage_info = {}  # chunk_idx -> {enters: [], exits: []}
    
    for i, chunk_ref in enumerate(all_chunks):
        center = chunk_ref.center_hex
        coarse_q = int(round(center.q / gen.scale))
        coarse_r = int(round(center.r / gen.scale))
        coarse_s = -coarse_q - coarse_r
        coarse_pos = HexPosition(coarse_q, coarse_r, coarse_s)
        coarse_idx = coarse.hexGrid.hexposition_to_index(coarse_pos, origin_index=coarse.hexGrid.midpoint)
        
        if coarse_idx < 0:
            continue
            
        hex_obj = coarse.hexGrid.hexes[coarse_idx]
        
        # Check if chunk contains drainage
        # Approximate chunk coverage in coarse hexes
        chunk_radius_coarse = int(gen.chunk_rings / gen.scale) + 1
        
        enters = []
        exits = []
        has_drainage = False
        
        for ring in range(chunk_radius_coarse + 1):
            positions = [coarse_pos] if ring == 0 else list(coarse_pos.ring(ring))
            for pos in positions:
                idx = coarse.hexGrid.hexposition_to_index(pos, origin_index=coarse.hexGrid.midpoint)
                if idx >= 0 and idx in drainage_hexes:
                    has_drainage = True
                    # Check if on boundary (ring == chunk_radius_coarse)
                    if ring == chunk_radius_coarse:
                        # Determine if entering or exiting based on flow direction
                        flow_dir = coarse.flow_directions()
                        if flow_dir[idx] >= 0:
                            next_pos = coarse.hexGrid.index_to_hexposition(flow_dir[idx], origin_index=coarse.hexGrid.midpoint)
                            next_dist = next_pos.distance(coarse_pos)
                            curr_dist = pos.distance(coarse_pos)
                            if next_dist < curr_dist:
                                enters.append(idx)
                            else:
                                exits.append(idx)
        
        chunk_drainage_info[i] = {'enters': enters, 'exits': exits, 'has_drainage': has_drainage}
        
        # Color: green if has drainage, red otherwise
        color = "#00ff00" if has_drainage else "#ff6666"
        fill = "rgba(0,255,0,0.15)" if has_drainage else "rgba(255,100,100,0.1)"
        
        chunk_overlay += f'<circle cx="{hex_obj.center.x}" cy="{hex_obj.center.y}" r="12" fill="{fill}" stroke="{color}" stroke-width="2"/>\n'
        chunk_overlay += f'<text x="{hex_obj.center.x}" y="{hex_obj.center.y + 4}" text-anchor="middle" fill="white" font-size="10" font-weight="bold">{i}</text>\n'
        
        # Mark entry/exit points
        for idx in enters:
            h = coarse.hexGrid.hexes[idx]
            chunk_overlay += f'<circle cx="{h.center.x}" cy="{h.center.y}" r="5" fill="#00ff00" stroke="white" stroke-width="1"/>\n'
        for idx in exits:
            h = coarse.hexGrid.hexes[idx]
            chunk_overlay += f'<circle cx="{h.center.x}" cy="{h.center.y}" r="5" fill="#ff0000" stroke="white" stroke-width="1"/>\n'
    
    coarse.hexGrid.builder.adjust("chunks", chunk_overlay)
    
    # Title
    textStyle = StyleCSS("textStyle", fill="white", font_size="1.5em", 
                         filter="drop-shadow(3px 3px 2px rgba(0,0,0,.7))")
    coarse.hexGrid.builder.add_style(textStyle)
    coarse.hexGrid.builder.add_centered_text(
        "Coarse Drainage: Green=has flow, ●=entry, ●=exit", 
        y_offset=-coarse.hexGrid.builder.height/2 + 30, 
        class_name=textStyle.name
    )
    
    return coarse.hexGrid.builder, drainage_rivers, chunk_drainage_info



In [ ]:
#| export


# === RUN IT ===
with GeoStorageDebugger(keep_on_error=False) as debugger:
    gen = ChunkedTerrainGenerator(
        storage=debugger.server,
        coarse_rings=15,
        coarse_radius=50,
        fine_radius=10,
        chunk_rings=8,
        halo_rings=3
    )
    
    coarse = gen.generate_coarse(
        num_plates=8,
        ocean_fraction=0.35,
        oceanic_sides=['W'],
        name="drainage_demo",
        seed=42
    )
    
    builder, rivers, chunk_info = visualize_coarse_drainage(gen, coarse)
    
    print("\n=== CHUNK DRAINAGE SUMMARY ===")
    for i, info in chunk_info.items():
        if info['has_drainage']:
            print(f"Chunk {i}: {len(info['enters'])} entries, {len(info['exits'])} exits")
    
    display(builder.show())

In [ ]:
#| export
def compute_chunk_dependencies(chunk_info, flow_directions, coarse):
    """Build dependency graph: which chunks must process before which."""
    
    dependencies = {i: set() for i in chunk_info.keys()}  # chunk -> set of chunks it depends on
    
    # For each exit point, trace where it flows to
    for chunk_i, info in chunk_info.items():
        for exit_idx in info['exits']:
            # Follow flow until we hit another chunk's entry or ocean
            current = exit_idx
            while flow_directions[current] >= 0:
                next_hex = flow_directions[current]
                
                # Check if this hex is an entry to another chunk
                for chunk_j, info_j in chunk_info.items():
                    if chunk_j != chunk_i and next_hex in info_j['enters']:
                        # chunk_j depends on chunk_i (i must process first)
                        dependencies[chunk_j].add(chunk_i)
                        break
                
                current = next_hex
    
    return dependencies


In [ ]:
#| export
def visualize_chunk_dependencies(gen, coarse, chunk_info, max_ring=2):
    """Show chunk dependency graph with colored arrows."""
    
    flow_dir = coarse.flow_directions()
    deps = compute_chunk_dependencies(chunk_info, flow_dir, coarse)
    
    # Reuse the drainage visualization
    coarse.hexGrid.adjustRadius(12)
    coarse.colorMap()
    coarse.hexGrid.update()
    
    # Get chunk centers for arrow drawing
    all_chunks = list(gen.chunk_refs(max_ring=max_ring))
    chunk_centers = {}  # chunk_idx -> (screen_x, screen_y)
    
    for i, chunk_ref in enumerate(all_chunks):
        center = chunk_ref.center_hex
        coarse_q = int(round(center.q / gen.scale))
        coarse_r = int(round(center.r / gen.scale))
        coarse_s = -coarse_q - coarse_r
        coarse_pos = HexPosition(coarse_q, coarse_r, coarse_s)
        coarse_idx = coarse.hexGrid.hexposition_to_index(coarse_pos, origin_index=coarse.hexGrid.midpoint)
        
        if coarse_idx >= 0:
            hex_obj = coarse.hexGrid.hexes[coarse_idx]
            chunk_centers[i] = (hex_obj.center.x, hex_obj.center.y)
    
    # Compute processing order (topological sort)
    def topo_sort(deps):
        order = []
        remaining = set(deps.keys())
        satisfied = set()
        
        while remaining:
            # Find chunks with all dependencies satisfied
            ready = [c for c in remaining if deps[c] <= satisfied]
            if not ready:
                # Cycle or orphan - just take any
                ready = [min(remaining)]
            
            for c in sorted(ready):
                order.append(c)
                satisfied.add(c)
                remaining.remove(c)
        
        return order
    
    process_order = topo_sort(deps)
    order_map = {c: i for i, c in enumerate(process_order)}
    
    # Draw dependency arrows
    arrow_overlay = ""
    colors = StyleCSS.seaborn("viridis", len(process_order))
    
    for chunk_j, dep_set in deps.items():
        if chunk_j not in chunk_centers:
            continue
        jx, jy = chunk_centers[chunk_j]
        
        for chunk_i in dep_set:
            if chunk_i not in chunk_centers:
                continue
            ix, iy = chunk_centers[chunk_i]
            
            # Arrow from i to j (i processes before j)
            # Color by processing order of source
            color = colors[order_map.get(chunk_i, 0)].properties.get('fill', '#888888')
            
            # Shorten arrow so it doesn't overlap circles
            dx, dy = jx - ix, jy - iy
            length = math.sqrt(dx*dx + dy*dy)
            if length > 0:
                ux, uy = dx/length, dy/length
                # Start 15px from center, end 15px before target
                x1 = ix + ux * 15
                y1 = iy + uy * 15
                x2 = jx - ux * 15
                y2 = jy - uy * 15
                
                arrow_overlay += f'<line x1="{x1:.1f}" y1="{y1:.1f}" x2="{x2:.1f}" y2="{y2:.1f}" stroke="{color}" stroke-width="3" marker-end="url(#arrowhead)"/>\n'
    
    # Add arrowhead marker
    coarse.hexGrid.builder.add_definition(SVGDef(
        "marker", "arrowhead",
        '<path d="M 0 0 L 10 5 L 0 10 z" fill="#333"/>',
        viewBox="0 0 10 10", refX="10", refY="5",
        markerWidth="6", markerHeight="6", orient="auto"
    ))
    
    coarse.hexGrid.builder.adjust("dep_arrows", arrow_overlay)
    
    # Draw chunk circles with processing order numbers
    chunk_overlay = ""
    for i, (cx, cy) in chunk_centers.items():
        order_num = order_map.get(i, -1)
        color = colors[order_num].properties.get('fill', '#888888') if order_num >= 0 else '#888888'
        
        chunk_overlay += f'<circle cx="{cx}" cy="{cy}" r="18" fill="{color}" stroke="white" stroke-width="2"/>\n'
        chunk_overlay += f'<text x="{cx}" y="{cy + 5}" text-anchor="middle" fill="white" font-size="12" font-weight="bold">{order_num}</text>\n'
    
    coarse.hexGrid.builder.adjust("chunks", chunk_overlay)
    
    # Title
    textStyle = StyleCSS("textStyle", fill="white", font_size="1.2em", 
                         filter="drop-shadow(2px 2px 1px rgba(0,0,0,.7))")
    coarse.hexGrid.builder.add_style(textStyle)
    coarse.hexGrid.builder.add_centered_text(
        f"Processing Order (0=first, {len(process_order)-1}=last) - Arrows show dependencies", 
        y_offset=-coarse.hexGrid.builder.height/2 + 30, 
        class_name=textStyle.name
    )
    
    print(f"Processing order: {process_order}")
    print(f"\nDependencies:")
    for c in process_order:
        if deps[c]:
            print(f"  Chunk {c} depends on: {sorted(deps[c])}")
    
    return coarse.hexGrid.builder, process_order, deps




In [ ]:

# === RUN IT ===
with GeoStorageDebugger(keep_on_error=False) as debugger:
    gen = ChunkedTerrainGenerator(
        storage=debugger.server,
        coarse_rings=15,
        coarse_radius=50,
        fine_radius=10,
        chunk_rings=8,
        halo_rings=3
    )
    
    coarse = gen.generate_coarse(
        num_plates=8,
        ocean_fraction=0.35,
        oceanic_sides=['W'],
        name="drainage_demo",
        seed=42
    )
    
    # First get drainage info
    _, rivers, chunk_info = visualize_coarse_drainage(gen, coarse)
    
    # Then visualize dependencies
    builder, order, deps = visualize_chunk_dependencies(gen, coarse, chunk_info)
    display(builder.show())

I see the pattern. so the idea is to highest patterns first.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Exactly! The pattern is:

**Upstream → Downstream** (headwaters first, outlets last)

```
Headwaters (no deps)     Middle              Outlets (many deps)
┌─────────────────┐    ┌─────────────────┐    ┌─────────────────┐
│  Process first  │ →  │  Process middle │ →  │  Process last   │
│  0,3,4,5,6...   │    │  1,2,18...      │    │  7              │
│                 │    │                 │    │                 │
│  Just drain     │    │  Receive from   │    │  All water      │
│  downhill       │    │  upstream halos │    │  accumulates    │
└─────────────────┘    └─────────────────┘    └─────────────────┘
```

**Why this order works:**
1. **Headwater chunks** - No incoming flow, just carve downhill to exit points
2. **Middle chunks** - Load upstream neighbors' cores into halo, water enters correctly
3. **Outlet chunks** - All upstream is settled, just route to ocean

**For the "just elevations" approach:**
- Upstream chunks carve their terrain
- Their carved elevations become the halo for downstream chunks
- Downstream chunks naturally continue the valley

This is also the right order for **weather**:
- Prevailing winds carry moisture from ocean → inland
- Process in wind direction order
- Rain shadow effects propagate correctly

Want to test the "elevations only" approach on a few chunks to see if drainage connects properly at boundaries?

In [ ]:
#| export
def test_chunk_drainage_continuity(gen, coarse, process_order, num_chunks=3):
    """Generate chunks in order, check if drainage connects at boundaries."""
    
    # Pick first few chunks from process order
    test_chunks = process_order[:num_chunks]
    
    generated = {}  # chunk_idx -> HexChunk
    
    for chunk_idx in test_chunks:
        chunk_ref = list(gen.chunk_refs(max_ring=2))[chunk_idx]
        
        # Generate chunk (inherits carved elevations from coarse)
        chunk = gen.generate_chunk(chunk_ref)
        generated[chunk_idx] = chunk
        
        # Copy halo from already-processed neighbors
        for prev_idx, prev_chunk in generated.items():
            if prev_idx != chunk_idx:
                chunk.copy_from(prev_chunk)
        
        print(f"Chunk {chunk_idx}: {chunk.core_size} core hexes")
    
    # Visualize the combined chunks with flow arrows
    chunks = [generated[i] for i in test_chunks]
    terrain = gen.chunks_to_terrain(chunks)
    
    terrain.colorMap()
    terrain.hexGrid.update()
    
    # Add flow diagram
    flow_overlay = terrain.flow_diagram()
    terrain.hexGrid.builder.adjust("flow", flow_overlay)
    
    return terrain.hexGrid.builder, generated


In [ ]:
#| export
# Filter flow diagram to only covered hexes
@patch
def flow_diagram_filtered(self: Terrain, valid_indices: set) -> str:
    """Flow diagram only for specified hexes."""
    n = len(self.elevations)
    overlay = ""
    style = StyleCSS("arrow", stroke="black", stroke_width=1)
    self.builder.add_style(style)

    directions = self.flow_directions()
    for i in valid_indices:
        x = directions[i]
        if 0 <= x < n and self.elevations[i] > 0 and x in valid_indices:
            overlay += self.hexGrid.arrow(i, x, style=style, fromMiddle=True, factor=0.1) + "\n"
    
    return overlay

# Filter local minima to only covered hexes
def find_local_minima_filtered(terrain: Terrain, valid_indices: set) -> list[int]:
    """Local minima only within valid region."""
    directions = terrain.flow_directions()
    return [i for i in valid_indices 
            if directions[i] == -1 and terrain.elevations[i] > 0]


In [ ]:
# Modify the test to mark invalid hexes
with GeoStorageDebugger(keep_on_error=False) as debugger:
    gen = ChunkedTerrainGenerator(
        storage=debugger.server,
        coarse_rings=15, coarse_radius=50, fine_radius=10,
        chunk_rings=8, halo_rings=3
    )
    
    coarse = gen.generate_coarse(num_plates=8, ocean_fraction=0.35, oceanic_sides=['W'], seed=42)
    
    # Get processing order
    _, rivers, chunk_info = visualize_coarse_drainage(gen, coarse)
    flow_dir = coarse.flow_directions()
    deps = compute_chunk_dependencies(chunk_info, flow_dir, coarse)
    
    def topo_sort(deps):
        order, remaining, satisfied = [], set(deps.keys()), set()
        while remaining:
            ready = [c for c in remaining if deps[c] <= satisfied] or [min(remaining)]
            for c in sorted(ready):
                order.append(c)
                satisfied.add(c)
                remaining.remove(c)
        return order
    
    process_order = topo_sort(deps)
    all_chunk_refs = list(gen.chunk_refs(max_ring=2))
    
    # Generate chunks in order
    chunks = []
    for chunk_idx in process_order[:5]:
        chunk = gen.generate_chunk(all_chunk_refs[chunk_idx])
        for prev_chunk in chunks:
            chunk.copy_from(prev_chunk)
        chunks.append(chunk)
    
    chunk_set = ChunkSet(chunks, coarse, gen.scale)
    
    # Convert to terrain
    terrain = chunk_set.to_terrain(fill_gaps=True)
    terrain.hexGrid.adjustRadius(15)
    terrain.colorLevels = coarse.colorLevels
    terrain.seaLevel = coarse.seaLevel
    terrain.elevationDelta = coarse.elevationDelta
    
    for style in terrain.colorLevels:
        terrain.hexGrid.builder.add_style(style)
    terrain.hexGrid.builder.add_style(terrain.seaLevel)
    
    # === MARK INVALID HEXES ===
    # Get bounds from chunk_set positions
    positions = list(chunk_set.pos_to_chunk.keys())
    min_q = min(p.q for p in positions)
    min_r = min(p.r for p in positions)
    origin_world = HexPosition(min_q, min_r, -min_q - min_r)
    
    # Mark hexes that aren't covered by any chunk
    covered_indices = set()
    for world_pos in positions:
        local_pos = world_pos - origin_world
        grid_idx = terrain.hexGrid.hexposition_to_index(local_pos, origin_index=0)
        if 0 <= grid_idx < len(terrain.hexGrid.hexes):
            covered_indices.add(grid_idx)
    
    # All indices NOT in covered_indices are invalid
    all_indices = set(range(len(terrain.hexGrid.hexes)))
    terrain.hexGrid.invalidRegion = all_indices - covered_indices
    
    print(f"Total hexes: {len(all_indices)}")
    print(f"Covered by chunks: {len(covered_indices)}")
    print(f"Marked invalid: {len(terrain.hexGrid.invalidRegion)}")
    
    terrain.colorMap()
    terrain.hexGrid.update()

    # Use filtered versions
    # Use filtered versions
    flow_overlay = terrain.flow_diagram_filtered(covered_indices)
    terrain.hexGrid.builder.adjust("flow", flow_overlay)

    local_minima = find_local_minima_filtered(terrain, covered_indices)
    print(f"Local minima in valid region: {len(local_minima)}")

    # Create minima overlay
    minima_style = StyleCSS("minima_dot", fill="#ff0000", stroke="white", stroke_width=1)
    terrain.hexGrid.builder.add_style(minima_style)

    minima_overlay = ""
    for idx in local_minima:
        hex_obj = terrain.hexGrid.hexes[idx]
        minima_overlay += f'<circle cx="{hex_obj.center.x}" cy="{hex_obj.center.y}" r="4" class="minima_dot"/>\n'

    terrain.hexGrid.builder.adjust("minima", minima_overlay)
    terrain.hexGrid.builder.adjust("flow", flow_overlay)

    
    terrain.hexGrid.builder.show()


## chunck Set idea

## going to fix the drain

can you wire thi scarve_chunk_drainage and others up with the dependency order and test on the 5-chunk region? can you have it so that it patches chunkset , ChunkGraph, ChunkedTerrainGenerator when possible

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
import heapq
from dataclasses import dataclass
from typing import Iterator

# === ChunkGraph: Formalized dependency structure ===

@dataclass
class ChunkGraph:
    """Reusable structure for ordered chunk traversal."""
    chunk_refs: list[ChunkRef]
    dependencies: dict[int, set[int]]  # chunk_idx -> depends on these
    process_order: list[int]           # topological sort
    chunk_info: dict  # drainage entry/exit info
    
    @classmethod
    def from_coarse(cls, gen: 'ChunkedTerrainGenerator', coarse: Terrain, max_ring: int = 2):
        """Build chunk graph from coarse drainage analysis."""
        all_refs = list(gen.chunk_refs(max_ring=max_ring))
        
        # Get drainage info
        _, chunk_info = cls._analyze_drainage(gen, coarse, all_refs)
        
        # Build dependencies
        flow_dir = coarse.flow_directions()
        deps = cls._compute_dependencies(chunk_info, flow_dir, coarse)
        
        # Topological sort
        order = cls._topo_sort(deps)
        
        return cls(chunk_refs=all_refs, dependencies=deps, 
                   process_order=order, chunk_info=chunk_info)
    
    @staticmethod
    def _analyze_drainage(gen, coarse, all_refs):
        """Analyze which chunks have drainage."""
        drainage_hexes = set()
        chunk_info = {}
        
        for i, chunk_ref in enumerate(all_refs):
            center = chunk_ref.center_hex
            coarse_q = int(round(center.q / gen.scale))
            coarse_r = int(round(center.r / gen.scale))
            coarse_s = -coarse_q - coarse_r
            coarse_pos = HexPosition(coarse_q, coarse_r, coarse_s)
            
            chunk_radius_coarse = int(gen.chunk_rings / gen.scale) + 1
            
            enters, exits = [], []
            has_drainage = False
            flow_dir = coarse.flow_directions()
            
            for ring in range(chunk_radius_coarse + 1):
                positions = [coarse_pos] if ring == 0 else list(coarse_pos.ring(ring))
                for pos in positions:
                    idx = coarse.hexGrid.hexposition_to_index(pos, origin_index=coarse.hexGrid.midpoint)
                    if idx >= 0 and coarse.elevations[idx] > 0:
                        has_drainage = True
                        if ring == chunk_radius_coarse and flow_dir[idx] >= 0:
                            next_pos = coarse.hexGrid.index_to_hexposition(
                                flow_dir[idx], origin_index=coarse.hexGrid.midpoint)
                            if next_pos.distance(coarse_pos) < pos.distance(coarse_pos):
                                enters.append(idx)
                            else:
                                exits.append(idx)
            
            chunk_info[i] = {'enters': enters, 'exits': exits, 'has_drainage': has_drainage}
        
        return drainage_hexes, chunk_info
    
    @staticmethod
    def _compute_dependencies(chunk_info, flow_directions, coarse):
        """Build dependency graph from drainage flow."""
        dependencies = {i: set() for i in chunk_info.keys()}
        
        for chunk_i, info in chunk_info.items():
            for exit_idx in info['exits']:
                current = exit_idx
                while flow_directions[current] >= 0:
                    next_hex = flow_directions[current]
                    for chunk_j, info_j in chunk_info.items():
                        if chunk_j != chunk_i and next_hex in info_j['enters']:
                            dependencies[chunk_j].add(chunk_i)
                            break
                    current = next_hex
        
        return dependencies
    
    @staticmethod
    def _topo_sort(deps):
        """Topological sort of chunks."""
        order, remaining, satisfied = [], set(deps.keys()), set()
        while remaining:
            ready = [c for c in remaining if deps[c] <= satisfied] or [min(remaining)]
            for c in sorted(ready):
                order.append(c)
                satisfied.add(c)
                remaining.remove(c)
        return order
    
    def iter_ordered(self) -> Iterator[tuple[int, ChunkRef]]:
        """Yield (index, ChunkRef) in dependency order."""
        for idx in self.process_order:
            yield idx, self.chunk_refs[idx]
    
    def processed_before(self, chunk_idx: int) -> list[int]:
        """Which chunks are processed before this one."""
        my_pos = self.process_order.index(chunk_idx)
        return self.process_order[:my_pos]


# === ChunkSet patches for drainage ===

@patch
def find_local_minima(self: ChunkSet) -> list[tuple[HexPosition, float]]:
    """Find local minima in CORE hexes only."""
    minima = []
    for pos in self.iter_positions():  # Core only
        elev = self.elevation_at(pos)
        if elev <= 0:
            continue
        
        lowest = self.lowest_neighbor(pos)
        lowest_elev = self.elevation_at(lowest)
        
        if lowest_elev >= elev:
            minima.append((pos, elev))
    
    return minima


@patch
def find_drainage_path(self: ChunkSet, 
                       start: HexPosition,
                       core_positions: set = None) -> list[HexPosition]:
    """Dijkstra path from start to ocean or halo exit."""
    if core_positions is None:
        core_positions = set(self.iter_positions())
    
    def is_target(pos):
        return self.elevation_at(pos) <= 0 or pos not in core_positions
    
    if is_target(start):
        return [start]
    
    pq = [(0, id(start), start, [start])]  # Added id for tie-breaking
    visited = set()
    
    while pq:
        cost, _, current, path = heapq.heappop(pq)
        
        if current in visited:
            continue
        visited.add(current)
        
        if is_target(current):
            return path
        
        for neighbor in self.neighbors_of(current):
            if neighbor in visited:
                continue
            
            curr_elev = self.elevation_at(current)
            neigh_elev = self.elevation_at(neighbor)
            
            carve_cost = max(0, neigh_elev - curr_elev + 1)
            
            heapq.heappush(pq, (cost + carve_cost, id(neighbor), neighbor, path + [neighbor]))
    
    return []


@patch
def carve_path(self: ChunkSet, path: list[HexPosition], core_positions: set = None):
    """Carve elevation along path, only modifying core hexes."""
    if core_positions is None:
        core_positions = set(self.iter_positions())
    
    if len(path) < 2:
        return
    
    for i in range(len(path) - 1):
        curr, next_pos = path[i], path[i + 1]
        
        curr_elev = self.elevation_at(curr)
        next_elev = self.elevation_at(next_pos)
        
        if next_elev >= curr_elev and next_pos in core_positions:
            self.set_elevation(next_pos, curr_elev - 1)


@patch
def carve_drainage(self: ChunkSet, max_lakes: int = 2) -> list[list[HexPosition]]:
    """Carve drainage paths within ChunkSet, respecting core/halo boundaries."""
    core_positions = set(self.iter_positions())
    
    # Find and sort minima
    minima = self.find_local_minima()
    minima.sort(key=lambda x: x[1], reverse=True)
    
    # Keep highest as lakes, drain the rest
    drain_these = minima[max_lakes:]
    
    carved_paths = []
    for start_pos, _ in drain_these:
        path = self.find_drainage_path(start_pos, core_positions)
        
        if len(path) >= 2:
            self.carve_path(path, core_positions)
            carved_paths.append(path)
    
    return carved_paths


# === ChunkedTerrainGenerator patches ===

@patch
def build_chunk_graph(self: ChunkedTerrainGenerator, max_ring: int = 2) -> ChunkGraph:
    """Build dependency graph from coarse terrain."""
    if self._coarse_terrain is None:
        raise ValueError("Must generate coarse terrain first")
    return ChunkGraph.from_coarse(self, self._coarse_terrain, max_ring)


@patch
def process_chunks_with_drainage(self: ChunkedTerrainGenerator,
                                  graph: ChunkGraph,
                                  chunk_indices: list[int] = None,
                                  max_lakes_per_chunk: int = 1) -> tuple[ChunkSet, list]:
    """Generate and carve chunks in dependency order."""
    
    if chunk_indices is None:
        chunk_indices = graph.process_order
    
    generated = {}  # chunk_idx -> HexChunk
    all_paths = []  # (chunk_idx, paths)
    
    for chunk_idx in graph.process_order:
        if chunk_idx not in chunk_indices:
            continue
        
        # Generate chunk
        chunk_ref = graph.chunk_refs[chunk_idx]
        chunk = self.generate_chunk(chunk_ref)
        
        # Copy halo from already-processed neighbors
        for prev_idx in graph.processed_before(chunk_idx):
            if prev_idx in generated:
                chunk.copy_from(generated[prev_idx])
        
        generated[chunk_idx] = chunk
        
        # Build ChunkSet with just this chunk + neighbors for drainage
        neighbor_chunks = [generated[i] for i in graph.processed_before(chunk_idx) 
                          if i in generated]
        chunk_set = ChunkSet([chunk] + neighbor_chunks, self._coarse_terrain, self.scale)
        
        # Carve drainage
        paths = chunk_set.carve_drainage(max_lakes=max_lakes_per_chunk)
        if paths:
            all_paths.append((chunk_idx, paths))
        
        print(f"Chunk {chunk_idx}: {chunk.core_size} hexes, {len(paths)} drainage paths carved")
    
    # Build final ChunkSet from all generated chunks
    final_chunks = [generated[i] for i in chunk_indices if i in generated]
    final_set = ChunkSet(final_chunks, self._coarse_terrain, self.scale)
    
    return final_set, all_paths
```

Now let's test it:

```python
# === TEST ===
with GeoStorageDebugger(keep_on_error=False) as debugger:
    gen = ChunkedTerrainGenerator(
        storage=debugger.server,
        coarse_rings=15, coarse_radius=50, fine_radius=10,
        chunk_rings=8, halo_rings=3
    )
    
    # Generate coarse with drainage
    coarse = gen.generate_coarse(num_plates=8, ocean_fraction=0.35, oceanic_sides=['W'], seed=42)
    coarse.carve_to_ocean(num_lakes=3, max_iters=5)
    print("Coarse drainage carved")
    
    # Build dependency graph
    graph = gen.build_chunk_graph(max_ring=2)
    print(f"\nChunk graph: {len(graph.chunk_refs)} chunks")
    print(f"Process order: {graph.process_order[:10]}...")
    
    # Process first 5 chunks with drainage
    test_indices = graph.process_order[:5]
    print(f"\nProcessing chunks: {test_indices}")
    
    chunk_set, all_paths = gen.process_chunks_with_drainage(
        graph, 
        chunk_indices=test_indices,
        max_lakes_per_chunk=1
    )
    
    print(f"\nTotal paths carved: {sum(len(p) for _, p in all_paths)}")
    
    # Visualize
    terrain = chunk_set.to_terrain(fill_gaps=True)
    terrain.colorLevels = coarse.colorLevels
    terrain.seaLevel = coarse.seaLevel
    terrain.elevationDelta = coarse.elevationDelta
    
    for style in terrain.colorLevels:
        terrain.hexGrid.builder.add_style(style)
    terrain.hexGrid.builder.add_style(terrain.seaLevel)
    
    terrain.hexGrid.adjustRadius(12)
    terrain.colorMap()
    terrain.hexGrid.update()
    
    # Mark covered region
    positions = list(chunk_set.pos_to_chunk.keys())
    min_q = min(p.q for p in positions)
    min_r = min(p.r for p in positions)
    origin_world = HexPosition(min_q, min_r, -min_q - min_r)
    
    covered_indices = set()
    for world_pos in positions:
        local_pos = world_pos - origin_world
        grid_idx = terrain.hexGrid.hexposition_to_index(local_pos, origin_index=0)
        if 0 <= grid_idx < len(terrain.hexGrid.hexes):
            covered_indices.add(grid_idx)
    
    terrain.hexGrid.invalidRegion = set(range(len(terrain.hexGrid.hexes))) - covered_indices
    terrain.colorMap()
    terrain.hexGrid.update()
    
    # Add flow diagram
    flow_overlay = terrain.flow_diagram_filtered(covered_indices)
    terrain.hexGrid.builder.adjust("flow", flow_overlay)
    
    # Draw carved paths
    path_style = StyleCSS("carved_path", fill="none", stroke="#0066ff", stroke_width=3)
    terrain.hexGrid.builder.add_style(path_style)
    
    path_overlay = ""
    for chunk_idx, paths in all_paths:
        for path in paths:
            if len(path) >= 2:
                points = []
                for pos in path:
                    local_pos = pos - origin_world
                    grid_idx = terrain.hexGrid.hexposition_to_index(local_pos, origin_index=0)
                    if 0 <= grid_idx < len(terrain.hexGrid.hexes):
                        hex_obj = terrain.hexGrid.hexes[grid_idx]
                        points.append(f"{hex_obj.center.x},{hex_obj.center.y}")
                
                if len(points) >= 2:
                    path_overlay += f'<polyline points="{" ".join(points)}" class="carved_path"/>\n'
    
    terrain.hexGrid.builder.adjust("carved_paths", path_overlay)
    
    # Mark minima (post-carving)
    minima = chunk_set.find_local_minima()
    minima_overlay = ""
    for pos, elev in minima:
        local_pos = pos - origin_world
        grid_idx = terrain.hexGrid.hexposition_to_index(local_pos, origin_index=0)
        if 0 <= grid_idx < len(terrain.hexGrid.hexes):
            hex_obj = terrain.hexGrid.hexes[grid_idx]
            minima_overlay += f'<circle cx="{hex_obj.center.x}" cy="{hex_obj.center.y}" r="5" fill="red" stroke="white" stroke-width="1"/>\n'
    
    terrain.hexGrid.builder.adjust("minima", minima_overlay)
    
    textStyle = StyleCSS("textStyle", fill="white", font_size="1.2em", 
                         filter="drop-shadow(2px 2px 1px rgba(0,0,0,.7))")
    terrain.hexGrid.builder.add_style(textStyle)
    terrain.hexGrid.builder.add_centered_text(
        f"5-Chunk Drainage: {len(minima)} lakes (red), carved paths (blue)", 
        y_offset=-terrain.hexGrid.builder.height/2 + 30, 
        class_name=textStyle.name
    )
    
    terrain.hexGrid.builder.show()
```

This wires up:
- **ChunkGraph** - formalizes dependency structure with `from_coarse()` factory
- **ChunkSet patches** - `find_local_minima()`, `find_drainage_path()`, `carve_path()`, `carve_drainage()`
- **ChunkedTerrainGenerator patches** - `build_chunk_graph()`, `process_chunks_with_drainage()`

## Chunk Graph

In [ ]:
#| export
 #=== ChunkGraph: Formalized dependency structure ===

@dataclass
class ChunkGraph:
    """Reusable structure for ordered chunk traversal."""
    chunk_refs: list[ChunkRef]
    dependencies: dict[int, set[int]]  # chunk_idx -> depends on these
    process_order: list[int]           # topological sort
    chunk_info: dict  # drainage entry/exit info
    
    @classmethod
    def from_coarse(cls, gen: 'ChunkedTerrainGenerator', coarse: Terrain, max_ring: int = 2):
        """Build chunk graph from coarse drainage analysis."""
        all_refs = list(gen.chunk_refs(max_ring=max_ring))
        
        # Get drainage info
        _, chunk_info = cls._analyze_drainage(gen, coarse, all_refs)
        
        # Build dependencies
        flow_dir = coarse.flow_directions()
        deps = cls._compute_dependencies(chunk_info, flow_dir, coarse)
        
        # Topological sort
        order = cls._topo_sort(deps)
        
        return cls(chunk_refs=all_refs, dependencies=deps, 
                   process_order=order, chunk_info=chunk_info)
    
    @staticmethod
    def _analyze_drainage(gen, coarse, all_refs):
        """Analyze which chunks have drainage."""
        drainage_hexes = set()
        chunk_info = {}
        
        for i, chunk_ref in enumerate(all_refs):
            center = chunk_ref.center_hex
            coarse_q = int(round(center.q / gen.scale))
            coarse_r = int(round(center.r / gen.scale))
            coarse_s = -coarse_q - coarse_r
            coarse_pos = HexPosition(coarse_q, coarse_r, coarse_s)
            
            chunk_radius_coarse = int(gen.chunk_rings / gen.scale) + 1
            
            enters, exits = [], []
            has_drainage = False
            flow_dir = coarse.flow_directions()
            
            for ring in range(chunk_radius_coarse + 1):
                positions = [coarse_pos] if ring == 0 else list(coarse_pos.ring(ring))
                for pos in positions:
                    idx = coarse.hexGrid.hexposition_to_index(pos, origin_index=coarse.hexGrid.midpoint)
                    if idx >= 0 and coarse.elevations[idx] > 0:
                        has_drainage = True
                        if ring == chunk_radius_coarse and flow_dir[idx] >= 0:
                            next_pos = coarse.hexGrid.index_to_hexposition(
                                flow_dir[idx], origin_index=coarse.hexGrid.midpoint)
                            if next_pos.distance(coarse_pos) < pos.distance(coarse_pos):
                                enters.append(idx)
                            else:
                                exits.append(idx)
            
            chunk_info[i] = {'enters': enters, 'exits': exits, 'has_drainage': has_drainage}
        
        return drainage_hexes, chunk_info
    
    @staticmethod
    def _compute_dependencies(chunk_info, flow_directions, coarse):
        """Build dependency graph from drainage flow."""
        dependencies = {i: set() for i in chunk_info.keys()}
        
        for chunk_i, info in chunk_info.items():
            for exit_idx in info['exits']:
                current = exit_idx
                while flow_directions[current] >= 0:
                    next_hex = flow_directions[current]
                    for chunk_j, info_j in chunk_info.items():
                        if chunk_j != chunk_i and next_hex in info_j['enters']:
                            dependencies[chunk_j].add(chunk_i)
                            break
                    current = next_hex
        
        return dependencies
    
    @staticmethod
    def _topo_sort(deps):
        """Topological sort of chunks."""
        order, remaining, satisfied = [], set(deps.keys()), set()
        while remaining:
            ready = [c for c in remaining if deps[c] <= satisfied] or [min(remaining)]
            for c in sorted(ready):
                order.append(c)
                satisfied.add(c)
                remaining.remove(c)
        return order
    
    def iter_ordered(self) -> Iterator[tuple[int, ChunkRef]]:
        """Yield (index, ChunkRef) in dependency order."""
        for idx in self.process_order:
            yield idx, self.chunk_refs[idx]
    
    def processed_before(self, chunk_idx: int) -> list[int]:
        """Which chunks are processed before this one."""
        my_pos = self.process_order.index(chunk_idx)
        return self.process_order[:my_pos]


In [ ]:
#| export
# === ChunkSet patches for drainage ===

@patch
def find_local_minima(self: ChunkSet) -> list[tuple[HexPosition, float]]:
    """Find local minima in CORE hexes only."""
    minima = []
    for pos in self.iter_positions():  # Core only
        elev = self.elevation_at(pos)
        if elev <= 0:
            continue
        
        lowest = self.lowest_neighbor(pos)
        lowest_elev = self.elevation_at(lowest)
        
        if lowest_elev >= elev:
            minima.append((pos, elev))
    
    return minima


@patch
def find_drainage_path(self: ChunkSet, 
                       start: HexPosition,
                       core_positions: set = None) -> list[HexPosition]:
    """Dijkstra path from start to ocean or halo exit."""
    if core_positions is None:
        core_positions = set(self.iter_positions())
    
    def is_target(pos):
        return self.elevation_at(pos) <= 0 or pos not in core_positions
    
    if is_target(start):
        return [start]
    
    pq = [(0, id(start), start, [start])]  # Added id for tie-breaking
    visited = set()
    
    while pq:
        cost, _, current, path = heapq.heappop(pq)
        
        if current in visited:
            continue
        visited.add(current)
        
        if is_target(current):
            return path
        
        for neighbor in self.neighbors_of(current):
            if neighbor in visited:
                continue
            
            curr_elev = self.elevation_at(current)
            neigh_elev = self.elevation_at(neighbor)
            
            carve_cost = max(0, neigh_elev - curr_elev + 1)
            
            heapq.heappush(pq, (cost + carve_cost, id(neighbor), neighbor, path + [neighbor]))
    
    return []


@patch
def carve_path(self: ChunkSet, path: list[HexPosition], core_positions: set = None):
    """Carve elevation along path, only modifying core hexes."""
    if core_positions is None:
        core_positions = set(self.iter_positions())
    
    if len(path) < 2:
        return
    
    for i in range(len(path) - 1):
        curr, next_pos = path[i], path[i + 1]
        
        curr_elev = self.elevation_at(curr)
        next_elev = self.elevation_at(next_pos)
        
        if next_elev >= curr_elev and next_pos in core_positions:
            self.set_elevation(next_pos, curr_elev - 1)


@patch
def carve_drainage(self: ChunkSet, max_lakes: int = 2) -> list[list[HexPosition]]:
    """Carve drainage paths within ChunkSet, respecting core/halo boundaries."""
    core_positions = set(self.iter_positions())
    
    # Find and sort minima
    minima = self.find_local_minima()
    minima.sort(key=lambda x: x[1], reverse=True)
    
    # Keep highest as lakes, drain the rest
    drain_these = minima[max_lakes:]
    
    carved_paths = []
    for start_pos, _ in drain_these:
        path = self.find_drainage_path(start_pos, core_positions)
        
        if len(path) >= 2:
            self.carve_path(path, core_positions)
            carved_paths.append(path)
    
    return carved_paths

In [ ]:
#| export
# === ChunkedTerrainGenerator patches ===

@patch
def build_chunk_graph(self: ChunkedTerrainGenerator, max_ring: int = 2) -> ChunkGraph:
    """Build dependency graph from coarse terrain."""
    if self._coarse_terrain is None:
        raise ValueError("Must generate coarse terrain first")
    return ChunkGraph.from_coarse(self, self._coarse_terrain, max_ring)


@patch
def process_chunks_with_drainage(self: ChunkedTerrainGenerator,
                                  graph: ChunkGraph,
                                  chunk_indices: list[int] = None,
                                  max_lakes_per_chunk: int = 1) -> tuple[ChunkSet, list]:
    """Generate and carve chunks in dependency order."""
    
    if chunk_indices is None:
        chunk_indices = graph.process_order
    
    generated = {}  # chunk_idx -> HexChunk
    all_paths = []  # (chunk_idx, paths)
    
    for chunk_idx in graph.process_order:
        if chunk_idx not in chunk_indices:
            continue
        
        # Generate chunk
        chunk_ref = graph.chunk_refs[chunk_idx]
        chunk = self.generate_chunk(chunk_ref)
        
        # Copy halo from already-processed neighbors
        for prev_idx in graph.processed_before(chunk_idx):
            if prev_idx in generated:
                chunk.copy_from(generated[prev_idx])
        
        generated[chunk_idx] = chunk
        
        # Build ChunkSet with just this chunk + neighbors for drainage
        neighbor_chunks = [generated[i] for i in graph.processed_before(chunk_idx) 
                          if i in generated]
        chunk_set = ChunkSet([chunk] + neighbor_chunks, self._coarse_terrain, self.scale)
        
        # Carve drainage
        paths = chunk_set.carve_drainage(max_lakes=max_lakes_per_chunk)
        if paths:
            all_paths.append((chunk_idx, paths))
        
        print(f"Chunk {chunk_idx}: {chunk.core_size} hexes, {len(paths)} drainage paths carved")
    
    # Build final ChunkSet from all generated chunks
    final_chunks = [generated[i] for i in chunk_indices if i in generated]
    final_set = ChunkSet(final_chunks, self._coarse_terrain, self.scale)
    
    return final_set, all_paths


In [ ]:
#| export
# === TEST ===
with GeoStorageDebugger(keep_on_error=False) as debugger:
    gen = ChunkedTerrainGenerator(
        storage=debugger.server,
        coarse_rings=15, coarse_radius=50, fine_radius=10,
        chunk_rings=8, halo_rings=3
    )
    
    # Generate coarse with drainage
    coarse = gen.generate_coarse(num_plates=8, ocean_fraction=0.35, oceanic_sides=['W'], seed=42)
    coarse.carve_to_ocean(num_lakes=3, max_iters=5)
    print("Coarse drainage carved")
    
    # Build dependency graph
    graph = gen.build_chunk_graph(max_ring=2)
    print(f"\nChunk graph: {len(graph.chunk_refs)} chunks")
    print(f"Process order: {graph.process_order[:10]}...")
    
    # Process first 5 chunks with drainage
    test_indices = graph.process_order[:5]
    print(f"\nProcessing chunks: {test_indices}")
    
    chunk_set, all_paths = gen.process_chunks_with_drainage(
        graph, 
        chunk_indices=test_indices,
        max_lakes_per_chunk=1
    )
    
    print(f"\nTotal paths carved: {sum(len(p) for _, p in all_paths)}")
    
    # Visualize
    terrain = chunk_set.to_terrain(fill_gaps=True)
    terrain.colorLevels = coarse.colorLevels
    terrain.seaLevel = coarse.seaLevel
    terrain.elevationDelta = coarse.elevationDelta
    
    for style in terrain.colorLevels:
        terrain.hexGrid.builder.add_style(style)
    terrain.hexGrid.builder.add_style(terrain.seaLevel)
    
    terrain.hexGrid.adjustRadius(12)
    terrain.colorMap()
    terrain.hexGrid.update()
    
    # Mark covered region
    positions = list(chunk_set.pos_to_chunk.keys())
    min_q = min(p.q for p in positions)
    min_r = min(p.r for p in positions)
    origin_world = HexPosition(min_q, min_r, -min_q - min_r)
    
    covered_indices = set()
    for world_pos in positions:
        local_pos = world_pos - origin_world
        grid_idx = terrain.hexGrid.hexposition_to_index(local_pos, origin_index=0)
        if 0 <= grid_idx < len(terrain.hexGrid.hexes):
            covered_indices.add(grid_idx)
    
    terrain.hexGrid.invalidRegion = set(range(len(terrain.hexGrid.hexes))) - covered_indices
    terrain.colorMap()
    terrain.hexGrid.update()
    
    # Add flow diagram
    flow_overlay = terrain.flow_diagram_filtered(covered_indices)
    terrain.hexGrid.builder.adjust("flow", flow_overlay)
    
    # Draw carved paths
    path_style = StyleCSS("carved_path", fill="none", stroke="#0066ff", stroke_width=3)
    terrain.hexGrid.builder.add_style(path_style)
    
    path_overlay = ""
    for chunk_idx, paths in all_paths:
        for path in paths:
            if len(path) >= 2:
                points = []
                for pos in path:
                    local_pos = pos - origin_world
                    grid_idx = terrain.hexGrid.hexposition_to_index(local_pos, origin_index=0)
                    if 0 <= grid_idx < len(terrain.hexGrid.hexes):
                        hex_obj = terrain.hexGrid.hexes[grid_idx]
                        points.append(f"{hex_obj.center.x},{hex_obj.center.y}")
                
                if len(points) >= 2:
                    path_overlay += f'<polyline points="{" ".join(points)}" class="carved_path"/>\n'
    
    terrain.hexGrid.builder.adjust("carved_paths", path_overlay)
    
    # Mark minima (post-carving)
    minima = chunk_set.find_local_minima()
    minima_overlay = ""
    for pos, elev in minima:
        local_pos = pos - origin_world
        grid_idx = terrain.hexGrid.hexposition_to_index(local_pos, origin_index=0)
        if 0 <= grid_idx < len(terrain.hexGrid.hexes):
            hex_obj = terrain.hexGrid.hexes[grid_idx]
            minima_overlay += f'<circle cx="{hex_obj.center.x}" cy="{hex_obj.center.y}" r="5" fill="red" stroke="white" stroke-width="1"/>\n'
    
    terrain.hexGrid.builder.adjust("minima", minima_overlay)
    
    textStyle = StyleCSS("textStyle", fill="white", font_size="1.2em", 
                         filter="drop-shadow(2px 2px 1px rgba(0,0,0,.7))")
    terrain.hexGrid.builder.add_style(textStyle)
    terrain.hexGrid.builder.add_centered_text(
        f"5-Chunk Drainage: {len(minima)} lakes (red), carved paths (blue)", 
        y_offset=-terrain.hexGrid.builder.height/2 + 30, 
        class_name=textStyle.name
    )
    
    terrain.hexGrid.builder.show()


That is much better

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Excellent! The key wins here:

1. **Dependency ordering works** - chunks process upstream→downstream so halos contain correct carved elevations
2. **58 drainage paths** carved across 5 chunks, connecting through boundaries
3. **Core/halo separation** - we only modify core hexes, read from halos
4. **ChunkGraph** gives us a reusable traversal structure

The remaining lakes (red dots) are the intentional `max_lakes_per_chunk=1` survivors - those become actual lakes rather than getting drained.

**Next steps when you're ready:**
- Weather propagation (same dependency ordering, wind direction instead of drainage)
- Watersheds (flood-fill from drainage paths, respecting chunk boundaries)
- Persist carved chunks back to storage so we don't regenerate

The `ChunkGraph` structure will serve all of these - just swap the ordering criteria (drainage → wind → whatever).

## Going up and down.

In many ways trying to think about the chunking and expanding was its own rabbit hole, or more to the point of alice in wonderland of a series of potions that said Eat Me or Drink me.
We need to debug how all of this going. a few things
1. downsample and upsample work but you need to think about the parameters
2. Sealevel is a StyleCSS and not a float. Sealevel is always at 0, but we might different colors (though not as the red or black seas)
3. It might be a case that a child might have hexes that won't map to the parent. when we create a child we should also have two regions - one invalid hexes in it and two a region in the parent where it was extracted. 
4. some algorithms need to make sure that we don't use invalid hexes.
5. The largest a terrain that works as a view port is about 3,000 hexes. we should have things that scale terrain to approximately those sizes. we probaly could handle 6,000 hexes (viewed) but not 60,000
6. red blog https://www.redblobgames.com/grids/hexagons/#pathfinding talks about about algorithms that go up and down in scale. It make sense for path finding (where you do global paths, before dropping down and doing indivual hexes. We are going to need these kinds of things a great deal for merging and wandering. in particular our watershed algorim is n^2 sp we need to reduce the n to just the itmes.
7. I do have more guides about hexpositions, but one rule of thumb is the more you are in that system the better things should work. the x/y is really only for rendering and not concepts.
8. The other system is just a list of points (the fields inside terrain for instance. my other rule of thumb is that numpy makes things better. If we are doing math calcs we should consider it


In [ ]:
#| export
# === ChunkSet weather methods ===


@patch
def apply_local_temperature(self: ChunkSet, 
                            coarse: Terrain,
                            lapse_rate: float = 6.5):
    """Adjust temperature based on fine-scale elevation differences."""
    
    for world_pos, (chunk, idx, is_core) in self.pos_to_chunk.items():
        if not is_core:
            continue
        
        if 'temperature' not in chunk.fields:
            continue
        
        fine_elev = chunk.elevations[idx]
        
        # Get coarse elevation for comparison
        coarse_elev = self._query_coarse(world_pos)
        
        # Elevation difference drives temperature adjustment
        elev_diff = fine_elev - coarse_elev  # Positive = higher than coarse
        temp_adjust = -(elev_diff / 1000.0) * lapse_rate
        
        chunk.fields['temperature'][idx] += temp_adjust


@patch  
def apply_local_rain_shadow(self: ChunkSet,
                            wind_dir: float = 270.0,
                            shadow_factor: float = 0.3):
    """Apply local rain shadow from fine-scale terrain.
    
    Hexes downwind of local peaks get reduced precipitation.
    """
    # Wind direction to check upwind
    wind_rad = math.radians(wind_dir)
    # Find which hex direction is most upwind
    upwind_dirs = []
    for d in range(6):
        neighbor = HexPosition.origin().neighbor(d)
        angle = math.atan2(neighbor.r, neighbor.q)
        # How aligned is this direction with incoming wind?
        alignment = math.cos(angle - wind_rad + math.pi)  # +pi because wind comes FROM
        upwind_dirs.append((d, alignment))
    
    upwind_dirs.sort(key=lambda x: x[1], reverse=True)
    primary_upwind = upwind_dirs[0][0]
    secondary_upwind = upwind_dirs[1][0]
    
    for world_pos, (chunk, idx, is_core) in self.pos_to_chunk.items():
        if not is_core:
            continue
        
        if 'precipitation' not in chunk.fields:
            continue
        
        my_elev = chunk.elevations[idx]
        
        # Check upwind neighbors
        upwind_pos = world_pos.neighbor(primary_upwind)
        upwind_elev = self.elevation_at(upwind_pos)
        
        # If upwind is significantly higher, we're in shadow
        elev_diff = upwind_elev - my_elev
        
        if elev_diff > 50:  # Upwind hex is >50m higher
            # Reduce precipitation
            shadow_strength = min(1.0, elev_diff / 500.0)  # Max effect at 500m diff
            reduction = 1.0 - (shadow_factor * shadow_strength)
            chunk.fields['precipitation'][idx] *= reduction


@patch
def compute_weather(self: ChunkSet, 
                    coarse: Terrain,
                    scale: float,
                    lapse_rate: float = 6.5,
                    wind_dir: float = 270.0):
    """Full weather computation for ChunkSet."""
    
    # 1. Interpolate from coarse
    self.interpolate_weather_from_coarse(coarse, scale)
    
    # 2. Local temperature adjustment
    self.apply_local_temperature(coarse, lapse_rate)
    
    # 3. Local rain shadow
    self.apply_local_rain_shadow(wind_dir)

In [ ]:
#| export
@patch
def interpolate_weather_from_coarse(self: ChunkSet, 
                                     coarse: Terrain,
                                     scale: float,
                                     sigma: float = 0.8):
    """Interpolate coarse weather to all chunk positions (core AND halo).
    
    Args:
        coarse: Terrain with computed weather fields
        scale: Resolution ratio (coarse_radius / fine_radius)
        sigma: Gaussian smoothing parameter
    """
    
    coarse_grid = coarse.hexGrid
    has_temp = 'temperature' in coarse.fields
    has_precip = 'precipitation' in coarse.fields
    
    if not has_temp or not has_precip:
        raise ValueError("Coarse terrain must have temperature and precipitation computed")
    
    for world_pos, (chunk, idx, is_core) in self.pos_to_chunk.items():
        # Process ALL hexes (core AND halo) - removed is_core check
        
        # Skip ocean hexes - they don't need weather
        if chunk.elevations[idx] <= 0:
            continue
        
        # Initialize fields if needed
        if 'temperature' not in chunk.fields:
            chunk.add_field('temperature', default=15.0)
        if 'precipitation' not in chunk.fields:
            chunk.add_field('precipitation', default=500.0)
        
        # Map to fractional coarse coordinates
        frac_q = world_pos.q / scale
        frac_r = world_pos.r / scale
        
        # Nearest coarse hex center
        near_q = int(round(frac_q))
        near_r = int(round(frac_r))
        near_s = -near_q - near_r
        center = HexPosition(near_q, near_r, near_s)
        
        # Sample center + ring 1 + ring 2 (19 hexes for smooth blending)
        candidates = [center] + list(center.ring(1)) + list(center.ring(2))
        
        temp_weights = []
        temp_values = []
        precip_weights = []
        precip_values = []
        
        for cpos in candidates:
            coarse_idx = coarse_grid.hexposition_to_index(cpos, origin_index=coarse_grid.midpoint)
            
            if coarse_idx < 0 or coarse_idx >= len(coarse.elevations):
                continue
            
            # Skip coarse ocean hexes for land interpolation
            if coarse.elevations[coarse_idx] <= 0:
                continue
            
            # Squared distance in fractional coarse space
            dq = cpos.q - frac_q
            dr = cpos.r - frac_r
            dist_sq = dq**2 + dr**2 + dq*dr  # Hex metric
            
            # Gaussian falloff
            w = math.exp(-dist_sq / (2 * sigma**2))
            
            temp_weights.append(w)
            temp_values.append(coarse.fields['temperature'][coarse_idx])
            precip_weights.append(w)
            precip_values.append(coarse.fields['precipitation'][coarse_idx])
        
        # Compute weighted averages
        if temp_weights:
            total_w = sum(temp_weights)
            chunk.fields['temperature'][idx] = sum(w * v for w, v in zip(temp_weights, temp_values)) / total_w
            chunk.fields['precipitation'][idx] = sum(w * v for w, v in zip(precip_weights, precip_values)) / total_w
        else:
            # Fallback: no valid coarse neighbors found (edge of map)
            if coarse.climate:
                chunk.fields['temperature'][idx] = sum(coarse.climate.base_temp_range) / 2
                chunk.fields['precipitation'][idx] = coarse.climate.precip_bins[1] if coarse.climate.precip_bins else 500.0
            else:
                chunk.fields['temperature'][idx] = 15.0
                chunk.fields['precipitation'][idx] = 500.0


In [ ]:
#| export
@patch
def to_weather_terrain(chunk_set: ChunkSet, 
                       coarse: Terrain,
                       scale: float,
                       fill_gaps: bool = False) -> tuple[Terrain, set]:
    """Convert ChunkSet to Terrain with weather fields.
    
    Args:
        fill_gaps: If False, mark non-chunk hexes as invalid (hide them)
    """
    
    # Build terrain from chunks
    terrain = chunk_set.to_terrain(fill_gaps=fill_gaps)
    
    # Copy coarse styling
    terrain.colorLevels = coarse.colorLevels
    terrain.seaLevel = coarse.seaLevel
    terrain.elevationDelta = coarse.elevationDelta
    terrain.climate = coarse.climate
    terrain.geo = coarse.geo
    
    # Add styles
    for style in terrain.colorLevels:
        terrain.hexGrid.builder.add_style(style)
    terrain.hexGrid.builder.add_style(terrain.seaLevel)
    
    # Compute origin
    positions = list(chunk_set.pos_to_chunk.keys())
    min_q = min(p.q for p in positions)
    min_r = min(p.r for p in positions)
    origin_world = HexPosition(min_q, min_r, -min_q - min_r)
    
    # Initialize weather fields
    n_hexes = len(terrain.elevations)
    terrain.fields['temperature'] = np.zeros(n_hexes)
    terrain.fields['precipitation'] = np.zeros(n_hexes)
    
    # Track which hexes are actually covered by chunks
    chunk_covered = set()
    
    # === PASS 1: Copy from chunk positions (core AND halo) ===
    for world_pos, (chunk, idx, is_core) in chunk_set.pos_to_chunk.items():
        local_pos = world_pos - origin_world
        grid_idx = terrain.hexGrid.hexposition_to_index(local_pos, origin_index=0)
        
        if grid_idx < 0 or grid_idx >= n_hexes:
            continue
        
        chunk_covered.add(grid_idx)
        
        if 'temperature' in chunk.fields:
            if is_core or terrain.fields['temperature'][grid_idx] == 0:
                terrain.fields['temperature'][grid_idx] = chunk.fields['temperature'][idx]
        if 'precipitation' in chunk.fields:
            if is_core or terrain.fields['precipitation'][grid_idx] == 0:
                terrain.fields['precipitation'][grid_idx] = chunk.fields['precipitation'][idx]
    
    # === Mark non-chunk hexes as invalid ===
    terrain.hexGrid.invalidRegion = set(range(n_hexes)) - chunk_covered
    
    return terrain, chunk_covered


In [ ]:
#| export
# === WEATHER FOR CHUNKS ===

@patch
def compute_coarse_weather(self: ChunkedTerrainGenerator, 
                           climate: ClimatePreset = None,geo: GeoBounds = None) -> Terrain:
    """Compute weather on coarse terrain."""
    coarse = self._coarse_terrain

    # Set geographic bounds
    if geo:
        coarse.geo = geo
    elif coarse.geo is None:
        # Default bounds (California-ish)
        coarse.geo = GeoBounds(
            lat_min=32.5, lat_max=42.0,
            lon_min=-124.5, lon_max=-114.5
        )
    
    if climate:
        coarse.climate = climate
    elif coarse.climate is None:
        # Default to mediterranean
        coarse.climate = TerrainPatterns(coarse).weatherPatterns()['mediterranean']
    
    coarse.compute_weather(force_recompute=True)
    return coarse


@patch
def _wind_direction_order(self: ChunkedTerrainGenerator, 
                          graph: ChunkGraph) -> list[int]:
    """Reorder chunks by wind direction (upwind first).
    
    Wind blows FROM wind_dir degrees, so chunks on the upwind side
    should be processed first.
    """
    if self._coarse_terrain.climate is None:
        return graph.process_order  # Fallback to drainage order
    
    wind_dir = self._coarse_terrain.climate.wind_dir
    # Convert to unit vector (wind blows FROM this direction)
    wind_rad = math.radians(wind_dir)
    wind_dx = -math.sin(wind_rad)  # Wind vector points downwind
    wind_dy = -math.cos(wind_rad)
    
    # Score chunks by how far upwind they are
    chunk_scores = []
    for idx, chunk_ref in enumerate(graph.chunk_refs):
        center = chunk_ref.center_hex
        # Project onto wind direction (higher = more upwind)
        score = center.q * wind_dx + center.r * wind_dy
        chunk_scores.append((idx, -score))  # Negative so upwind is first
    
    chunk_scores.sort(key=lambda x: x[1])
    return [idx for idx, _ in chunk_scores]

In [ ]:
#| export
@patch
def process_chunks_with_weather(self: ChunkedTerrainGenerator,
                                graph: ChunkGraph,
                                chunk_indices: list[int] = None,
                                climate: ClimatePreset = None,
                                geo: GeoBounds = None,
                                use_cache: bool = True) -> tuple[ChunkSet, dict]:
    """Generate chunks and compute weather, using cache when available."""
    
    if geo is None:
        geo = GeoBounds(lat_min=32.5, lat_max=42.0, lon_min=-124.5, lon_max=-114.5)
    
    coarse = self.compute_coarse_weather(climate, geo)
    climate_name = coarse.climate.name if coarse.climate else ""
    wind_dir = coarse.climate.wind_dir if coarse.climate else 270.0
    
    if chunk_indices is None:
        chunk_indices = graph.process_order
    
    wind_order = self._wind_direction_order(graph)
    ordered_indices = [i for i in wind_order if i in chunk_indices]
    
    generated = {}
    weather_stats = {}
    cache_hits = 0
    cache_misses = 0
    
    for chunk_idx in ordered_indices:
        chunk_ref = graph.chunk_refs[chunk_idx]
        chunk = self.generate_chunk(chunk_ref)
        
        # Copy halo from processed neighbors
        for prev_idx in graph.processed_before(chunk_idx):
            if prev_idx in generated:
                chunk.copy_from(generated[prev_idx])
        
        generated[chunk_idx] = chunk
        
        # === CHECK CACHE ===
        if use_cache and self.coarse_world_id:
            has_cached = self.storage.has_chunk_weather(
                self.coarse_world_id, chunk_ref,
                climate_name=climate_name, min_count=chunk.core_size // 2
            )
            
            if has_cached:
                # Load from cache
                load_result = self.storage.load_chunk_weather(
                    self.coarse_world_id, chunk, chunk_ref
                )
                if load_result.status == 'loaded':
                    cache_hits += 1
                    print(f"Chunk {chunk_idx}: loaded from cache")
                    
                    # Still collect stats
                    if 'temperature' in chunk.fields:
                        temps = chunk.fields['temperature'][:chunk.core_size]
                        precips = chunk.fields['precipitation'][:chunk.core_size]
                        weather_stats[chunk_idx] = {
                            'temp_range': (temps.min(), temps.max()),
                            'precip_range': (precips.min(), precips.max())
                        }
                    continue
        
        # === COMPUTE WEATHER ===
        cache_misses += 1
        
        neighbor_chunks = [generated[i] for i in ordered_indices 
                          if i in generated and i != chunk_idx]
        chunk_set = ChunkSet([chunk] + neighbor_chunks, coarse, self.scale)
        
        chunk_set.compute_weather(
            coarse, self.scale,
            lapse_rate=coarse.climate.lapse_rate if coarse.climate else 6.5,
            wind_dir=wind_dir
        )
        
        # === SAVE TO CACHE ===
        if use_cache and self.coarse_world_id:
            save_result = self.storage.save_chunk_weather(
                chunk, chunk_ref, self.coarse_world_id,
                climate_name=climate_name,
                wind_dir=wind_dir
            )
            if save_result.status == 'saved':
                print(f"Chunk {chunk_idx}: computed and cached ({save_result.context})")
            else:
                print(f"Chunk {chunk_idx}: computed (cache save failed: {save_result.context})")
        else:
            print(f"Chunk {chunk_idx}: computed (no cache)")
        
        # Collect stats
        if 'temperature' in chunk.fields:
            temps = chunk.fields['temperature'][:chunk.core_size]
            precips = chunk.fields['precipitation'][:chunk.core_size]
            weather_stats[chunk_idx] = {
                'temp_range': (temps.min(), temps.max()),
                'precip_range': (precips.min(), precips.max())
            }
    
    print(f"\nCache: {cache_hits} hits, {cache_misses} misses")
    
    final_chunks = [generated[i] for i in chunk_indices if i in generated]
    final_set = ChunkSet(final_chunks, coarse, self.scale)
    
    return final_set, weather_stats


In [ ]:
# === FULL WEATHER DEMO - FIXED ===
with GeoStorageDebugger(keep_on_error=False) as debugger:
    gen = ChunkedTerrainGenerator(
        storage=debugger.server,
        coarse_rings=15, coarse_radius=50, fine_radius=10,
        chunk_rings=8, halo_rings=3
    )
    
    coarse = gen.generate_coarse(num_plates=8, ocean_fraction=0.35, oceanic_sides=['W'], seed=42)
    coarse.carve_to_ocean(num_lakes=3, max_iters=5)
    
    graph = gen.build_chunk_graph(max_ring=2)
    climate = TerrainPatterns(coarse).weatherPatterns()['mediterranean']
    
    geo = GeoBounds(
        lat_min=32.5, lat_max=42.0,
        lon_min=-124.5, lon_max=-114.5
    )
    
    # Process more chunks
    test_indices = graph.process_order[:7]
    print(f"Processing chunks: {test_indices}")
    
    chunk_set, weather_stats = gen.process_chunks_with_weather(
        graph,
        chunk_indices=test_indices,
        climate=climate,
        geo=geo
    )
    
    # === USE FIXED TERRAIN BUILDER ===
    # Use fill_gaps=False to hide non-chunk areas
    terrain, covered_indices = chunk_set.to_weather_terrain(coarse, gen.scale, fill_gaps=False)

    
    # === RENDER ELEVATION MAP ===
    terrain.hexGrid.adjustRadius(10)
    terrain.colorMap()
    terrain.hexGrid.update()
    
    textStyle = StyleCSS("textStyle", fill="white", font_size="1.2em", 
                         filter="drop-shadow(2px 2px 1px rgba(0,0,0,.7))")
    terrain.hexGrid.builder.add_style(textStyle)
    
    print(f"\n=== ELEVATION MAP ===")
    print(f"Grid: {terrain.hexGrid.nRows}x{terrain.hexGrid.nCols} = {len(terrain.hexGrid.hexes)} total")
    print(f"Covered (with weather): {len(covered_indices)} hexes")
    print(f"Invalid: {len(terrain.hexGrid.invalidRegion)} hexes")
    terrain.hexGrid.builder.add_centered_text(
        f"Detailed Terrain - {len(covered_indices)} hexes", 
        y_offset=-terrain.hexGrid.builder.height/2 + 30, 
        class_name=textStyle.name
    )
    display(terrain.hexGrid.builder.show())
    
    # === RENDER TEMPERATURE MAP ===
    print(f"\n=== TEMPERATURE MAP ===")
    temp_overlay = terrain.render_icon_temperature()
    terrain.hexGrid.builder.adjust("hexes", temp_overlay)
    terrain.hexGrid.builder.add_centered_text(
        f"Temperature: {climate.name} climate", 
        y_offset=-terrain.hexGrid.builder.height/2 + 30, 
        class_name=textStyle.name
    )
    
    # Filter to covered land
    covered_list = list(covered_indices)
    land_mask = terrain.elevations[covered_list] > 0
    land_temps = terrain.fields['temperature'][covered_list][land_mask]
    print(f"Land temperatures: {land_temps.min():.1f}°C to {land_temps.max():.1f}°C")
    display(terrain.hexGrid.builder.show())
    
    # === RENDER PRECIPITATION MAP ===
    print(f"\n=== PRECIPITATION MAP ===")
    terrain.colorMap()
    terrain.hexGrid.update()
    terrain.add_rain_overlay(layer_name="precip_overlay")
    terrain.hexGrid.builder.add_centered_text(
        f"Precipitation: wind from {climate.wind_dir}°", 
        y_offset=-terrain.hexGrid.builder.height/2 + 30, 
        class_name=textStyle.name
    )
    
    land_precip = terrain.fields['precipitation'][covered_list][land_mask]
    print(f"Land precipitation: {land_precip.min():.0f}mm to {land_precip.max():.0f}mm/year")
    display(terrain.hexGrid.builder.show())
    
    detail_terrain = terrain
    print(f"\n✓ detail_terrain ready - no gaps!")


Thoughts? 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

This is looking solid! Key observations:

**What's working well:**
- **Temperature range** (12.8-16.1°C) is tight but realistic for a coastal mediterranean region at similar latitudes
- **Precipitation range** (397-2114mm) shows ~5x variation - that's the rain shadow and orographic effects kicking in
- **Cache infrastructure** ready - first run computes, subsequent runs will hit cache
- **Invalid region masking** hides gaps cleanly - no more "ocean triangles"

**The pattern is now established:**

```
┌─────────────────────────────────────────────────────────────┐
│  Coarse layer:  Compute once with full algorithm            │
│  Chunk layer:   Interpolate + local adjustments             │
│  Cache:         Save computed chunks for reuse              │
│  ChunkSet:      Handles boundaries transparently            │
└─────────────────────────────────────────────────────────────┘
```

**For watersheds** (your next item), same pattern applies:
1. Coarse watersheds computed on coarse terrain
2. Chunks inherit watershed IDs by interpolation
3. Local refinement at boundaries using halo data
4. ChunkSet handles the cross-boundary queries

**One refinement to consider:** The current temperature spread is small because all 7 chunks are at similar elevations. When you expand to more chunks (especially coastal vs. mountain), you'll see the lapse rate create more dramatic variation.

Want to tackle watersheds next, or persist the chunks/weather to storage first?

In [ ]:
#| export
# === TEST CACHING ===
with GeoStorageDebugger(keep_on_error=False) as debugger:
    gen = ChunkedTerrainGenerator(
        storage=debugger.server,
        coarse_rings=15, coarse_radius=50, fine_radius=10,
        chunk_rings=8, halo_rings=3
    )
    
    coarse = gen.generate_coarse(num_plates=8, ocean_fraction=0.35, oceanic_sides=['W'], seed=42)
    coarse.carve_to_ocean(num_lakes=3, max_iters=5)
    
    graph = gen.build_chunk_graph(max_ring=2)
    climate = TerrainPatterns(coarse).weatherPatterns()['mediterranean']
    
    test_indices = graph.process_order[:5]
    
    # First run - should compute and cache
    print("=== FIRST RUN (compute) ===")
    chunk_set1, stats1 = gen.process_chunks_with_weather(
        graph, chunk_indices=test_indices, climate=climate, use_cache=True
    )
    
    # Second run - should hit cache
    print("\n=== SECOND RUN (should use cache) ===")
    chunk_set2, stats2 = gen.process_chunks_with_weather(
        graph, chunk_indices=test_indices, climate=climate, use_cache=True
    )
    
    # Verify data matches
    for idx in test_indices:
        t1 = stats1[idx]['temp_range']
        t2 = stats2[idx]['temp_range']
        assert np.isclose(t1[0], t2[0], atol=0.1), f"Chunk {idx} temp mismatch"
    print("\n✓ Cached data matches computed data")


## Watershed by chunks

In [ ]:
SVGBuilder.BUILDERHIDE = False

#| export
@patch
def compute_coarse_watersheds(self: ChunkedTerrainGenerator, 
                               num_lakes: int = 3,
                               num_peaks: int = 50,
                               debug: bool = False) -> list[Watershed]:
    """Compute watersheds using existing infrastructure."""
    coarse = self._coarse_terrain
    
    # 1. Ensure weather computed (needed for find_river_sources)
    if 'precipitation' not in coarse.fields:
        coarse.compute_precipitation_sb()
    
    # 2. Carve drainage first - this ensures connectivity
    if not hasattr(self, '_carved_rivers') or not self._carved_rivers:
        self._carved_rivers = coarse.carve_to_ocean(num_lakes=num_lakes, max_iters=5)
        if debug:
            print(f"Carved {len(self._carved_rivers)} drainage paths")
    
    # 3. Use existing Watershed.compute_all() - it already does everything!
    watersheds = Watershed.compute_all(coarse, num_peaks=num_peaks, debug=debug)
    
    # 4. Store watershed IDs in coarse terrain
    coarse.fields['watershed_id'] = np.full(len(coarse.elevations), -1, dtype=np.int32)
    for ws_id, watershed in enumerate(watersheds):
        for hex_idx in watershed.region.hexes:
            coarse.fields['watershed_id'][hex_idx] = ws_id
    
    self._coarse_watersheds = watersheds
    return watersheds


## This is where we start to go wrong

#| export
@patch
def assign_watersheds(self: ChunkSet, 
                      coarse: Terrain,
                      scale: float,
                      watersheds: list[Watershed]) -> dict[HexPosition, int]:
    """Assign watershed IDs to chunk hexes, processing coast→inland."""
    
    # 1. Build coarse watershed lookup
    coarse_ws = coarse.fields.get('watershed_id', {})
    
    # 2. Get all core positions with interpolated distance
    positions_with_dist = []
    for pos in self.iter_positions():
        if self.elevation_at(pos) <= 0:
            continue  # Skip ocean
        
        # Interpolate distance_to_coast from coarse
        dist = _interpolate_field(pos, coarse, scale, 'distance_to_coast')
        positions_with_dist.append((pos, dist))
    
    # 3. Sort by distance (coast first)
    positions_with_dist.sort(key=lambda x: x[1])
    
    # 4. Assign in order
    assignments = {}
    for pos, dist in positions_with_dist:
        # Check downstream first
        lowest = self.lowest_neighbor(pos)
        if lowest in assignments:
            assignments[pos] = assignments[lowest]
        else:
            # Fallback to coarse interpolation
            assignments[pos] = _lookup_coarse_watershed(pos, coarse, scale)
    
    return assignments


#| export
@patch
def process_chunks_with_watersheds(self: ChunkedTerrainGenerator,
                                    graph: ChunkGraph,
                                    chunk_indices: list[int] = None,
                                    num_coarse_watersheds: int = 20,
                                    use_cache: bool = True,
                                    debug: bool = False) -> tuple[ChunkSet, dict]:
    """Generate chunks and assign watersheds with caching."""
    
    if not hasattr(self, '_coarse_watersheds') or self._coarse_watersheds is None:
        self.compute_coarse_watersheds(num_peaks=num_coarse_watersheds, debug=debug)
    
    coarse = self._coarse_terrain
    watersheds = self._coarse_watersheds
    
    if chunk_indices is None:
        chunk_indices = graph.process_order
    
    generated = {}
    watershed_stats = {}
    cache_hits = 0
    cache_misses = 0
    
    for chunk_idx in graph.process_order:
        if chunk_idx not in chunk_indices:
            continue
        
        chunk_ref = graph.chunk_refs[chunk_idx]
        chunk = self.generate_chunk(chunk_ref)
        
        # Copy halo from processed neighbors
        for prev_idx in graph.processed_before(chunk_idx):
            if prev_idx in generated:
                chunk.copy_from(generated[prev_idx])
        
        generated[chunk_idx] = chunk
        
        # === CHECK CACHE ===
        if use_cache and self.coarse_world_id:
            has_cached = self.storage.has_chunk_watersheds(
                self.coarse_world_id, chunk_ref, min_count=chunk.core_size // 4
            )
            
            if has_cached:
                load_result = self.storage.load_chunk_watersheds(
                    self.coarse_world_id, chunk, chunk_ref
                )
                if load_result.status == 'loaded':
                    cache_hits += 1
                    if debug:
                        print(f"Chunk {chunk_idx}: loaded from cache")
                    
                    # Collect stats
                    from collections import Counter
                    ws_ids = [int(chunk.fields['watershed_id'][i]) for i in chunk.iter_core()]
                    watershed_stats[chunk_idx] = dict(Counter(ws_ids))
                    continue
        
        # === COMPUTE WATERSHEDS ===
        cache_misses += 1
        
        neighbor_chunks = [generated[i] for i in graph.processed_before(chunk_idx) 
                          if i in generated]
        chunk_set = ChunkSet([chunk] + neighbor_chunks, coarse, self.scale)
        
        assignments = chunk_set.assign_watersheds(coarse, self.scale, watersheds)
        chunk_set.store_watersheds(assignments)
        
        # === SAVE TO CACHE ===
        if use_cache and self.coarse_world_id:
            save_result = self.storage.save_chunk_watersheds(
                chunk, chunk_ref, self.coarse_world_id
            )
            if debug:
                print(f"Chunk {chunk_idx}: computed and cached ({save_result.context})")
        elif debug:
            print(f"Chunk {chunk_idx}: computed (no cache)")
        
        # Stats
        from collections import Counter
        watershed_stats[chunk_idx] = dict(Counter(assignments.values()))
    
    if debug:
        print(f"\nCache: {cache_hits} hits, {cache_misses} misses")
    
    final_chunks = [generated[i] for i in chunk_indices if i in generated]
    final_set = ChunkSet(final_chunks, coarse, self.scale)
    
    return final_set, watershed_stats

# === TEST WATERSHEDS ===
with GeoStorageDebugger(keep_on_error=False) as debugger:
    gen = ChunkedTerrainGenerator(
        storage=debugger.server,
        coarse_rings=15, coarse_radius=50, fine_radius=10,
        chunk_rings=8, halo_rings=3
    )
    
    coarse = gen.generate_coarse(num_plates=8, ocean_fraction=0.35, oceanic_sides=['W'], seed=42)
    gen._carved_rivers = coarse.carve_to_ocean(num_lakes=3, max_iters=5)  # ← Store them!

    
    # Compute coarse weather (needed for watersheds)
    climate = TerrainPatterns(coarse).weatherPatterns()['mediterranean']
    gen.compute_coarse_weather(climate)
    
    # Compute coarse watersheds
    print("=== COARSE WATERSHEDS ===")
    watersheds = gen.compute_coarse_watersheds(num_lakes=3, debug=True)
    print(f"Created {len(watersheds)} coarse watersheds")
    
    # Process chunks with watersheds
    graph = gen.build_chunk_graph(max_ring=2)
    test_indices = graph.process_order[:5]
    
    print(f"\n=== CHUNK WATERSHEDS ===")
    chunk_set, ws_stats = gen.process_chunks_with_watersheds(
        graph, chunk_indices=test_indices, debug=True
    )
    
    # Visualize
    terrain, covered = chunk_set.to_weather_terrain(coarse, gen.scale, fill_gaps=False)
    terrain.hexGrid.adjustRadius(12)
    terrain.colorMap()
    terrain.hexGrid.update()
    
    # Color by watershed
    positions = list(chunk_set.pos_to_chunk.keys())
    min_q = min(p.q for p in positions)
    min_r = min(p.r for p in positions)
    origin_world = HexPosition(min_q, min_r, -min_q - min_r)
    
    # Get watershed colors from coarse
    ws_colors = [ws.style.properties.get('fill', '#cccccc') for ws in watersheds]
    
    ws_overlay = ""
    for world_pos, (chunk, idx, is_core) in chunk_set.pos_to_chunk.items():
        if not is_core:
            continue
        
        local_pos = world_pos - origin_world
        grid_idx = terrain.hexGrid.hexposition_to_index(local_pos, origin_index=0)
        
        if grid_idx < 0 or grid_idx >= len(terrain.hexGrid.hexes):
            continue
        
        ws_id = int(chunk.fields.get('watershed_id', np.array([-1]))[idx])
        if ws_id >= 0 and ws_id < len(ws_colors):
            hex_obj = terrain.hexGrid.hexes[grid_idx]
            color = ws_colors[ws_id]
            ws_overlay += f'<polygon points="{" ".join(f"{v.x},{v.y}" for v in hex_obj.v)}" fill="{color}" opacity="0.4"/>\n'
    
    #sanity check
    
    terrain.hexGrid.builder.adjust("watersheds", ws_overlay)
    
    textStyle = StyleCSS("textStyle", fill="white", font_size="1.2em", 
                         filter="drop-shadow(2px 2px 1px rgba(0,0,0,.7))")
    terrain.hexGrid.builder.add_style(textStyle)
    terrain.hexGrid.builder.add_centered_text(
        f"Chunk Watersheds: {len(watersheds)} basins", 
        y_offset=-terrain.hexGrid.builder.height/2 + 30, 
        class_name=textStyle.name
    )
    
    terrain.hexGrid.builder.show()


I think we should use some of our existing infrastructure already to compute watershed. but the carve to ocean should give us a sanity check to see if it done correctly. can you refactor compute_coarse_watersheds so it takes advantage of earlier algorithms but also builds large ones

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

You're right - let's use `Watershed.compute_all()` which already handles:
1. Peak finding via `find_river_sources()`
2. River creation + merging via `River.combine_rivers()`
3. Hex-to-river assignment by tracing downhill

The key is to **carve first**, then let `Watershed.compute_all()` work on the carved terrain:

```python
@patch
def compute_coarse_watersheds(self: ChunkedTerrainGenerator, 
                               num_lakes: int = 3,
                               num_peaks: int = 50,
                               debug: bool = False) -> list[Watershed]:
    """Compute watersheds using existing infrastructure."""
    coarse = self._coarse_terrain
    
    # 1. Ensure weather computed (needed for find_river_sources)
    if 'precipitation' not in coarse.fields:
        coarse.compute_precipitation_sb()
    
    # 2. Carve drainage first - this ensures connectivity
    if not hasattr(self, '_carved_rivers') or not self._carved_rivers:
        self._carved_rivers = coarse.carve_to_ocean(num_lakes=num_lakes, max_iters=5)
        if debug:
            print(f"Carved {len(self._carved_rivers)} drainage paths")
    
    # 3. Use existing Watershed.compute_all() - it already does everything!
    watersheds = Watershed.compute_all(coarse, num_peaks=num_peaks, debug=debug)
    
    # 4. Store watershed IDs in coarse terrain
    coarse.fields['watershed_id'] = np.full(len(coarse.elevations), -1, dtype=np.int32)
    for ws_id, watershed in enumerate(watersheds):
        for hex_idx in watershed.region.hexes:
            coarse.fields['watershed_id'][hex_idx] = ws_id
    
    self._coarse_watersheds = watersheds
    return watersheds
```

The difference from before:
- **Uses `Watershed.compute_all()`** instead of reimplementing the tracing logic
- **Carve first** so `compute_all()` sees the carved elevations
- **Simpler** - delegates to working code

Does this approach make sense? The `compute_all()` traces downhill from every land hex until it hits a river, which should work correctly on the carved terrain.

# === TEST WATERSHED CACHING ===
with GeoStorageDebugger(keep_on_error=False) as debugger:
    gen = ChunkedTerrainGenerator(
        storage=debugger.server,
        coarse_rings=15, coarse_radius=50, fine_radius=10,
        chunk_rings=8, halo_rings=3
    )
    
    coarse = gen.generate_coarse(num_plates=8, ocean_fraction=0.35, oceanic_sides=['W'], seed=42)
    coarse.carve_to_ocean(num_lakes=3, max_iters=5)
    
    climate = TerrainPatterns(coarse).weatherPatterns()['mediterranean']
    gen.compute_coarse_weather(climate)
    
    graph = gen.build_chunk_graph(max_ring=2)
    test_indices = graph.process_order[:5]
    
    # First run - computes and caches
    print("=== FIRST RUN (compute) ===")
    chunk_set1, stats1 = gen.process_chunks_with_watersheds(
        graph, chunk_indices=test_indices, use_cache=True, debug=True
    )
    
    # Second run - should hit cache
    print("\n=== SECOND RUN (should use cache) ===")
    chunk_set2, stats2 = gen.process_chunks_with_watersheds(
        graph, chunk_indices=test_indices, use_cache=True, debug=True
    )
    
    # Verify counts match
    for idx in test_indices:
        assert stats1[idx] == stats2[idx], f"Chunk {idx} stats mismatch"
    print("\n✓ Cached watershed data matches computed")


So the issue is that Watershed.compute_all works on terrains and not in our chunkgraph/chunkset world. is there a version that we could create that could be fed the coarse one and then would work with these data structures?



##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good observation! The pattern we've established is:

1. **Coarse**: Full algorithm (works on complete Terrain)
2. **Chunks**: Interpolate from coarse + local refinement

For watersheds, the strategy would be:

```
┌─────────────────────────────────────────────────────────────┐
│  COARSE: Watershed.compute_all(coarse_terrain)              │
│  → Stores watershed_id in coarse.fields                     │
│  → This already works!                                      │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│  CHUNKS: Interpolate watershed_id from coarse               │
│  → Similar to interpolate_weather_from_coarse()             │
│  → Map fine hex → nearest coarse hex → copy watershed_id    │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│  LOCAL REFINEMENT (optional):                               │
│  → At chunk boundaries, use lowest_neighbor() to verify     │
│  → Hexes should flow to same watershed as their downstream  │
└─────────────────────────────────────────────────────────────┘
```

The key insight: **watershed assignment is mostly spatial** - if a coarse hex belongs to watershed 3, all the fine hexes within it likely do too.

Should I sketch out `ChunkSet.assign_watersheds()` that:
1. Interpolates from coarse `watershed_id` field
2. Optionally validates by checking if `lowest_neighbor()` flows to same watershed?

This make sense. so it does feel we need sort of a reverse dependecy graph since we want to calculate the known watersheds closest to the ocean
please sketch out ChunkSet.assign_watersheds()

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

You're right about the reverse order! Watersheds are defined by where water ends up (outlet), so we want to "lock in" outlet chunks first, then propagate upstream.

Here's the sketch:

```python
@patch
def assign_watersheds(self: ChunkSet, 
                      coarse: Terrain,
                      scale: float,
                      watersheds: list[Watershed]) -> dict[HexPosition, int]:
    """Assign watershed IDs to chunk hexes.
    
    Strategy:
    1. Interpolate watershed_id from coarse (spatial lookup)
    2. Validate: if my lowest_neighbor has a watershed, I should match
    3. Return {world_pos: watershed_id}
    """
    assignments = {}
    
    for pos in self.iter_positions():  # Core only
        elev = self.elevation_at(pos)
        if elev <= 0:
            continue  # Skip ocean
        
        # Step 1: Interpolate from coarse
        coarse_ws_id = _lookup_coarse_watershed(pos, coarse, scale)
        
        # Step 2: Validate via downstream flow
        lowest = self.lowest_neighbor(pos)
        if lowest in assignments:
            downstream_ws = assignments[lowest]
            if downstream_ws != coarse_ws_id:
                # Trust downstream (already validated)
                coarse_ws_id = downstream_ws
        
        assignments[pos] = coarse_ws_id
    
    return assignments
```

The key question: **what order do we iterate `iter_positions()`?** 

Currently it's arbitrary. We'd need to sort by elevation (low→high) so downstream hexes are assigned first. Does that make sense as the approach?

The other option besides elevation would be distance from the coast. We would want to build up big watersheds

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Distance from coast makes more sense conceptually! Water flows *toward* the coast, so:

```
Coast (dist=0) → process first → outlet watersheds established
       ↓
Inland (dist=1,2,3...) → each hex joins the watershed its water flows into
       ↓
Headwaters → final assignments, tributary branches resolved
```

This naturally builds big watersheds because:
- Coastal hexes claim their outlet identity early
- Inland hexes "flow into" already-assigned downstream hexes
- Small tributaries merge into main basins rather than fragmenting

**Two approaches for distance:**

1. Use existing `coarse.fields['distance_to_coast']` (if computed)
2. Compute within ChunkSet using `elevation_at()` to find ocean neighbors

Which would you prefer, or should we support both?

Lets use distance from the coast. this way we could avoid creating one hex beach watersheds as well

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good choice! Here's the ordering strategy:

```
Distance 0 (coast): Skip - these are ocean hexes
Distance 1 (beach): Assign from coarse - these are watershed outlets
Distance 2+: Check downstream neighbor first, fallback to coarse
```

The key insight: **beach hexes (dist=1) ARE the outlet**, so they define the watershed. Everything upstream joins them.

For `ChunkSet.assign_watersheds()`, we'd need:

1. **Compute distance within ChunkSet** - BFS from ocean hexes (where `elevation_at() <= 0`)
2. **Sort positions by distance** - ascending order
3. **Assign in order** - downstream always assigned before upstream

Want me to sketch out the distance computation, or would you prefer to tackle that piece?

In [ ]:
!cat ../HexMagic/voronoi.py

#| export
so we should have a good distance to the coast already. we needed it for our weather as well

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

You're right! `distance_to_coast` should already be in `coarse.fields` from weather computation. Let me sketch the full `assign_watersheds()`:

```python
@patch
def assign_watersheds(self: ChunkSet, 
                      coarse: Terrain,
                      scale: float,
                      watersheds: list[Watershed]) -> dict[HexPosition, int]:
    """Assign watershed IDs to chunk hexes, processing coast→inland."""
    
    # 1. Build coarse watershed lookup
    coarse_ws = coarse.fields.get('watershed_id', {})
    
    # 2. Get all core positions with interpolated distance
    positions_with_dist = []
    for pos in self.iter_positions():
        if self.elevation_at(pos) <= 0:
            continue  # Skip ocean
        
        # Interpolate distance_to_coast from coarse
        dist = _interpolate_field(pos, coarse, scale, 'distance_to_coast')
        positions_with_dist.append((pos, dist))
    
    # 3. Sort by distance (coast first)
    positions_with_dist.sort(key=lambda x: x[1])
    
    # 4. Assign in order
    assignments = {}
    for pos, dist in positions_with_dist:
        # Check downstream first
        lowest = self.lowest_neighbor(pos)
        if lowest in assignments:
            assignments[pos] = assignments[lowest]
        else:
            # Fallback to coarse interpolation
            assignments[pos] = _lookup_coarse_watershed(pos, coarse, scale)
    
    return assignments
```

The key insight: by processing coast→inland, when we check `lowest_neighbor`, it's already assigned (because it's closer to coast).

Should I also add a helper to store the assignments back into the chunks' `watershed_id` field?

So how do we start making great water sheds for our detail views? we want to process through our chunks and have things drain into known chunks. one test is to make sure that carve_to_ocean and the chunk have the name number of watersheds that end in lakes/local miniuma. when we start stiching we can connect the local miniumn to our downstream set of chunks

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good thinking! The validation strategy:

```
┌─────────────────────────────────────────────────────────────┐
│  COARSE: carve_to_ocean(num_lakes=N)                        │
│  → Tells us: "this region should have N lakes"              │
│  → All other minima drain to ocean or neighbor chunks       │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│  CHUNK: carve locally, count minima                         │
│  → Core minima that drain to HALO = exits (good!)           │
│  → Core minima that stay in CORE = lakes (count these)      │
│  → Sanity check: lake count ≤ expected from coarse          │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│  STITCH: downstream chunk's halo = upstream chunk's outlet  │
│  → Local minima at boundary become entry points             │
│  → Rivers connect across chunk boundaries                   │
└─────────────────────────────────────────────────────────────┘
```

**First step:** Can we count how many coarse lakes fall within each chunk's footprint? This gives us the "expected lake count" per chunk.

Would you like me to sketch a `count_coarse_lakes_per_chunk()` function that queries which coarse local minima (after carving) are inside each chunk's region?

Isn't this just local minima + costal ocean tiles?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes, exactly! Much simpler:

- **Ocean outlets** = land hexes adjacent to ocean (dist_to_coast = 1)
- **Lakes** = remaining local minima after carving

So for a chunk:
1. Count local minima in core
2. Subtract those that touch halo/ocean (these drain out)
3. What's left = lakes

The coarse `carve_to_ocean(num_lakes=N)` guarantees exactly N true lakes globally. Each chunk should have ≤ the coarse lakes that fall within its footprint.

Want to test this by running `find_local_minima()` on a carved chunk and checking which ones touch the halo boundary?

# === HELPER FUNCTIONS ===

def _interpolate_field(world_pos: HexPosition, coarse: Terrain, scale: float, field_name: str) -> float:
    """Interpolate a coarse field value to a fine world position."""
    coarse_q = int(round(world_pos.q / scale))
    coarse_r = int(round(world_pos.r / scale))
    coarse_s = -coarse_q - coarse_r
    coarse_pos = HexPosition(coarse_q, coarse_r, coarse_s)
    
    coarse_idx = coarse.hexGrid.hexposition_to_index(coarse_pos, origin_index=coarse.hexGrid.midpoint)
    
    if 0 <= coarse_idx < len(coarse.elevations) and field_name in coarse.fields:
        return coarse.fields[field_name][coarse_idx]
    return 0.0


def _lookup_coarse_watershed(world_pos: HexPosition, coarse: Terrain, scale: float) -> int:
    """Get watershed_id from coarse terrain for a fine world position."""
    coarse_q = int(round(world_pos.q / scale))
    coarse_r = int(round(world_pos.r / scale))
    coarse_s = -coarse_q - coarse_r
    coarse_pos = HexPosition(coarse_q, coarse_r, coarse_s)
    
    coarse_idx = coarse.hexGrid.hexposition_to_index(coarse_pos, origin_index=coarse.hexGrid.midpoint)
    
    if 0 <= coarse_idx < len(coarse.elevations) and 'watershed_id' in coarse.fields:
        return int(coarse.fields['watershed_id'][coarse_idx])
    return -1


# === CHUNKSET WATERSHED METHODS ===

@patch
def assign_watersheds(self: ChunkSet, 
                      coarse: Terrain,
                      scale: float,
                      watersheds: list[Watershed]) -> dict[HexPosition, int]:
    """Assign watershed IDs to chunk hexes, processing coast→inland."""
    
    # 1. Get all core positions with interpolated distance
    positions_with_dist = []
    for pos in self.iter_positions():
        if self.elevation_at(pos) <= 0:
            continue  # Skip ocean
        
        # Interpolate distance_to_coast from coarse
        dist = _interpolate_field(pos, coarse, scale, 'distance_to_coast')
        positions_with_dist.append((pos, dist))
    
    # 2. Sort by distance (coast first)
    positions_with_dist.sort(key=lambda x: x[1])
    
    # 3. Assign in order
    assignments = {}
    for pos, dist in positions_with_dist:
        # Check downstream first - if neighbor already assigned, join that watershed
        lowest = self.lowest_neighbor(pos)
        if lowest in assignments:
            assignments[pos] = assignments[lowest]
        else:
            # Fallback to coarse interpolation
            assignments[pos] = _lookup_coarse_watershed(pos, coarse, scale)
    
    return assignments


@patch
def store_watersheds(self: ChunkSet, assignments: dict[HexPosition, int]):
    """Store watershed assignments back into chunk fields."""
    for world_pos, ws_id in assignments.items():
        if world_pos in self.pos_to_chunk:
            chunk, idx, is_core = self.pos_to_chunk[world_pos]
            if is_core:
                if 'watershed_id' not in chunk.fields:
                    chunk.add_field('watershed_id', default=-1)
                chunk.fields['watershed_id'][idx] = ws_id


@patch
def find_chunk_minima(self: ChunkSet) -> tuple[list[HexPosition], list[HexPosition]]:
    """Find local minima in chunk, categorized by drainage.
    
    Returns:
        (lakes, outlets) where:
        - lakes: minima that stay in core (true lakes)
        - outlets: minima that drain to halo/ocean (exits)
    """
    core_positions = set(self.iter_positions())
    
    lakes = []
    outlets = []
    
    for pos in core_positions:
        elev = self.elevation_at(pos)
        if elev <= 0:
            continue
        
        # Check if local minimum
        lowest = self.lowest_neighbor(pos)
        lowest_elev = self.elevation_at(lowest)
        
        if lowest_elev >= elev:
            # This is a local minimum - where does it go?
            if lowest in core_positions:
                # Stays in core = lake
                lakes.append(pos)
            else:
                # Drains to halo/ocean = outlet
                outlets.append(pos)
    
    return lakes, outlets


@patch  
def count_coarse_lakes_in_region(self: ChunkSet, coarse: Terrain, scale: float) -> int:
    """Count how many coarse local minima (lakes) fall within this ChunkSet's footprint."""
    
    # Get coarse minima (post-carving)
    coarse_directions = coarse.flow_directions()
    coarse_minima = [i for i in range(len(coarse.elevations)) 
                     if coarse_directions[i] == -1 and coarse.elevations[i] > 0]
    
    # Check which fall within our chunk region
    count = 0
    for coarse_idx in coarse_minima:
        coarse_pos = coarse.hexGrid.index_to_hexposition(coarse_idx, origin_index=coarse.hexGrid.midpoint)
        
        # Convert to fine world coordinates
        fine_q = int(coarse_pos.q * scale)
        fine_r = int(coarse_pos.r * scale)
        fine_s = -fine_q - fine_r
        fine_pos = HexPosition(fine_q, fine_r, fine_s)
        
        # Check if within our core
        if fine_pos in self.pos_to_chunk:
            _, _, is_core = self.pos_to_chunk[fine_pos]
            if is_core:
                count += 1
    
    return count


# === CHUNKED TERRAIN GENERATOR METHODS ===

@patch
def compute_coarse_watersheds(self: ChunkedTerrainGenerator, 
                               num_lakes: int = 3,
                               num_peaks: int = 50,
                               debug: bool = False) -> list[Watershed]:
    """Compute watersheds on coarse terrain using existing infrastructure."""
    coarse = self._coarse_terrain
    
    # 1. Ensure weather computed (needed for find_river_sources)
    if 'precipitation' not in coarse.fields:
        coarse.compute_precipitation_sb()
    
    # 2. Ensure distance_to_coast computed
    if 'distance_to_coast' not in coarse.fields:
        coarse.compute_distance_from_coast()
    
    # 3. Carve drainage first - this ensures connectivity
    if not hasattr(self, '_carved_rivers') or not self._carved_rivers:
        self._carved_rivers = coarse.carve_to_ocean(num_lakes=num_lakes, max_iters=5)
        if debug:
            print(f"Carved {len(self._carved_rivers)} drainage paths")
    
    # 4. Use existing Watershed.compute_all()
    watersheds = Watershed.compute_all(coarse, num_peaks=num_peaks, debug=debug)
    
    # 5. Store watershed IDs in coarse terrain
    coarse.fields['watershed_id'] = np.full(len(coarse.elevations), -1, dtype=np.int32)
    for ws_id, watershed in enumerate(watersheds):
        for hex_idx in watershed.region.hexes:
            coarse.fields['watershed_id'][hex_idx] = ws_id
    
    self._coarse_watersheds = watersheds
    return watersheds




@patch
def find_chunk_minima(self: ChunkSet) -> tuple[list[HexPosition], list[HexPosition]]:
    """Find local minima in chunk, categorized by drainage.
    
    Returns:
        (lakes, outlets) where:
        - lakes: minima that stay in core (true lakes)
        - outlets: minima that drain to halo/ocean (exits)
    """
    core_positions = set(self.iter_positions())
    
    lakes = []
    outlets = []
    
    for pos in core_positions:
        elev = self.elevation_at(pos)
        if elev <= 0:
            continue
        
        # Check if local minimum
        lowest = self.lowest_neighbor(pos)
        lowest_elev = self.elevation_at(lowest)
        
        if lowest_elev >= elev:
            # This is a local minimum - where does it go?
            if lowest in core_positions:
                # Stays in core = lake
                lakes.append(pos)
            else:
                # Drains to halo/ocean = outlet
                outlets.append(pos)
    
    return lakes, outlets


@patch  
def count_coarse_lakes_in_region(self: ChunkSet, coarse: Terrain, scale: float) -> int:
    """Count how many coarse local minima (lakes) fall within this ChunkSet's footprint."""
    
    # Get coarse minima (post-carving)
    coarse_directions = coarse.flow_directions()
    coarse_minima = [i for i in range(len(coarse.elevations)) 
                     if coarse_directions[i] == -1 and coarse.elevations[i] > 0]
    
    # Check which fall within our chunk region
    count = 0
    for coarse_idx in coarse_minima:
        coarse_pos = coarse.hexGrid.index_to_hexposition(coarse_idx, origin_index=coarse.hexGrid.midpoint)
        
        # Convert to fine world coordinates
        fine_q = int(coarse_pos.q * scale)
        fine_r = int(coarse_pos.r * scale)
        fine_s = -fine_q - fine_r
        fine_pos = HexPosition(fine_q, fine_r, fine_s)
        
        # Check if within our core
        if fine_pos in self.pos_to_chunk:
            _, _, is_core = self.pos_to_chunk[fine_pos]
            if is_core:
                count += 1
    
    return count


@patch
def store_watersheds(self: ChunkSet, assignments: dict[HexPosition, int]):
    """Store watershed assignments back into chunk fields."""
    for world_pos, ws_id in assignments.items():
        if world_pos in self.pos_to_chunk:
            chunk, idx, is_core = self.pos_to_chunk[world_pos]
            if is_core:
                if 'watershed_id' not in chunk.fields:
                    chunk.add_field('watershed_id', default=-1)
                chunk.fields['watershed_id'][idx] = ws_id


#| export
@patch
def process_chunks_with_watersheds(self: ChunkedTerrainGenerator,
                                    graph: ChunkGraph,
                                    chunk_indices: list[int] = None,
                                    num_coarse_lakes: int = 3,
                                    max_lakes_per_chunk: int = 1,
                                    debug: bool = False) -> tuple[ChunkSet, dict]:
    """Generate chunks, carve drainage, then assign watersheds."""
    
    # Ensure coarse watersheds computed
    if not hasattr(self, '_coarse_watersheds') or self._coarse_watersheds is None:
        self.compute_coarse_watersheds(num_lakes=num_coarse_lakes, debug=debug)
    
    coarse = self._coarse_terrain
    watersheds = self._coarse_watersheds
    
    if chunk_indices is None:
        chunk_indices = graph.process_order
    
    generated = {}
    watershed_stats = {}
    
    for chunk_idx in graph.process_order:
        if chunk_idx not in chunk_indices:
            continue
        
        chunk_ref = graph.chunk_refs[chunk_idx]
        chunk = self.generate_chunk(chunk_ref)
        
        # Copy halo from processed neighbors
        for prev_idx in graph.processed_before(chunk_idx):
            if prev_idx in generated:
                chunk.copy_from(generated[prev_idx])
        
        generated[chunk_idx] = chunk
        
        # Build ChunkSet for this chunk + neighbors
        neighbor_chunks = [generated[i] for i in graph.processed_before(chunk_idx) 
                          if i in generated]
        chunk_set = ChunkSet([chunk] + neighbor_chunks, coarse, self.scale)
        
        # === CARVE DRAINAGE FIRST ===
        expected_lakes = chunk_set.count_coarse_lakes_in_region(coarse, self.scale)
        max_lakes = max(max_lakes_per_chunk, expected_lakes)  # Keep at least expected
        carved_paths = chunk_set.carve_drainage(max_lakes=max_lakes)
        
        if debug:
            print(f"Chunk {chunk_idx}: carved {len(carved_paths)} paths")
        
        # Find remaining minima (should be much fewer now)
        lakes, outlets = chunk_set.find_chunk_minima()
        
        # Assign watersheds
        assignments = chunk_set.assign_watersheds(coarse, self.scale, watersheds)
        chunk_set.store_watersheds(assignments)
        
        # Collect stats
        from collections import Counter
        ws_counts = Counter(assignments.values())
        
        watershed_stats[chunk_idx] = {
            'lakes': len(lakes),
            'outlets': len(outlets),
            'expected_lakes': expected_lakes,
            'carved_paths': len(carved_paths),
            'watershed_counts': dict(ws_counts)
        }
        
        if debug:
            status = "✓" if len(lakes) <= max_lakes else "⚠"
            print(f"  → {len(lakes)} lakes (max {max_lakes}), "
                  f"{len(outlets)} outlets, {len(ws_counts)} watersheds {status}")
    
    # Build final ChunkSet
    final_chunks = [generated[i] for i in chunk_indices if i in generated]
    final_set = ChunkSet(final_chunks, coarse, self.scale)
    
    return final_set, watershed_stats


#| export
# === TEST WATERSHED ASSIGNMENT ===
with GeoStorageDebugger(keep_on_error=False) as debugger:
    gen = ChunkedTerrainGenerator(
        storage=debugger.server,
        coarse_rings=15, coarse_radius=50, fine_radius=10,
        chunk_rings=8, halo_rings=3
    )
    
    coarse = gen.generate_coarse(num_plates=8, ocean_fraction=0.35, oceanic_sides=['W'], seed=42)
    
    # Compute weather first (for distance_to_coast)
    climate = TerrainPatterns(coarse).weatherPatterns()['mediterranean']
    gen.compute_coarse_weather(climate)
    
    # Build graph
    graph = gen.build_chunk_graph(max_ring=2)
    
    # Process chunks with watersheds
    test_indices = graph.process_order[:5]
    print(f"Processing chunks: {test_indices}")
    print(f"Coarse lakes: {len([i for i in range(len(coarse.elevations)) if coarse.flow_directions()[i] == -1 and coarse.elevations[i] > 0])}")
    
    
    chunk_set, ws_stats = gen.process_chunks_with_watersheds(
    graph, 
    chunk_indices=test_indices,
    num_coarse_lakes=3,  # ← fix this
    debug=True
)



    
    print(f"\n=== SUMMARY ===")
    total_lakes = sum(s['lakes'] for s in ws_stats.values())
    total_outlets = sum(s['outlets'] for s in ws_stats.values())
    total_expected = sum(s['expected_lakes'] for s in ws_stats.values())
    
    print(f"Total lakes: {total_lakes} (expected ≤{total_expected})")
    print(f"Total outlets: {total_outlets}")
    
    # Visualize
    terrain, covered = chunk_set.to_weather_terrain(coarse, gen.scale, fill_gaps=False)
    terrain.hexGrid.adjustRadius(12)
    terrain.colorMap()
    terrain.hexGrid.update()
    
    # Overlay watershed colors
    positions = list(chunk_set.pos_to_chunk.keys())
    min_q = min(p.q for p in positions)
    min_r = min(p.r for p in positions)
    origin_world = HexPosition(min_q, min_r, -min_q - min_r)
    
    ws_colors = StyleCSS.seaborn("tab20", 20)
    ws_overlay = ""
    
    for world_pos, (chunk, idx, is_core) in chunk_set.pos_to_chunk.items():
        if not is_core:
            continue
        
        ws_id = int(chunk.fields.get('watershed_id', np.array([-1]))[idx])
        if ws_id < 0:
            continue
        
        local_pos = world_pos - origin_world
        grid_idx = terrain.hexGrid.hexposition_to_index(local_pos, origin_index=0)
        
        if 0 <= grid_idx < len(terrain.hexGrid.hexes):
            hex_obj = terrain.hexGrid.hexes[grid_idx]
            color = ws_colors[ws_id % len(ws_colors)].properties.get('fill', '#888')
            ws_overlay += f'<polygon points="{" ".join(f"{v.x},{v.y}" for v in hex_obj.v)}" fill="{color}" opacity="0.4"/>\n'
    
    terrain.hexGrid.builder.adjust("watersheds", ws_overlay)
    
    # Mark lakes and outlets
    lakes_all, outlets_all = chunk_set.find_chunk_minima()
    
    marker_overlay = ""
    for pos in lakes_all:
        local_pos = pos - origin_world
        grid_idx = terrain.hexGrid.hexposition_to_index(local_pos, origin_index=0)
        if 0 <= grid_idx < len(terrain.hexGrid.hexes):
            hex_obj = terrain.hexGrid.hexes[grid_idx]
            marker_overlay += f'<circle cx="{hex_obj.center.x}" cy="{hex_obj.center.y}" r="6" fill="blue" stroke="white" stroke-width="2"/>\n'
    
    for pos in outlets_all:
        local_pos = pos - origin_world
        grid_idx = terrain.hexGrid.hexposition_to_index(local_pos, origin_index=0)
        if 0 <= grid_idx < len(terrain.hexGrid.hexes):
            hex_obj = terrain.hexGrid.hexes[grid_idx]
            marker_overlay += f'<circle cx="{hex_obj.center.x}" cy="{hex_obj.center.y}" r="4" fill="green" stroke="white" stroke-width="1"/>\n'
    
    terrain.hexGrid.builder.adjust("markers", marker_overlay)
    
    textStyle = StyleCSS("textStyle", fill="white", font_size="1.2em", 
                         filter="drop-shadow(2px 2px 1px rgba(0,0,0,.7))")
    terrain.hexGrid.builder.add_style(textStyle)
    terrain.hexGrid.builder.add_centered_text(
        f"Watersheds: {total_lakes} lakes (blue), {total_outlets} outlets (green)", 
        y_offset=-terrain.hexGrid.builder.height/2 + 30, 
        class_name=textStyle.name
    )
    
    terrain.hexGrid.builder.show()


#| export
This looks much better



##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great progress! The coarse watershed infrastructure is working well - 108 watersheds computed with proper drainage paths.

The test failed at the end because `count_coarse_lakes_in_region` wasn't patched before running. The cell defining that patch is above but wasn't executed before the test cell.

Quick fix - run the cell with `@patch def count_coarse_lakes_in_region(...)` first, then re-run the test.

Or shall we consolidate the watershed methods into a single cell to avoid ordering issues?

@patch
def store_watersheds(self: ChunkSet, assignments: dict[HexPosition, int]):
    """Store watershed assignments back into chunk fields."""
    for world_pos, ws_id in assignments.items():
        if world_pos in self.pos_to_chunk:
            chunk, idx, is_core = self.pos_to_chunk[world_pos]
            if is_core:
                if 'watershed_id' not in chunk.fields:
                    chunk.add_field('watershed_id', default=-1)
                chunk.fields['watershed_id'][idx] = ws_id


## Run it

In [ ]:
SVGBuilder.BUILDERHIDE = True

# === VISUALIZE CHUNK WATERSHEDS ===
terr = None
with GeoStorageDebugger(keep_on_error=False) as debugger:
    gen = ChunkedTerrainGenerator(
        storage=debugger.server,
        coarse_rings=15, coarse_radius=50, fine_radius=10,
        chunk_rings=8, halo_rings=3
    )
    
    coarse = gen.generate_coarse(num_plates=8, ocean_fraction=0.35, oceanic_sides=['W'], seed=42)
    
    # Compute weather (for distance_to_coast)
    climate = TerrainPatterns(coarse).weatherPatterns()['mediterranean']
    gen.compute_coarse_weather(climate)
    
    # Build graph and process chunks
    graph = gen.build_chunk_graph(max_ring=2)
    test_indices = graph.process_order #[:7]
    
    print(f"Processing chunks: {test_indices}")
    
    chunk_set, ws_stats = gen.process_chunks_with_watersheds(
        graph, 
        chunk_indices=test_indices,
        num_coarse_lakes=3,
        max_lakes_per_chunk=1,
        debug=True
    )
    
    # === BUILD TERRAIN ===
    terrain, covered = chunk_set.to_weather_terrain(coarse, gen.scale, fill_gaps=False)
    terrain.hexGrid.adjustRadius(10)
    terrain.colorMap()
    terrain.hexGrid.update()
    
    # Get origin for coordinate mapping
    positions = list(chunk_set.pos_to_chunk.keys())
    min_q = min(p.q for p in positions)
    min_r = min(p.r for p in positions)
    origin_world = HexPosition(min_q, min_r, -min_q - min_r)
    
    # === WATERSHED OVERLAY ===
    # Use distinct colors for watersheds
    num_watersheds = len(gen._coarse_watersheds) if gen._coarse_watersheds else 20
    ws_colors = StyleCSS.seaborn("tab20", min(20, num_watersheds))
    
    ws_overlay = ""
    ws_hex_counts = {}  # Track how many hexes per watershed
    
    for world_pos, (chunk, idx, is_core) in chunk_set.pos_to_chunk.items():
        if not is_core:
            continue
        
        if 'watershed_id' not in chunk.fields:
            continue
            
        ws_id = int(chunk.fields['watershed_id'][idx])
        if ws_id < 0:
            continue
        
        ws_hex_counts[ws_id] = ws_hex_counts.get(ws_id, 0) + 1
        
        local_pos = world_pos - origin_world
        grid_idx = terrain.hexGrid.hexposition_to_index(local_pos, origin_index=0)
        
        if 0 <= grid_idx < len(terrain.hexGrid.hexes):
            hex_obj = terrain.hexGrid.hexes[grid_idx]
            color = ws_colors[ws_id % len(ws_colors)].properties.get('fill', '#888')
            ws_overlay += f'<polygon points="{" ".join(f"{v.x},{v.y}" for v in hex_obj.v)}" fill="{color}" opacity="0.5"/>\n'
    
    terrain.hexGrid.builder.adjust("watersheds", ws_overlay)
    
    # === MARK LAKES AND OUTLETS ===
    lakes_all, outlets_all = chunk_set.find_chunk_minima()
    
    marker_overlay = ""
    
    # Lakes = blue circles
    for pos in lakes_all:
        local_pos = pos - origin_world
        grid_idx = terrain.hexGrid.hexposition_to_index(local_pos, origin_index=0)
        if 0 <= grid_idx < len(terrain.hexGrid.hexes):
            hex_obj = terrain.hexGrid.hexes[grid_idx]
            marker_overlay += f'<circle cx="{hex_obj.center.x}" cy="{hex_obj.center.y}" r="8" fill="#0066ff" stroke="white" stroke-width="2"/>\n'
    
    # Outlets = green diamonds
    for pos in outlets_all:
        local_pos = pos - origin_world
        grid_idx = terrain.hexGrid.hexposition_to_index(local_pos, origin_index=0)
        if 0 <= grid_idx < len(terrain.hexGrid.hexes):
            hex_obj = terrain.hexGrid.hexes[grid_idx]
            cx, cy = hex_obj.center.x, hex_obj.center.y
            marker_overlay += f'<polygon points="{cx},{cy-6} {cx+5},{cy} {cx},{cy+6} {cx-5},{cy}" fill="#00cc44" stroke="white" stroke-width="1"/>\n'
    
    terrain.hexGrid.builder.adjust("markers", marker_overlay)
    
    # === ADD FLOW ARROWS ===
    flow_overlay = terrain.flow_diagram_filtered(covered)
    terrain.hexGrid.builder.adjust("flow", flow_overlay)
    
    # === SUMMARY STATS ===
    total_lakes = sum(s['lakes'] for s in ws_stats.values())
    total_outlets = sum(s['outlets'] for s in ws_stats.values())
    total_carved = sum(s['carved_paths'] for s in ws_stats.values())
    unique_watersheds = len(ws_hex_counts)
    
    print(f"\n=== SUMMARY ===")
    print(f"Chunks processed: {len(test_indices)}")
    print(f"Lakes (blue): {total_lakes}")
    print(f"Outlets (green): {total_outlets}")
    print(f"Drainage paths carved: {total_carved}")
    print(f"Unique watersheds: {unique_watersheds}")
    print(f"Top 5 watersheds by size: {sorted(ws_hex_counts.items(), key=lambda x: -x[1])[:5]}")
    
    # === TITLE ===
    textStyle = StyleCSS("textStyle", fill="white", font_size="1.2em", 
                         filter="drop-shadow(2px 2px 1px rgba(0,0,0,.7))")
    terrain.hexGrid.builder.add_style(textStyle)
    terrain.hexGrid.builder.add_centered_text(
        f"Chunk Watersheds: {unique_watersheds} basins, {total_lakes} lakes, {total_outlets} outlets", 
        y_offset=-terrain.hexGrid.builder.height/2 + 30, 
        class_name=textStyle.name
    )
    
    # === LEGEND ===
    legend_styles = [
        StyleCSS("Lakes", fill="#0066ff", stroke="white", stroke_width=2),
        StyleCSS("Outlets", fill="#00cc44", stroke="white", stroke_width=1),
    ]
    for s in legend_styles:
        terrain.hexGrid.builder.add_style(s)
    
    legend_svg = terrain.hexGrid.builder.legendBelow(legend_styles, gap=20, bg_fill="rgba(255,255,255,0.9)")
    terrain.hexGrid.builder.adjust("legend", legend_svg)
    terr = terrain
    
terr.hexGrid.builder.show()


How big of a map is this


##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Based on the output:

- **19 chunks** processed (all chunks in the `max_ring=2` graph)
- **~4,100 core hexes** (19 chunks × 217 hexes/chunk)
- **42 unique watersheds** with watershed 0 covering 1,392 hexes (the dominant basin)

In terms of resolution:
- **Coarse map**: 15 rings, 50m radius hexes → ~900 hexes
- **Fine map**: 5× finer (10m radius) → each coarse hex becomes ~25 fine hexes
- **Scale factor**: 5:1 between coarse and fine

The SVG renders at ~1500×1350 pixels, which is a comfortable viewing size. This is well under the ~3,000 hex "smooth viewport" limit you mentioned earlier.

To cover a larger area, you could increase `max_ring` in the chunk graph (currently 2), or increase `coarse_rings` (currently 15).

can you test with a 30k size mapp. will that break watershed

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
# === TEST 30K HEX MAP ===
# To get ~30k hexes: need ~140 chunks (at 217 hexes/chunk)
# max_ring=6 gives ~127 chunks, max_ring=7 gives ~169 chunks

with GeoStorageDebugger(keep_on_error=False) as debugger:
    gen = ChunkedTerrainGenerator(
        storage=debugger.server,
        coarse_rings=25,      # Larger coarse map
        coarse_radius=50,
        fine_radius=10,
        chunk_rings=8,
        halo_rings=3
    )
    
    coarse = gen.generate_coarse(num_plates=12, ocean_fraction=0.35, oceanic_sides=['W'], seed=42)
    
    # Compute weather
    climate = TerrainPatterns(coarse).weatherPatterns()['mediterranean']
    gen.compute_coarse_weather(climate)
    
    # Build graph with more rings
    graph = gen.build_chunk_graph(max_ring=6)  # ~127 chunks
    
    print(f"Chunk graph: {len(graph.chunk_refs)} chunks")
    print(f"Expected hexes: ~{len(graph.chunk_refs) * 217:,}")
    
    import time
    start = time.time()
    
    chunk_set, ws_stats = gen.process_chunks_with_watersheds(
        graph, 
        chunk_indices=graph.process_order,
        num_coarse_lakes=5,
        max_lakes_per_chunk=1,
        debug=False  # Less verbose
    )
    
    elapsed = time.time() - start
    
    # Count actual hexes
    total_hexes = len(list(chunk_set.iter_positions()))
    lakes, outlets = chunk_set.find_chunk_minima()
    
    print(f"\n=== 30K MAP RESULTS ===")
    print(f"Total chunks: {len(graph.chunk_refs)}")
    print(f"Total core hexes: {total_hexes:,}")
    print(f"Lakes: {len(lakes)}")
    print(f"Outlets: {len(outlets)}")
    print(f"Unique watersheds: {len(set(ws_stats[i]['watershed_counts'].keys() for i in ws_stats))}")
    print(f"Time: {elapsed:.1f}s")
    print(f"Rate: {total_hexes/elapsed:.0f} hexes/sec")
    
    # Check if it broke
    if len(lakes) > 500:
        print(f"\n⚠ WARNING: {len(lakes)} lakes seems high - may need more carving iterations")
    else:
        print(f"\n✓ Watershed assignment completed successfully")
```

This should create approximately:
- 127 chunks × 217 hexes = **~27,500 hexes**
- Larger coarse map (25 rings vs 15)
- Same 5:1 scale factor

Run it and we'll see if the O(n²) watershed tracing becomes a bottleneck at this scale.

# === TEST 30K HEX MAP ===
# To get ~30k hexes: need ~140 chunks (at 217 hexes/chunk)
# max_ring=6 gives ~127 chunks, max_ring=7 gives ~169 chunks

with GeoStorageDebugger(keep_on_error=False) as debugger:
    gen = ChunkedTerrainGenerator(
        storage=debugger.server,
        coarse_rings=15,      # Larger coarse map
        coarse_radius=50,
        fine_radius=10,
        chunk_rings=8,
        halo_rings=3
    )
    
    coarse = gen.generate_coarse(num_plates=12, ocean_fraction=0.35, oceanic_sides=['W'], seed=42)
    
    # Compute weather
    climate = TerrainPatterns(coarse).weatherPatterns()['mediterranean']
    gen.compute_coarse_weather(climate)
    
    # Build graph with more rings
    graph = gen.build_chunk_graph(max_ring=6)  # ~127 chunks
    
    print(f"Chunk graph: {len(graph.chunk_refs)} chunks")
    print(f"Expected hexes: ~{len(graph.chunk_refs) * 217:,}")
    
    import time
    start = time.time()
    
    chunk_set, ws_stats = gen.process_chunks_with_watersheds(
        graph, 
        chunk_indices=graph.process_order,
        num_coarse_lakes=5,
        max_lakes_per_chunk=1,
        debug=False  # Less verbose
    )
    
    elapsed = time.time() - start
    
    # Count actual hexes
    total_hexes = len(list(chunk_set.iter_positions()))
    lakes, outlets = chunk_set.find_chunk_minima()
    
    print(f"\n=== 30K MAP RESULTS ===")
    print(f"Total chunks: {len(graph.chunk_refs)}")
    print(f"Total core hexes: {total_hexes:,}")
    print(f"Lakes: {len(lakes)}")
    print(f"Outlets: {len(outlets)}")
    # Fix the unique watersheds count
    all_ws_ids = set()
    for i in ws_stats:
        all_ws_ids.update(ws_stats[i]['watershed_counts'].keys())

    print(f"Unique watersheds: {len(all_ws_ids)}")
    print(f"Time: {elapsed:.1f}s")
    print(f"Rate: {total_hexes/elapsed:.0f} hexes/sec")

    print(f"Time: {elapsed:.1f}s")
    print(f"Rate: {total_hexes/elapsed:.0f} hexes/sec")
    
    # Check if it broke
    if len(lakes) > 500:
        print(f"\n⚠ WARNING: {len(lakes)} lakes seems high - may need more carving iterations")
    else:
        print(f"\n✓ Watershed assignment completed successfully")


It might be too slow maybe we should cache or compute itteratively.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

You're right - 158 hexes/sec is too slow. At 100K hexes that's 10+ minutes.

**The bottleneck**: For each chunk we rebuild a ChunkSet with ALL previously processed chunks. That's O(n²) behavior.

**Quick wins:**

1. **Only include neighbor chunks** - not ALL previous chunks:
```python
# Instead of:
neighbor_chunks = [generated[i] for i in graph.processed_before(chunk_idx) if i in generated]

# Use:
adjacent_indices = get_adjacent_chunk_indices(chunk_idx, graph)  # Just 6 neighbors
neighbor_chunks = [generated[i] for i in adjacent_indices if i in generated]
```

2. **Cache chunk results** - don't re-process:
```python
# In process_chunks_with_watersheds, check cache first
if self.storage.has_chunk_watersheds(world_id, chunk_ref):
    chunk = self.storage.load_chunk_with_fields(world_id, chunk_ref)
    continue  # Skip carving/assignment
```

3. **Batch the carving** - carve multiple chunks at once if they don't share boundaries

**Bigger refactor** (if needed):
- Pre-compute all chunks first (parallelizable)
- Run watershed assignment as a second pass using numpy arrays instead of dict lookups

Which approach would you like to try first? The "neighbor-only ChunkSet" is probably the biggest win for minimal code change.

### The pivot

In [ ]:
!cat ../HexMagic/water/*.py

In [ ]:
!cat ../HexMagic/database.py

So the issue is that n^2 algorithims aren't going to work on the full map. I think everything works through getting a coarse map. what we need is a way of projecting the rivers and watersheds at the coarse level down (and not worry so much if they violated local changes). 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

You're absolutely right. The key insight: **coarse watersheds are authoritative** - fine hexes just inherit them.

Instead of computing watersheds bottom-up (trace every fine hex → find its outlet), we should project top-down:

```
COARSE (fast):
  - Compute watersheds once (~700 hexes)
  - Each coarse hex has watershed_id

FINE (projection):
  - Fine hex → lookup nearest coarse hex → copy watershed_id
  - No tracing, no carving needed for watersheds
  - Just spatial interpolation
```

**What we lose**: Perfect drainage at fine scale (a fine hex might not flow to exactly the right watershed)

**What we gain**: O(n) instead of O(n²), works at any scale

The rivers/carving can still be local (improves terrain realism), but watershed assignment becomes a pure lookup.

Want to refactor `assign_watersheds()` to be purely interpolation-based, skipping the `lowest_neighbor()` validation?

yes lets start there. the goal of all of this is when we create a detailed terrain (or even with coatrse) we can create a drainagebasin by database look up. we only do the coarse algorithm once.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Perfect! So the new flow is:

```
1. Coarse (once):
   - Compute watersheds using full algorithm
   - Save to database with watershed_id

2. Fine/Chunks (many times):
   - Load coarse watershed_id field
   - Interpolate to fine hexes (nearest coarse hex)
   - No tracing, just spatial lookup
```

**First step**: Let's add a `load_coarse_watersheds()` helper that returns a simple lookup:

```python
@patch
def load_coarse_watershed_lookup(self: GeoStorage, world_id: int) -> dict:
    """Load coarse watershed assignments as {(q,r,s): watershed_id}."""
    hex_rows = self.query_hexes_latest(world_id)
    return {
        (row['q'], row['r'], row['s']): row.get('watershed_id', -1)
        for row in hex_rows
        if row.get('watershed_id') is not None
    }
```

Then we can use this in `ChunkSet.assign_watersheds()` - just map each fine hex to nearest coarse (q,r,s) and copy the ID.

Should I sketch out the refactored `assign_watersheds()` that uses this lookup instead of tracing?

@patch
def load_coarse_watershed_lookup(self: GeoStorage, world_id: int) -> dict:
    """Load coarse watershed assignments as {(q,r,s): watershed_id}."""
    hex_rows = self.query_hexes_latest(world_id)
    return {
        (row['q'], row['r'], row['s']): row.get('watershed_id', -1)
        for row in hex_rows
        if row.get('watershed_id') is not None
    }


In [ ]:
??Geology

I think we need to go back to the begining just after we created our weather. I don't know how much I should erase of the above cells, since I think the history is ok.
 we need now not just a coarse map, but a coarse geolgy. We do want that persitable in our database. lets call this masterGelogy.
from the masterGeologu we project down its watershed down so we start to create detail geology. (soil systems are easily computed). we don't want detailGeology computing watershed.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good call. Let's establish the clean architecture:

```
┌─────────────────────────────────────────────────────────────┐
│  MasterGeology (coarse, computed once, persisted)           │
│  ─────────────────────────────────────────────────────────  │
│  • Terrain (with carved drainage)                           │
│  • Plates                                                   │
│  • Weather (temperature, precipitation)                     │
│  • DrainageBasins / Watersheds                              │
│  • Saved to DB with world_id                                │
└─────────────────────────────────────────────────────────────┘
                            ↓ project
┌─────────────────────────────────────────────────────────────┐
│  DetailGeology (fine, computed on demand)                   │
│  ─────────────────────────────────────────────────────────  │
│  • Terrain (interpolated + local noise)                     │
│  • watershed_id (lookup from master, NOT computed)          │
│  • Soils (computed locally - cheap)                         │
│  • Weather (interpolated from master)                       │
└─────────────────────────────────────────────────────────────┘
```

```python
@dataclass
class MasterGeology:
    """Coarse-level geology with precomputed watersheds."""
    terrain: Terrain
    plates: list
    basins: DrainageBasins
    world_id: int = None  # DB reference
    
    @classmethod
    def create(cls, gen: ChunkedTerrainGenerator, 
               num_lakes: int = 3, 
               climate: ClimatePreset = None) -> 'MasterGeology':
        """Generate and persist master geology."""
        coarse = gen._coarse_terrain
        plates = gen._plates
        
        # Compute weather
        if climate:
            coarse.climate = climate
        coarse.compute_weather()
        
        # Carve drainage
        coarse.carve_to_ocean(num_lakes=num_lakes)
        
        # Compute watersheds/basins
        basins = DrainageBasins(coarse)
        
        return cls(terrain=coarse, plates=plates, basins=basins, 
                   world_id=gen.coarse_world_id)
    
    def watershed_lookup(self) -> dict:
        """Return {(q,r,s): watershed_id} for projection."""
        # Build from basins
        lookup = {}
        for ws_id, ws in enumerate(self.basins.watersheds):
            for hex_idx in ws.region.hexes:
                pos = self.terrain.hexGrid.index_to_hexposition(
                    hex_idx, origin_index=self.terrain.hexGrid.midpoint)
                lookup[(pos.q, pos.r, pos.s)] = ws_id
        return lookup


class DetailGeology:
    """Fine-level geology projected from master."""
    
    def __init__(self, terrain: Terrain, master: MasterGeology, scale: float):
        self.terrain = terrain
        self.master = master
        self.scale = scale
        
        # Project watersheds (lookup only, no computation)
        self._project_watersheds()
        
        # Compute soils locally (cheap)
        self.soils = SoilSystem.from_terrain(terrain)
    
    def _project_watersheds(self):
        """Assign watershed_id by interpolating from master."""
        lookup = self.master.watershed_lookup()
        
        n = len(self.terrain.elevations)
        self.terrain.fields['watershed_id'] = np.full(n, -1, dtype=np.int32)
        
        for idx in range(n):
            if self.terrain.elevations[idx] <= 0:
                continue
            
            pos = self.terrain.hexGrid.index_to_hexposition(
                idx, origin_index=self.terrain.hexGrid.midpoint)
            
            # Map to coarse coordinates
            coarse_q = int(round(pos.q / self.scale))
            coarse_r = int(round(pos.r / self.scale))
            coarse_s = -coarse_q - coarse_r
            
            ws_id = lookup.get((coarse_q, coarse_r, coarse_s), -1)
            self.terrain.fields['watershed_id'][idx] = ws_id
```

Should I write this as a full cell you can place after the weather tests?

I like this and don't want to completely abandon the chunk architecture. so perhaps a detail geolgy could store a list of chunkID of the hex chunks that cover it. 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good idea! The chunks become the building blocks for DetailGeology:

```python
@dataclass
class MasterGeology:
    """Coarse-level geology with precomputed watersheds."""
    terrain: Terrain
    plates: list
    basins: DrainageBasins
    world_id: int = None
    
    @classmethod
    def create(cls, gen: ChunkedTerrainGenerator, 
               num_lakes: int = 3,
               climate: ClimatePreset = None) -> 'MasterGeology':
        coarse = gen._coarse_terrain
        plates = gen._plates
        
        if climate:
            coarse.climate = climate
        coarse.compute_weather()
        coarse.carve_to_ocean(num_lakes=num_lakes)
        
        basins = DrainageBasins(coarse)
        
        return cls(terrain=coarse, plates=plates, basins=basins,
                   world_id=gen.coarse_world_id)
    
    def watershed_lookup(self) -> dict:
        """Return {(q,r,s): watershed_id} for projection."""
        lookup = {}
        for ws_id, ws in enumerate(self.basins.watersheds):
            for hex_idx in ws.region.hexes:
                pos = self.terrain.hexGrid.index_to_hexposition(
                    hex_idx, origin_index=self.terrain.hexGrid.midpoint)
                lookup[(pos.q, pos.r, pos.s)] = ws_id
        return lookup


@dataclass
class DetailGeology:
    """Fine-level geology projected from master, built from chunks."""
    terrain: Terrain
    master: MasterGeology
    scale: float
    chunk_refs: list[ChunkRef]  # Which chunks cover this detail
    
    @classmethod
    def from_chunks(cls, chunk_set: ChunkSet, 
                    master: MasterGeology,
                    scale: float) -> 'DetailGeology':
        """Build DetailGeology from a ChunkSet."""
        
        # Convert ChunkSet to Terrain
        terrain = chunk_set.to_terrain(fill_gaps=False)
        terrain.colorLevels = master.terrain.colorLevels
        terrain.seaLevel = master.terrain.seaLevel
        terrain.elevationDelta = master.terrain.elevationDelta
        terrain.climate = master.terrain.climate
        
        # Extract chunk refs from the chunks
        chunk_refs = [ChunkRef(c.center_world, c.core_rings) 
                      for c in chunk_set.chunks]
        
        detail = cls(terrain=terrain, master=master, 
                     scale=scale, chunk_refs=chunk_refs)
        
        # Project watersheds (O(n) lookup)
        detail._project_watersheds(chunk_set)
        
        # Project weather
        detail._project_weather(chunk_set)
        
        return detail
    
    def _project_watersheds(self, chunk_set: ChunkSet):
        """Assign watershed_id by lookup from master."""
        lookup = self.master.watershed_lookup()
        
        positions = list(chunk_set.pos_to_chunk.keys())
        min_q = min(p.q for p in positions)
        min_r = min(p.r for p in positions)
        origin_world = HexPosition(min_q, min_r, -min_q - min_r)
        
        n = len(self.terrain.elevations)
        self.terrain.fields['watershed_id'] = np.full(n, -1, dtype=np.int32)
        
        for world_pos, (chunk, idx, is_core) in chunk_set.pos_to_chunk.items():
            if not is_core or chunk.elevations[idx] <= 0:
                continue
            
            # Map to coarse
            coarse_q = int(round(world_pos.q / self.scale))
            coarse_r = int(round(world_pos.r / self.scale))
            coarse_s = -coarse_q - coarse_r
            
            ws_id = lookup.get((coarse_q, coarse_r, coarse_s), -1)
            
            # Store in terrain
            local_pos = world_pos - origin_world
            grid_idx = self.terrain.hexGrid.hexposition_to_index(local_pos, origin_index=0)
            if 0 <= grid_idx < n:
                self.terrain.fields['watershed_id'][grid_idx] = ws_id
    
    def _project_weather(self, chunk_set: ChunkSet):
        """Copy weather from chunks to terrain."""
        positions = list(chunk_set.pos_to_chunk.keys())
        min_q = min(p.q for p in positions)
        min_r = min(p.r for p in positions)
        origin_world = HexPosition(min_q, min_r, -min_q - min_r)
        
        n = len(self.terrain.elevations)
        self.terrain.fields['temperature'] = np.zeros(n)
        self.terrain.fields['precipitation'] = np.zeros(n)
        
        for world_pos, (chunk, idx, is_core) in chunk_set.pos_to_chunk.items():
            if not is_core:
                continue
            
            local_pos = world_pos - origin_world
            grid_idx = self.terrain.hexGrid.hexposition_to_index(local_pos, origin_index=0)
            
            if 0 <= grid_idx < n:
                if 'temperature' in chunk.fields:
                    self.terrain.fields['temperature'][grid_idx] = chunk.fields['temperature'][idx]
                if 'precipitation' in chunk.fields:
                    self.terrain.fields['precipitation'][grid_idx] = chunk.fields['precipitation'][idx]
    
    @property
    def soils(self):
        """Compute soils lazily (cheap operation)."""
        if not hasattr(self, '_soils'):
            self._soils = SoilSystem.from_terrain(self.terrain)
        return self._soils
```

Usage would be:

```python
# Once: create and persist master
master = MasterGeology.create(gen, num_lakes=3, climate=climate)

# Many times: create detail from chunks
chunk_set, _ = gen.process_chunks_with_weather(graph, chunk_indices=some_indices)
detail = DetailGeology.from_chunks(chunk_set, master, gen.scale)

# Detail knows its chunks
print(f"Detail covers {len(detail.chunk_refs)} chunks")
```

Should I write this as a runnable cell?

The ulitmate goal is that things outside of the module don't know about chunkset sets. so we probably want a function that would take a bounds on the MasterGeology and generate a detail one.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Exactly. Clean API:

```python
@dataclass
class MasterGeology:
    """Coarse-level geology with precomputed watersheds."""
    terrain: Terrain
    plates: list
    basins: DrainageBasins
    world_id: int = None
    _gen: 'ChunkedTerrainGenerator' = None  # Internal, not exposed
    
    @classmethod
    def create(cls, gen: ChunkedTerrainGenerator, 
               num_lakes: int = 3,
               climate: ClimatePreset = None) -> 'MasterGeology':
        coarse = gen._coarse_terrain
        plates = gen._plates
        
        if climate:
            coarse.climate = climate
        coarse.compute_weather()
        coarse.carve_to_ocean(num_lakes=num_lakes)
        
        basins = DrainageBasins(coarse)
        
        master = cls(terrain=coarse, plates=plates, basins=basins,
                     world_id=gen.coarse_world_id, _gen=gen)
        return master
    
    def watershed_lookup(self) -> dict:
        """Return {(q,r,s): watershed_id} for projection."""
        lookup = {}
        for ws_id, ws in enumerate(self.basins.watersheds):
            for hex_idx in ws.region.hexes:
                pos = self.terrain.hexGrid.index_to_hexposition(
                    hex_idx, origin_index=self.terrain.hexGrid.midpoint)
                lookup[(pos.q, pos.r, pos.s)] = ws_id
        return lookup
    
    def detail_for_rect(self, min_q: int, max_q: int, 
                        min_r: int, max_r: int) -> 'DetailGeology':
        """Generate detail geology for a rectangular region."""
        # Internally uses chunks - caller doesn't need to know
        chunk_refs = self._gen._chunks_intersecting_rect(min_q, max_q, min_r, max_r)
        chunks = [self._gen.generate_chunk(ref) for ref in chunk_refs]
        
        # Copy halos between adjacent chunks
        for i, chunk in enumerate(chunks):
            for other in chunks[:i]:
                chunk.copy_from(other)
        
        chunk_set = ChunkSet(chunks, self.terrain, self._gen.scale)
        
        # Compute weather on chunks
        chunk_set.compute_weather(self.terrain, self._gen.scale)
        
        return DetailGeology._from_chunk_set(chunk_set, self, self._gen.scale, chunk_refs)
    
    def detail_for_center(self, center: HexPosition, radius: int) -> 'DetailGeology':
        """Generate detail geology around a center point."""
        min_q = center.q - radius
        max_q = center.q + radius
        min_r = center.r - radius
        max_r = center.r + radius
        return self.detail_for_rect(min_q, max_q, min_r, max_r)
    
    def detail_for_watershed(self, watershed_id: int, padding: int = 5) -> 'DetailGeology':
        """Generate detail geology for a specific watershed."""
        ws = self.basins.watersheds[watershed_id]
        
        # Get bounds from watershed hexes
        positions = [self.terrain.hexGrid.index_to_hexposition(
            idx, origin_index=self.terrain.hexGrid.midpoint) 
            for idx in ws.region.hexes]
        
        # Scale to fine coordinates
        scale = self._gen.scale
        qs = [int(p.q * scale) for p in positions]
        rs = [int(p.r * scale) for p in positions]
        
        return self.detail_for_rect(
            min(qs) - padding, max(qs) + padding,
            min(rs) - padding, max(rs) + padding
        )


@dataclass  
class DetailGeology:
    """Fine-level geology projected from master."""
    terrain: Terrain
    master: MasterGeology
    scale: float
    chunk_refs: list[ChunkRef]  # Internal tracking
    
    @classmethod
    def _from_chunk_set(cls, chunk_set: ChunkSet, master: MasterGeology,
                        scale: float, chunk_refs: list) -> 'DetailGeology':
        """Internal: build from ChunkSet."""
        terrain, _ = chunk_set.to_weather_terrain(master.terrain, scale, fill_gaps=False)
        
        detail = cls(terrain=terrain, master=master, 
                     scale=scale, chunk_refs=chunk_refs)
        detail._project_watersheds(chunk_set)
        return detail
    
    def _project_watersheds(self, chunk_set: ChunkSet):
        """Assign watershed_id by lookup from master."""
        lookup = self.master.watershed_lookup()
        
        positions = list(chunk_set.pos_to_chunk.keys())
        min_q = min(p.q for p in positions)
        min_r = min(p.r for p in positions)
        origin_world = HexPosition(min_q, min_r, -min_q - min_r)
        
        n = len(self.terrain.elevations)
        self.terrain.fields['watershed_id'] = np.full(n, -1, dtype=np.int32)
        
        for world_pos, (chunk, idx, is_core) in chunk_set.pos_to_chunk.items():
            if not is_core or chunk.elevations[idx] <= 0:
                continue
            
            coarse_q = int(round(world_pos.q / self.scale))
            coarse_r = int(round(world_pos.r / self.scale))
            coarse_s = -coarse_q - coarse_r
            
            ws_id = lookup.get((coarse_q, coarse_r, coarse_s), -1)
            
            local_pos = world_pos - origin_world
            grid_idx = self.terrain.hexGrid.hexposition_to_index(local_pos, origin_index=0)
            if 0 <= grid_idx < n:
                self.terrain.fields['watershed_id'][grid_idx] = ws_id
    
    @property
    def soils(self):
        if not hasattr(self, '_soils'):
            self._soils = SoilSystem.from_terrain(self.terrain)
        return self._soils
```

Usage becomes simple:

```python
# Create master once
master = MasterGeology.create(gen, num_lakes=3, climate=climate)

# Get detail for any region - chunks are hidden
detail = master.detail_for_center(HexPosition(0, 0, 0), radius=50)
detail = master.detail_for_watershed(3)  # Watershed #3
detail = master.detail_for_rect(-100, 100, -80, 80)

# Use detail
detail.terrain.colorMap()
detail.terrain.hexGrid.builder.show()
```


@dataclass
class MasterGeology:
    """Coarse-level geology with precomputed watersheds."""
    terrain: Terrain
    plates: list
    basins: DrainageBasins
    world_id: int = None
    _gen: 'ChunkedTerrainGenerator' = None  # Internal, not exposed
    
    @classmethod
    def create(cls, gen: ChunkedTerrainGenerator, 
               num_lakes: int = 3,
               climate: ClimatePreset = None) -> 'MasterGeology':
        coarse = gen._coarse_terrain
        plates = gen._plates
        
        if climate:
            coarse.climate = climate
        coarse.compute_weather()
        coarse.carve_to_ocean(num_lakes=num_lakes)
        
        basins = DrainageBasins(coarse)
        
        master = cls(terrain=coarse, plates=plates, basins=basins,
                     world_id=gen.coarse_world_id, _gen=gen)
        return master
    
    def watershed_lookup(self) -> dict:
        """Return {(q,r,s): watershed_id} for projection."""
        lookup = {}
        for ws_id, ws in enumerate(self.basins.watersheds):
            for hex_idx in ws.region.hexes:
                pos = self.terrain.hexGrid.index_to_hexposition(
                    hex_idx, origin_index=self.terrain.hexGrid.midpoint)
                lookup[(pos.q, pos.r, pos.s)] = ws_id
        return lookup
    
    def detail_for_rect(self, min_q: int, max_q: int, 
                        min_r: int, max_r: int) -> 'DetailGeology':
        """Generate detail geology for a rectangular region."""
        # Internally uses chunks - caller doesn't need to know
        chunk_refs = self._gen._chunks_intersecting_rect(min_q, max_q, min_r, max_r)
        chunks = [self._gen.generate_chunk(ref) for ref in chunk_refs]
        
        # Copy halos between adjacent chunks
        for i, chunk in enumerate(chunks):
            for other in chunks[:i]:
                chunk.copy_from(other)
        
        chunk_set = ChunkSet(chunks, self.terrain, self._gen.scale)
        
        # Compute weather on chunks
        chunk_set.compute_weather(self.terrain, self._gen.scale)
        
        return DetailGeology._from_chunk_set(chunk_set, self, self._gen.scale, chunk_refs)
    
    def detail_for_center(self, center: HexPosition, radius: int) -> 'DetailGeology':
        """Generate detail geology around a center point."""
        min_q = center.q - radius
        max_q = center.q + radius
        min_r = center.r - radius
        max_r = center.r + radius
        return self.detail_for_rect(min_q, max_q, min_r, max_r)
    
    def detail_for_watershed(self, watershed_id: int, padding: int = 5) -> 'DetailGeology':
        """Generate detail geology for a specific watershed."""
        ws = self.basins.watersheds[watershed_id]
        
        # Get bounds from watershed hexes
        positions = [self.terrain.hexGrid.index_to_hexposition(
            idx, origin_index=self.terrain.hexGrid.midpoint) 
            for idx in ws.region.hexes]
        
        # Scale to fine coordinates
        scale = self._gen.scale
        qs = [int(p.q * scale) for p in positions]
        rs = [int(p.r * scale) for p in positions]
        
        return self.detail_for_rect(
            min(qs) - padding, max(qs) + padding,
            min(rs) - padding, max(rs) + padding
        )


@dataclass  
class DetailGeology:
    """Fine-level geology projected from master."""
    terrain: Terrain
    master: MasterGeology
    scale: float
    chunk_refs: list[ChunkRef]  # Internal tracking
    
    @classmethod
    def _from_chunk_set(cls, chunk_set: ChunkSet, master: MasterGeology,
                        scale: float, chunk_refs: list) -> 'DetailGeology':
        """Internal: build from ChunkSet."""
        terrain, _ = chunk_set.to_weather_terrain(master.terrain, scale, fill_gaps=False)
        
        detail = cls(terrain=terrain, master=master, 
                     scale=scale, chunk_refs=chunk_refs)
        detail._project_watersheds(chunk_set)
        return detail
    
    def _project_watersheds(self, chunk_set: ChunkSet):
        """Assign watershed_id by lookup from master."""
        lookup = self.master.watershed_lookup()
        
        positions = list(chunk_set.pos_to_chunk.keys())
        min_q = min(p.q for p in positions)
        min_r = min(p.r for p in positions)
        origin_world = HexPosition(min_q, min_r, -min_q - min_r)
        
        n = len(self.terrain.elevations)
        self.terrain.fields['watershed_id'] = np.full(n, -1, dtype=np.int32)
        
        for world_pos, (chunk, idx, is_core) in chunk_set.pos_to_chunk.items():
            if not is_core or chunk.elevations[idx] <= 0:
                continue
            
            coarse_q = int(round(world_pos.q / self.scale))
            coarse_r = int(round(world_pos.r / self.scale))
            coarse_s = -coarse_q - coarse_r
            
            ws_id = lookup.get((coarse_q, coarse_r, coarse_s), -1)
            
            local_pos = world_pos - origin_world
            grid_idx = self.terrain.hexGrid.hexposition_to_index(local_pos, origin_index=0)
            if 0 <= grid_idx < n:
                self.terrain.fields['watershed_id'][grid_idx] = ws_id
    
    @property
    def soils(self):
        if not hasattr(self, '_soils'):
            self._soils = SoilSystem.from_terrain(self.terrain)
        return self._soils


Should these classes have a storage associate with it?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good question. I'd suggest:

**MasterGeology** - yes, needs storage for persistence:
- Save/load the computed basins, watersheds, weather
- This is the expensive computation we want to do once

**DetailGeology** - probably not directly:
- It's derived from master + chunks
- Chunks already have caching via `_gen.storage`
- Detail is cheap to regenerate from cached chunks

```python
@dataclass
class MasterGeology:
    terrain: Terrain
    plates: list
    basins: DrainageBasins
    world_id: int = None
    _gen: 'ChunkedTerrainGenerator' = None
    _storage: 'GeoStorage' = None  # For persistence
    
    @classmethod
    def create(cls, gen: ChunkedTerrainGenerator, ...) -> 'MasterGeology':
        # ... existing code ...
        master = cls(terrain=coarse, plates=plates, basins=basins,
                     world_id=gen.coarse_world_id, 
                     _gen=gen, 
                     _storage=gen.storage)
        return master
    
    def save(self):
        """Persist master geology to storage."""
        # Save terrain with all fields (watershed_id, weather, etc.)
        self._storage.save_world(self.terrain, name=f"master_{self.world_id}")
        # Save watershed metadata
        self._storage.save_watersheds(self.basins.watersheds, self.world_id)
    
    @classmethod
    def load(cls, storage: 'GeoStorage', world_id: int) -> 'MasterGeology':
        """Load master geology from storage."""
        terrain = storage.load_world(world_id)
        watersheds = storage.load_watersheds(world_id)
        basins = DrainageBasins._from_loaded(terrain, watersheds)
        # Note: _gen would be None - can't generate new detail without it
        return cls(terrain=terrain, plates=[], basins=basins,
                   world_id=world_id, _storage=storage)
```

The tradeoff: a loaded `MasterGeology` without `_gen` can't create new `DetailGeology`. You'd need to recreate the generator. Is that acceptable, or should we persist enough to reconstruct `_gen`?

I like this. I do think we want MasterGeology to be persitable using encoding/and decoding. mayve using something like the string encoding from terrain

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good idea. Following the pattern from `Geology.decode()`:

```python
@dataclass
class MasterGeology:
    terrain: Terrain
    plates: list
    basins: DrainageBasins
    world_id: int = None
    _gen: 'ChunkedTerrainGenerator' = None
    _storage: 'GeoStorage' = None
    
    # ... existing methods ...
    
    def encode(self) -> str:
        """Encode MasterGeology to string format."""
        lines = []
        lines.append(f"world_id:{self.world_id or 0}")
        lines.append(f"scale:{self._gen.scale if self._gen else 5.0}")
        
        # Encode plates
        lines.append(f"+plates:{len(self.plates)}")
        for plate in self.plates:
            lines.append(plate.encode())
        lines.append("-plates")
        
        # Encode terrain
        lines.append("+terrain")
        lines.append(self.terrain.encode())
        lines.append("-terrain")
        
        # Encode basins/watersheds
        lines.append("+basins")
        lines.append(self.basins.encode())
        lines.append("-basins")
        
        # Encode generator params (for reconstruction)
        if self._gen:
            lines.append("+gen_params")
            lines.append(f"coarse_rings:{self._gen.coarse_rings}")
            lines.append(f"coarse_radius:{self._gen.coarse_radius}")
            lines.append(f"fine_radius:{self._gen.fine_radius}")
            lines.append(f"chunk_rings:{self._gen.chunk_rings}")
            lines.append(f"halo_rings:{self._gen.halo_rings}")
            lines.append("-gen_params")
        
        return '\n'.join(lines)
    
    @classmethod
    def decode(cls, s: str, storage: 'GeoStorage' = None) -> 'MasterGeology':
        """Decode MasterGeology from string format."""
        lines = s.strip().split('\n')
        
        world_id = None
        scale = 5.0
        plates_raw = []
        terrain = None
        basins_raw = None
        gen_params = {}
        
        i = 0
        while i < len(lines):
            line = lines[i]
            
            if line.startswith('world_id:'):
                world_id = int(line.split(':')[1])
            
            elif line.startswith('scale:'):
                scale = float(line.split(':')[1])
            
            elif line.startswith('+plates:'):
                i += 1
                while not lines[i].startswith('-plates'):
                    plates_raw.append(lines[i])
                    i += 1
            
            elif line.startswith('+terrain'):
                terrain_lines = []
                i += 1
                while not lines[i].startswith('-terrain'):
                    terrain_lines.append(lines[i])
                    i += 1
                terrain = Terrain.decode('\n'.join(terrain_lines))
            
            elif line.startswith('+basins'):
                basins_lines = []
                i += 1
                while not lines[i].startswith('-basins'):
                    basins_lines.append(lines[i])
                    i += 1
                basins_raw = '\n'.join(basins_lines)
            
            elif line.startswith('+gen_params'):
                i += 1
                while not lines[i].startswith('-gen_params'):
                    key, val = lines[i].split(':')
                    gen_params[key] = float(val) if '.' in val else int(val)
                    i += 1
            
            i += 1
        
        # Decode plates with hex_grid
        plates = [Plate.decode(p, terrain.hexGrid) for p in plates_raw]
        
        # Decode basins
        soils = SoilSystem.from_plates(terrain, plates)
        basins = DrainageBasins.decode(basins_raw, terrain, soils)
        
        # Reconstruct generator if we have params and storage
        gen = None
        if gen_params and storage:
            gen = ChunkedTerrainGenerator(
                storage=storage,
                coarse_rings=gen_params.get('coarse_rings', 15),
                coarse_radius=gen_params.get('coarse_radius', 50),
                fine_radius=gen_params.get('fine_radius', 10),
                chunk_rings=gen_params.get('chunk_rings', 8),
                halo_rings=gen_params.get('halo_rings', 3)
            )
            gen._coarse_terrain = terrain
            gen._plates = plates
            gen.coarse_world_id = world_id
        
        return cls(
            terrain=terrain,
            plates=plates,
            basins=basins,
            world_id=world_id,
            _gen=gen,
            _storage=storage
        )
    
    def save_to_file(self, path: str):
        """Save to file."""
        Path(path).write_text(self.encode())
    
    @classmethod
    def load_from_file(cls, path: str, storage: 'GeoStorage' = None) -> 'MasterGeology':
        """Load from file."""
        return cls.decode(Path(path).read_text(), storage)
```

Key points:
- Encodes generator params so we can reconstruct `_gen` on decode
- If storage provided on decode, fully reconstructs the generator for creating details
- Without storage, you get read-only access to the master (can view, but not generate details)

Want me to add a test cell?

Can you write the full code for the MasterGeology class?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
@dataclass
class MasterGeology:
    """Coarse-level geology with precomputed watersheds."""
    terrain: Terrain
    plates: list
    basins: DrainageBasins
    world_id: int = None
    _gen: 'ChunkedTerrainGenerator' = None
    _storage: 'GeoStorage' = None
    
    @classmethod
    def create(cls, gen: ChunkedTerrainGenerator, 
               num_lakes: int = 3,
               climate: ClimatePreset = None) -> 'MasterGeology':
        """Generate and compute master geology."""
        coarse = gen._coarse_terrain
        plates = gen._plates
        
        if climate:
            coarse.climate = climate
        coarse.compute_weather()
        coarse.carve_to_ocean(num_lakes=num_lakes)
        
        basins = DrainageBasins(coarse)
        
        return cls(
            terrain=coarse, 
            plates=plates, 
            basins=basins,
            world_id=gen.coarse_world_id, 
            _gen=gen,
            _storage=gen.storage
        )
    
    def watershed_lookup(self) -> dict:
        """Return {(q,r,s): watershed_id} for projection."""
        lookup = {}
        for ws_id, ws in enumerate(self.basins.watersheds):
            for hex_idx in ws.region.hexes:
                pos = self.terrain.hexGrid.index_to_hexposition(
                    hex_idx, origin_index=self.terrain.hexGrid.midpoint)
                lookup[(pos.q, pos.r, pos.s)] = ws_id
        return lookup
    
    # === DETAIL GENERATION ===
    
    def detail_for_rect(self, min_q: int, max_q: int, 
                        min_r: int, max_r: int) -> 'DetailGeology':
        """Generate detail geology for a rectangular region."""
        if self._gen is None:
            raise ValueError("Cannot generate detail without generator. Load with storage to enable.")
        
        chunk_refs = self._gen._chunks_intersecting_rect(min_q, max_q, min_r, max_r)
        chunks = [self._gen.generate_chunk(ref) for ref in chunk_refs]
        
        # Copy halos between adjacent chunks
        for i, chunk in enumerate(chunks):
            for other in chunks[:i]:
                chunk.copy_from(other)
        
        chunk_set = ChunkSet(chunks, self.terrain, self._gen.scale)
        chunk_set.compute_weather(self.terrain, self._gen.scale)
        
        return DetailGeology._from_chunk_set(chunk_set, self, self._gen.scale, chunk_refs)
    
    def detail_for_center(self, center: HexPosition, radius: int) -> 'DetailGeology':
        """Generate detail geology around a center point."""
        min_q = center.q - radius
        max_q = center.q + radius
        min_r = center.r - radius
        max_r = center.r + radius
        return self.detail_for_rect(min_q, max_q, min_r, max_r)
    
    def detail_for_watershed(self, watershed_id: int, padding: int = 5) -> 'DetailGeology':
        """Generate detail geology for a specific watershed."""
        ws = self.basins.watersheds[watershed_id]
        
        positions = [self.terrain.hexGrid.index_to_hexposition(
            idx, origin_index=self.terrain.hexGrid.midpoint) 
            for idx in ws.region.hexes]
        
        scale = self._gen.scale
        qs = [int(p.q * scale) for p in positions]
        rs = [int(p.r * scale) for p in positions]
        
        return self.detail_for_rect(
            min(qs) - padding, max(qs) + padding,
            min(rs) - padding, max(rs) + padding
        )
    
    # === ENCODING/DECODING ===
    
    def encode(self) -> str:
        """Encode MasterGeology to string format."""
        lines = []
        lines.append(f"world_id:{self.world_id or 0}")
        lines.append(f"scale:{self._gen.scale if self._gen else 5.0}")
        
        # Encode plates
        lines.append(f"+plates:{len(self.plates)}")
        for plate in self.plates:
            lines.append(plate.encode())
        lines.append("-plates")
        
        # Encode terrain
        lines.append("+terrain")
        lines.append(self.terrain.encode())
        lines.append("-terrain")
        
        # Encode basins
        lines.append("+basins")
        lines.append(self.basins.encode())
        lines.append("-basins")
        
        # Encode generator params
        if self._gen:
            lines.append("+gen_params")
            lines.append(f"coarse_rings:{self._gen.coarse_rings}")
            lines.append(f"coarse_radius:{self._gen.coarse_radius}")
            lines.append(f"fine_radius:{self._gen.fine_radius}")
            lines.append(f"chunk_rings:{self._gen.chunk_rings}")
            lines.append(f"halo_rings:{self._gen.halo_rings}")
            lines.append("-gen_params")
        
        return '\n'.join(lines)
    
    @classmethod
    def decode(cls, s: str, storage: 'GeoStorage' = None) -> 'MasterGeology':
        """Decode MasterGeology from string format."""
        lines = s.strip().split('\n')
        
        world_id = None
        scale = 5.0
        plates_raw = []
        terrain = None
        basins_raw = None
        gen_params = {}
        
        i = 0
        while i < len(lines):
            line = lines[i]
            
            if line.startswith('world_id:'):
                world_id = int(line.split(':')[1])
            
            elif line.startswith('scale:'):
                scale = float(line.split(':')[1])
            
            elif line.startswith('+plates:'):
                i += 1
                while not lines[i].startswith('-plates'):
                    plates_raw.append(lines[i])
                    i += 1
            
            elif line.startswith('+terrain'):
                terrain_lines = []
                i += 1
                while not lines[i].startswith('-terrain'):
                    terrain_lines.append(lines[i])
                    i += 1
                terrain = Terrain.decode('\n'.join(terrain_lines))
            
            elif line.startswith('+basins'):
                basins_lines = []
                i += 1
                while not lines[i].startswith('-basins'):
                    basins_lines.append(lines[i])
                    i += 1
                basins_raw = '\n'.join(basins_lines)
            
            elif line.startswith('+gen_params'):
                i += 1
                while not lines[i].startswith('-gen_params'):
                    key, val = lines[i].split(':')
                    gen_params[key] = float(val) if '.' in val else int(val)
                    i += 1
            
            i += 1
        
        # Decode plates
        plates = [Plate.decode(p, terrain.hexGrid) for p in plates_raw]
        
        # Decode basins
        soils = SoilSystem.from_plates(terrain, plates)
        basins = DrainageBasins.decode(basins_raw, terrain, soils)
        
        # Reconstruct generator if possible
        gen = None
        if gen_params and storage:
            gen = ChunkedTerrainGenerator(
                storage=storage,
                coarse_rings=gen_params.get('coarse_rings', 15),
                coarse_radius=gen_params.get('coarse_radius', 50),
                fine_radius=gen_params.get('fine_radius', 10),
                chunk_rings=gen_params.get('chunk_rings', 8),
                halo_rings=gen_params.get('halo_rings', 3)
            )
            gen._coarse_terrain = terrain
            gen._plates = plates
            gen.coarse_world_id = world_id
        
        return cls(
            terrain=terrain,
            plates=plates,
            basins=basins,
            world_id=world_id,
            _gen=gen,
            _storage=storage
        )
    
    # === FILE I/O ===
    
    def save_to_file(self, path: str):
        """Save to file."""
        Path(path).write_text(self.encode())
    
    @classmethod
    def load_from_file(cls, path: str, storage: 'GeoStorage' = None) -> 'MasterGeology':
        """Load from file."""
        return cls.decode(Path(path).read_text(), storage)
    
    # === DISPLAY ===
    
    def show(self, radius: float = 12):
        """Display the coarse terrain."""
        self.terrain.hexGrid.adjustRadius(radius)
        self.terrain.colorMap()
        self.terrain.hexGrid.update()
        return self.terrain.hexGrid.builder.show()
    
    def show_watersheds(self, radius: float = 12):
        """Display terrain with watershed overlay."""
        self.terrain.hexGrid.adjustRadius(radius)
        self.terrain.colorMap()
        self.terrain.hexGrid.update()
        
        # Add watershed overlay
        ws_colors = StyleCSS.seaborn("tab20", min(20, len(self.basins.watersheds)))
        ws_overlay = ""
        
        for ws_id, ws in enumerate(self.basins.watersheds):
            color = ws_colors[ws_id % len(ws_colors)].properties.get('fill', '#888')
            for hex_idx in ws.region.hexes:
                hex_obj = self.terrain.hexGrid.hexes[hex_idx]
                ws_overlay += f'<polygon points="{" ".join(f"{v.x},{v.y}" for v in hex_obj.v)}" fill="{color}" opacity="0.4"/>\n'
        
        self.terrain.hexGrid.builder.adjust("watersheds", ws_overlay)
        return self.terrain.hexGrid.builder.show()
```

#| export
### Master Geology

In [ ]:
#| export
@dataclass
class MasterGeology:
    """Coarse-level geology with precomputed watersheds."""
    terrain: Terrain
    plates: list
    basins: DrainageBasins
    world_id: int = None
    _gen: 'ChunkedTerrainGenerator' = None
    _storage: 'GeoStorage' = None
    
    @classmethod
    def create(cls, gen: ChunkedTerrainGenerator, 
               num_lakes: int = 3,
               climate: ClimatePreset = None,
               geo:GeoBounds = None) -> 'MasterGeology':
        """Generate and compute master geology."""
        coarse = gen._coarse_terrain
        plates = gen._plates

        if geo is None:
            geo = GeoBounds(
            lat_min=20.57,   # Southern tip (near Makena)
            lat_max=21.03,   # Northern tip (near Kahakuloa)
            lon_min=-156.69, # Western tip (West Maui)
            lon_max=-155.97  # Eastern tip (Haleakalā/Hāna)
            )
        coarse.geo = geo
        
        if climate:
            coarse.climate = climate
        coarse.compute_weather()
        coarse.carve_to_ocean(num_lakes=num_lakes)
        
        basins = DrainageBasins(coarse)
        
        return cls(
            terrain=coarse, 
            plates=plates, 
            basins=basins,
            world_id=gen.coarse_world_id, 
            _gen=gen,
            _storage=gen.storage
        )
    
    def watershed_lookup(self) -> dict:
        """Return {(q,r,s): watershed_id} for projection."""
        lookup = {}
        for ws_id, ws in enumerate(self.basins.sheds):
            for hex_idx in ws.region.hexes:
                pos = self.terrain.hexGrid.index_to_hexposition(
                    hex_idx, origin_index=self.terrain.hexGrid.midpoint)
                lookup[(pos.q, pos.r, pos.s)] = ws_id
        return lookup
    
    # === DETAIL GENERATION ===
    
    def detail_for_rect(self, min_q: int, max_q: int, 
                        min_r: int, max_r: int) -> 'DetailGeology':
        """Generate detail geology for a rectangular region."""
        if self._gen is None:
            raise ValueError("Cannot generate detail without generator. Load with storage to enable.")
        
        chunk_refs = self._gen._chunks_intersecting_rect(min_q, max_q, min_r, max_r)
        chunks = [self._gen.generate_chunk(ref) for ref in chunk_refs]
        
        # Copy halos between adjacent chunks
        for i, chunk in enumerate(chunks):
            for other in chunks[:i]:
                chunk.copy_from(other)
        
        chunk_set = ChunkSet(chunks, self.terrain, self._gen.scale)
        chunk_set.compute_weather(self.terrain, self._gen.scale)
        
        return DetailGeology._from_chunk_set(chunk_set, self, self._gen.scale, chunk_refs)
    
    def detail_for_center(self, center: HexPosition, radius: int) -> 'DetailGeology':
        """Generate detail geology around a center point."""
        min_q = center.q - radius
        max_q = center.q + radius
        min_r = center.r - radius
        max_r = center.r + radius
        return self.detail_for_rect(min_q, max_q, min_r, max_r)
    
    def detail_for_watershed(self, watershed_id: int, padding: int = 5) -> 'DetailGeology':
        """Generate detail geology for a specific watershed."""
        ws = self.basins.watersheds[watershed_id]
        
        positions = [self.terrain.hexGrid.index_to_hexposition(
            idx, origin_index=self.terrain.hexGrid.midpoint) 
            for idx in ws.region.hexes]
        
        scale = self._gen.scale
        qs = [int(p.q * scale) for p in positions]
        rs = [int(p.r * scale) for p in positions]
        
        return self.detail_for_rect(
            min(qs) - padding, max(qs) + padding,
            min(rs) - padding, max(rs) + padding
        )
    
    # === ENCODING/DECODING ===
    
    def encode(self) -> str:
        """Encode MasterGeology to string format."""
        lines = []
        lines.append(f"world_id:{self.world_id or 0}")
        lines.append(f"scale:{self._gen.scale if self._gen else 5.0}")
        
        # Encode plates
        lines.append(f"+plates:{len(self.plates)}")
        for plate in self.plates:
            lines.append(plate.encode())
        lines.append("-plates")
        
        # Encode terrain
        lines.append("+terrain")
        lines.append(self.terrain.encode())
        lines.append("-terrain")
        
        # Encode basins
        lines.append("+basins")
        lines.append(self.basins.encode())
        lines.append("-basins")
        
        # Encode generator params
        if self._gen:
            lines.append("+gen_params")
            lines.append(f"coarse_rings:{self._gen.coarse_rings}")
            lines.append(f"coarse_radius:{self._gen.coarse_radius}")
            lines.append(f"fine_radius:{self._gen.fine_radius}")
            lines.append(f"chunk_rings:{self._gen.chunk_rings}")
            lines.append(f"halo_rings:{self._gen.halo_rings}")
            lines.append("-gen_params")
        
        return '\n'.join(lines)
    
    @classmethod
    def decode(cls, s: str, storage: 'GeoStorage' = None) -> 'MasterGeology':
        """Decode MasterGeology from string format."""
        lines = s.strip().split('\n')
        
        world_id = None
        scale = 5.0
        plates_raw = []
        terrain = None
        basins_raw = None
        gen_params = {}
        
        i = 0
        while i < len(lines):
            line = lines[i]
            
            if line.startswith('world_id:'):
                world_id = int(line.split(':')[1])
            
            elif line.startswith('scale:'):
                scale = float(line.split(':')[1])
            
            elif line.startswith('+plates:'):
                i += 1
                while not lines[i].startswith('-plates'):
                    plates_raw.append(lines[i])
                    i += 1
            
            elif line.startswith('+terrain'):
                terrain_lines = []
                i += 1
                while not lines[i].startswith('-terrain'):
                    terrain_lines.append(lines[i])
                    i += 1
                terrain = Terrain.decode('\n'.join(terrain_lines))
            
            elif line.startswith('+basins'):
                basins_lines = []
                i += 1
                while not lines[i].startswith('-basins'):
                    basins_lines.append(lines[i])
                    i += 1
                basins_raw = '\n'.join(basins_lines)
            
            elif line.startswith('+gen_params'):
                i += 1
                while not lines[i].startswith('-gen_params'):
                    key, val = lines[i].split(':')
                    gen_params[key] = float(val) if '.' in val else int(val)
                    i += 1
            
            i += 1
        
        # Decode plates
        plates = [Plate.decode(p, terrain.hexGrid) for p in plates_raw]
        
        # Decode basins
        soils = SoilSystem.from_plates(terrain, plates)
        basins = DrainageBasins.decode(basins_raw, terrain, soils)
        
        # Reconstruct generator if possible
        gen = None
        if gen_params and storage:
            gen = ChunkedTerrainGenerator(
                storage=storage,
                coarse_rings=gen_params.get('coarse_rings', 15),
                coarse_radius=gen_params.get('coarse_radius', 50),
                fine_radius=gen_params.get('fine_radius', 10),
                chunk_rings=gen_params.get('chunk_rings', 8),
                halo_rings=gen_params.get('halo_rings', 3)
            )
            gen._coarse_terrain = terrain
            gen._plates = plates
            gen.coarse_world_id = world_id
        
        return cls(
            terrain=terrain,
            plates=plates,
            basins=basins,
            world_id=world_id,
            _gen=gen,
            _storage=storage
        )
    
    # === FILE I/O ===
    
    def save_to_file(self, path: str):
        """Save to file."""
        Path(path).write_text(self.encode())
    
    @classmethod
    def load_from_file(cls, path: str, storage: 'GeoStorage' = None) -> 'MasterGeology':
        """Load from file."""
        return cls.decode(Path(path).read_text(), storage)
    
    # === DISPLAY ===
    
    def show(self, radius: float = 12):
        """Display the coarse terrain."""
        self.terrain.hexGrid.adjustRadius(radius)
        self.terrain.colorMap()
        self.terrain.hexGrid.update()
        return self.terrain.hexGrid.builder.show()
    
    def show_watersheds(self, radius: float = 12):
        """Display terrain with watershed overlay."""
        self.terrain.hexGrid.adjustRadius(radius)
        self.terrain.colorMap()
        self.terrain.hexGrid.update()
        
        # Add watershed overlay
        ws_colors = StyleCSS.seaborn("tab20", min(20, len(self.basins.sheds)))
        ws_overlay = ""
        
        for ws_id, ws in enumerate(self.basins.sheds):
            color = ws_colors[ws_id % len(ws_colors)].properties.get('fill', '#888')
            for hex_idx in ws.region.hexes:
                hex_obj = self.terrain.hexGrid.hexes[hex_idx]
                ws_overlay += f'<polygon points="{" ".join(f"{v.x},{v.y}" for v in hex_obj.v)}" fill="{color}" opacity="0.4"/>\n'
        
        self.terrain.hexGrid.builder.adjust("watersheds", ws_overlay)
        return self.terrain.hexGrid.builder.show()


### detail geology

In [ ]:
@dataclass  
class DetailGeology:
    """Fine-level geology projected from master."""
    terrain: Terrain
    master: MasterGeology
    scale: float
    chunk_refs: list[ChunkRef]
    _basins: DrainageBasins = None
    _chunk_set: ChunkSet = None  # Keep reference for projection
    
    @classmethod
    def _from_chunk_set(cls, chunk_set: ChunkSet, master: MasterGeology,
                        scale: float, chunk_refs: list) -> 'DetailGeology':
        """Internal: build from ChunkSet."""
        terrain, _ = chunk_set.to_weather_terrain(master.terrain, scale, fill_gaps=False)
        
        detail = cls(terrain=terrain, master=master, 
                     scale=scale, chunk_refs=chunk_refs,
                     _chunk_set=chunk_set)
        
        # Project watershed IDs from master (O(n) lookup)
        detail._project_watersheds()
        
        # Build basins from projected IDs using from_regions (handles carving)
        detail._basins = detail._build_basins_from_projection()
        
        return detail
    
    def _project_watersheds(self):
        """Assign watershed_id by lookup from master."""
        lookup = self.master.watershed_lookup()
        
        positions = list(self._chunk_set.pos_to_chunk.keys())
        min_q = min(p.q for p in positions)
        min_r = min(p.r for p in positions)
        origin_world = HexPosition(min_q, min_r, -min_q - min_r)
        
        n = len(self.terrain.elevations)
        self.terrain.fields['watershed_id'] = np.full(n, -1, dtype=np.int32)
        
        for world_pos, (chunk, idx, is_core) in self._chunk_set.pos_to_chunk.items():
            if not is_core or chunk.elevations[idx] <= 0:
                continue
            
            # Map fine position to coarse
            coarse_q = int(round(world_pos.q / self.scale))
            coarse_r = int(round(world_pos.r / self.scale))
            coarse_s = -coarse_q - coarse_r
            
            ws_id = lookup.get((coarse_q, coarse_r, coarse_s), -1)
            
            local_pos = world_pos - origin_world
            grid_idx = self.terrain.hexGrid.hexposition_to_index(local_pos, origin_index=0)
            if 0 <= grid_idx < n:
                self.terrain.fields['watershed_id'][grid_idx] = ws_id
    
    def _build_basins_from_projection(self) -> DrainageBasins:
        """Build DrainageBasins from projected watershed_id using from_regions."""
        
        # Group hexes by watershed_id
        ws_hexes = {}  # ws_id -> set of hex indices
        invalid = getattr(self.terrain.hexGrid, 'invalidRegion', set())
        
        for idx in range(len(self.terrain.elevations)):
            if idx in invalid:
                continue
            ws_id = int(self.terrain.fields.get('watershed_id', np.array([-1]))[idx])
            if ws_id >= 0:
                if ws_id not in ws_hexes:
                    ws_hexes[ws_id] = set()
                ws_hexes[ws_id].add(idx)
        
        # Build regions list: (hex_set, mouth_idx)
        # Mouth is the lowest elevation hex in each watershed
        regions = []
        for ws_id in sorted(ws_hexes.keys()):  # Sort for consistent ordering
            hex_set = ws_hexes[ws_id]
            if not hex_set:
                continue
            
            # Find mouth (lowest elevation in the set)
            mouth_idx = min(hex_set, key=lambda i: self.terrain.elevations[i])
            regions.append((hex_set, mouth_idx))
        
        # Use from_regions - handles carving and river building
        return DrainageBasins.from_regions(self.terrain, regions)
    
    @property
    def basins(self) -> DrainageBasins:
        """Get drainage basins (pre-built from projection)."""
        return self._basins
    
    @property
    def soils(self):
        """Compute soils lazily (cheap operation)."""
        if not hasattr(self, '_soils'):
            self._soils = SoilSystem.from_terrain(self.terrain)
        return self._soils
    
    
    def show_temperature(self, radius: float = 8):
        """Display temperature map."""
        self.terrain.hexGrid.adjustRadius(radius)
        temp_overlay = self.terrain.render_icon_temperature()
        self.terrain.hexGrid.builder.adjust("hexes", temp_overlay)
        return self.terrain.hexGrid.builder.show()
    
    def show_precipitation(self, radius: float = 8):
        """Display precipitation map."""
        self.terrain.hexGrid.adjustRadius(radius)
        self.terrain.colorMap()
        self.terrain.hexGrid.update()
        self.terrain.add_rain_overlay(layer_name="precip_overlay")
        return self.terrain.hexGrid.builder.show()


In [ ]:
# === DISPLAY ===
@patch
def show(self:DetailGeology, radius: float = 8):
    """Display detail terrain."""
    self.terrain.hexGrid.adjustRadius(radius)
    self.terrain.colorMap()
    self.terrain.hexGrid.update()
    return self.terrain.hexGrid.builder.show()

@patch
def show_watersheds(self:DetailGeology, radius: float = 8, top_n: int = 5, max_width: float = 6):
    """Display detail terrain with watersheds and rivers."""
    self.terrain.hexGrid.adjustRadius(radius)
    self.terrain.colorMap()
    self.terrain.hexGrid.update()
    
    # Draw watershed regions + rivers using DrainageBasins
    river_svg = self.basins.draw_watersheds(top_n=top_n, max_width=max_width)
    print(f"river layer {river_svg}")
    self.terrain.hexGrid.builder.adjust("rivers", river_svg)
    
    # Add watershed color overlay
    ws_colors = StyleCSS.seaborn("tab10", min(top_n, len(self.basins.sheds)))
    major_sheds = self.basins.get_major(top_n) if hasattr(self.basins, 'get_major') else self.basins.sheds[:top_n]
    
    ws_overlay = ""
    for i, ws in enumerate(major_sheds):
        color = ws_colors[i % len(ws_colors)].properties.get('fill', '#888')
        for hex_idx in ws.region.hexes:
            if hex_idx < len(self.terrain.hexGrid.hexes):
                hex_obj = self.terrain.hexGrid.hexes[hex_idx]
                ws_overlay += f'<polygon points="{" ".join(f"{v.x},{v.y}" for v in hex_obj.v)}" fill="{color}" opacity="0.3"/>\n'
    
    self.terrain.hexGrid.builder.adjust("watersheds", ws_overlay)
    
    # Title
    textStyle = StyleCSS("textStyle", fill="white", font_size="1.2em", 
                            filter="drop-shadow(2px 2px 1px rgba(0,0,0,.7))")
    self.terrain.hexGrid.builder.add_style(textStyle)
    
    valid_count = len(self.terrain.elevations) - len(getattr(self.terrain.hexGrid, 'invalidRegion', set()))
    self.terrain.hexGrid.builder.add_centered_text(
        f"Detail: {valid_count} hexes, {len(self.basins.sheds)} watersheds", 
        y_offset=-self.terrain.hexGrid.builder.height/2 + 30, 
        class_name=textStyle.name
    )
    
    return self.terrain.hexGrid.builder.show()
    

In [ ]:
@patch
def show_watersheds(self: DetailGeology, radius: float = 8, top_n: int = 5, max_width: float = 6):
    """Display detail terrain with watersheds projected from master."""
    self.terrain.hexGrid.adjustRadius(radius)
    self.terrain.colorMap()
    self.terrain.hexGrid.update()
    
    grid = self.terrain.hexGrid
    invalid = getattr(grid, 'invalidRegion', set())
    valid_indices = set(range(len(self.terrain.elevations))) - invalid
    
    # Get major watersheds from MASTER
    major = self.master.basins.get_major(top_n)
    ws_colors = StyleCSS.seaborn("tab10", len(major))
    
    # Map master watershed index -> color
    major_ids = {}
    for i, ws in enumerate(major):
        for full_id, full_ws in enumerate(self.master.basins.sheds):
            if full_ws is ws:
                major_ids[full_id] = ws_colors[i].properties.get('fill', '#888')
                break
    
    # Watershed overlay using projected IDs
    ws_overlay = ""
    for grid_idx in valid_indices:
        ws_id = int(self.terrain.fields.get('watershed_id', np.array([-1]))[grid_idx])
        if ws_id in major_ids:
            hex_obj = grid.hexes[grid_idx]
            ws_overlay += f'<polygon points="{" ".join(f"{v.x},{v.y}" for v in hex_obj.v)}" fill="{major_ids[ws_id]}" opacity="0.35"/>\n'
    
    grid.builder.adjust("watersheds", ws_overlay)
    
    # Project rivers from master onto detail grid
    river_svg = self._project_rivers_from_master(major, ws_colors, max_width)
    grid.builder.adjust("rivers", river_svg)
    
    # Title
    textStyle = StyleCSS("textStyle", fill="white", font_size="1.2em", 
                         filter="drop-shadow(2px 2px 1px rgba(0,0,0,.7))")
    grid.builder.add_style(textStyle)
    grid.builder.add_centered_text(
        f"Detail: {len(valid_indices)} hexes, top {top_n} watersheds", 
        y_offset=-grid.builder.height/2 + 30, 
        class_name=textStyle.name
    )
    
    return grid.builder.show()


@patch
def _project_rivers_from_master(self: DetailGeology, major_sheds: list, 
                                 ws_colors: list, max_width: float = 6) -> str:
    """Project master river paths onto detail terrain."""
    grid = self.terrain.hexGrid
    invalid = getattr(grid, 'invalidRegion', set())
    valid_indices = set(range(len(self.terrain.elevations))) - invalid
    
    if not valid_indices:
        return ""
    
    # Compute origin from chunk_set positions
    if self._chunk_set:
        positions = list(self._chunk_set.pos_to_chunk.keys())
        min_q = min(p.q for p in positions)
        min_r = min(p.r for p in positions)
        origin_world = HexPosition(min_q, min_r, -min_q - min_r)
    else:
        origin_world = HexPosition(0, 0, 0)
    
    river_svg = ""
    
    for i, ws in enumerate(major_sheds):
        color = ws_colors[i].properties.get('fill', '#1565c0')
        width = max(1, max_width * (1 - i * 0.1))
        
        # Get river hexes from master
        river_hexes = list(ws.tributary.hexes) if ws.tributary else []
        if not river_hexes:
            continue
        
        # Sort by elevation (high to low for drawing order)
        river_hexes.sort(key=lambda h: self.master.terrain.elevations[h], reverse=True)
        
        # Project each coarse river hex to fine coordinates
        points = []
        for coarse_idx in river_hexes:
            coarse_pos = self.master.terrain.hexGrid.index_to_hexposition(
                coarse_idx, origin_index=self.master.terrain.hexGrid.midpoint)
            
            # Scale to fine world coordinates
            fine_q = int(coarse_pos.q * self.scale)
            fine_r = int(coarse_pos.r * self.scale)
            fine_world = HexPosition(fine_q, fine_r, -fine_q - fine_r)
            
            # Convert to local grid coordinates
            local_pos = fine_world - origin_world
            grid_idx = grid.hexposition_to_index(local_pos, origin_index=0)
            
            if 0 <= grid_idx < len(grid.hexes) and grid_idx in valid_indices:
                hex_obj = grid.hexes[grid_idx]
                points.append((hex_obj.center.x, hex_obj.center.y))
        
        if len(points) >= 2:
            # Create smooth path
            path_d = f"M {points[0][0]},{points[0][1]}"
            for x, y in points[1:]:
                path_d += f" L {x},{y}"
            
            river_svg += f'<path d="{path_d}" fill="none" stroke="{color}" stroke-width="{width}" stroke-linecap="round" stroke-linejoin="round" opacity="0.8"/>\n'
    
    return river_svg





@patch
def show_watersheds(self: DetailGeology, radius: float = 8, top_n: int = 5):
    """Display detail terrain with watershed overlay and rivers."""
    self.terrain.hexGrid.adjustRadius(radius)
    self.terrain.colorMap()
    self.terrain.hexGrid.update()
    
    grid = self.terrain.hexGrid
    valid_indices = set(range(len(self.terrain.elevations))) - grid.invalidRegion
    
    # Get major watersheds and their IDs
    major = self.master.basins.get_major(top_n)
    ws_colors = StyleCSS.seaborn("tab10", top_n)
    
    # Map major watersheds to their full IDs
    major_id_to_color = {}
    for i, ws in enumerate(major):
        for full_id, full_ws in enumerate(self.master.basins.sheds):
            if full_ws is ws:
                major_id_to_color[full_id] = ws_colors[i].properties.get('fill', '#888')
                break
    
    # Overlay watershed colors on hexes
    ws_overlay = ""
    for grid_idx in valid_indices:
        ws_id = int(self.terrain.fields.get('watershed_id', np.array([-1]))[grid_idx])
        if ws_id in major_id_to_color:
            hex_obj = grid.hexes[grid_idx]
            color = major_id_to_color[ws_id]
            ws_overlay += f'<polygon points="{" ".join(f"{v.x},{v.y}" for v in hex_obj.v)}" fill="{color}" opacity="0.35"/>\n'
    
    grid.builder.adjust("watersheds", ws_overlay)
    
    # Draw rivers
    river_svg = self.draw_watersheds(top_n=top_n)
    grid.builder.adjust("rivers", river_svg)
    
    # Title
    textStyle = StyleCSS("textStyle", fill="white", font_size="1.2em", 
                         filter="drop-shadow(2px 2px 1px rgba(0,0,0,.7))")
    grid.builder.add_style(textStyle)
    grid.builder.add_centered_text(
        f"Detail: {len(valid_indices)} hexes, top {top_n} watersheds", 
        y_offset=-grid.builder.height/2 + 30, 
        class_name=textStyle.name
    )
    
    return grid.builder.show()


@patch
def draw_watersheds(self: DetailGeology, top_n: int = 5, max_width: float = 6) -> str:
    """Draw projected river paths from master watersheds onto detail terrain."""
    major = self.master.basins.get_major(top_n)
    ws_colors = StyleCSS.seaborn("tab10", top_n)
    
    grid = self.terrain.hexGrid
    valid_indices = set(range(len(self.terrain.elevations))) - grid.invalidRegion
    if not valid_indices:
        return ""
    
    # Get bounds from valid hexes directly instead of chunk_refs
    min_q = min_r = float('inf')
    max_q = max_r = float('-inf')
    
    for idx in valid_indices:
        pos = grid.index_to_hexposition(idx, origin_index=0)
        min_q = min(min_q, pos.q)
        max_q = max(max_q, pos.q)
        min_r = min(min_r, pos.r)
        max_r = max(max_r, pos.r)
    
    # Estimate world origin from terrain grid origin
    # The terrain was built from chunks centered around some world position
    if self.chunk_refs:
        # Use first chunk center as reference
        ref_center = self.chunk_refs[0].center_hex
        # Estimate that grid index 0 is at min bounds
        origin_world = HexPosition(int(ref_center.q - (max_q - min_q)//2),
                                   int(ref_center.r - (max_r - min_r)//2),
                                   0)
        origin_world = HexPosition(origin_world.q, origin_world.r, -origin_world.q - origin_world.r)
    else:
        origin_world = HexPosition(0, 0, 0)
    
    river_svg = ""
    
    for i, ws in enumerate(major):
        color = ws_colors[i].properties.get('fill', '#1565c0')
        
        style = StyleCSS(f"river_{i}", fill="none", stroke=color, 
                        stroke_width=max_width * (1 - i * 0.1), 
                        stroke_linecap="round", stroke_linejoin="round")
        grid.builder.add_style(style)
        
        river_hexes = ws.river.hexes if hasattr(ws.river, 'hexes') else []
        if not river_hexes:
            continue
        
        points = []
        for hex_idx in river_hexes:
            coarse_pos = self.master.terrain.hexGrid.index_to_hexposition(
                hex_idx, origin_index=self.master.terrain.hexGrid.midpoint)
            
            fine_q = int(coarse_pos.q * self.scale)
            fine_r = int(coarse_pos.r * self.scale)
            fine_s = -fine_q - fine_r
            fine_world = HexPosition(fine_q, fine_r, fine_s)
            
            local_pos = fine_world - origin_world
            grid_idx = grid.hexposition_to_index(local_pos, origin_index=0)
            
            if 0 <= grid_idx < len(grid.hexes) and grid_idx in valid_indices:
                hex_obj = grid.hexes[grid_idx]
                points.append((hex_obj.center.x, hex_obj.center.y))
        
        if len(points) >= 2:
            path_str = f'<polyline points="{" ".join(f"{x},{y}" for x, y in points)}" class="{style.name}"/>\n'
            river_svg += path_str
    
    return river_svg


### Put it together

In [ ]:
SVGBuilder.BUILDERHIDE = True

In [ ]:
# === VOLCANO ISLAND WITH LARGEST WATERSHED DETAIL ===


with GeoStorageDebugger(keep_on_error=False) as debugger:
    # 1. Create generator with volcanic terrain - larger rings
    gen = ChunkedTerrainGenerator(
        storage=debugger.server,
        coarse_rings=15,  # Larger for more land
        coarse_radius=50,
        fine_radius=10,
        chunk_rings=8,
        halo_rings=3
    )
    
    # Generate volcanic island (less ocean)
    coarse = gen.generate_coarse(
        num_plates=8, 
        ocean_fraction=0.35,  # Less ocean
        oceanic_sides=['W'],  # Only west is ocean (peninsula-like)
        formation_type='volcanic',
        seed=42
    )
    
    # 2. Create MasterGeology
    climate = TerrainPatterns(coarse).weatherPatterns()['tropical']
    master = MasterGeology.create(gen, num_lakes=2, climate=climate)
    
    print(f"=== MASTER GEOLOGY ===")
    print(f"Terrain: {len(master.terrain.elevations)} hexes")
    print(f"Watersheds: {len(master.basins.sheds)}")
    
    # Guard against empty watersheds
    if not master.basins.sheds:
        print("⚠ No watersheds found - showing coarse map only")
        master.show()
    else:
        # 3. Find largest watershed
        major_sheds = master.basins.get_major(top_n=10)
        
        if not major_sheds:
            print("⚠ No major watersheds - using all sheds")
            major_sheds = master.basins.sheds
        
        largest = max(major_sheds, key=lambda ws: len(ws.region.hexes))
        largest_id = next((i for i, ws in enumerate(master.basins.sheds) if ws is largest), 0)
        
        print(f"Largest watershed: {len(largest.region.hexes)} hexes (id={largest_id})")
        
        # 4. Get bounds of largest watershed for detail region
        ws_positions = [master.terrain.hexGrid.index_to_hexposition(
            idx, origin_index=master.terrain.hexGrid.midpoint) 
            for idx in largest.region.hexes]
        
        ws_min_q = min(p.q for p in ws_positions)
        ws_max_q = max(p.q for p in ws_positions)
        ws_min_r = min(p.r for p in ws_positions)
        ws_max_r = max(p.r for p in ws_positions)
        
        # Scale to fine coordinates with padding
        padding = 3
        fine_min_q = int(ws_min_q * gen.scale) - padding
        fine_max_q = int(ws_max_q * gen.scale) + padding
        fine_min_r = int(ws_min_r * gen.scale) - padding
        fine_max_r = int(ws_max_r * gen.scale) + padding
        
        print(f"Detail region: q[{fine_min_q}, {fine_max_q}], r[{fine_min_r}, {fine_max_r}]")
        
        # 5. Generate detail for watershed region
        detail = master.detail_for_rect(fine_min_q, fine_max_q, fine_min_r, fine_max_r)
        
        print(f"Detail hexes: {len(detail.terrain.elevations)}")
        print(f"Detail watersheds: {len(detail.basins.sheds)}")
        
        # === COARSE VIEW ===
        print("\n=== COARSE MAP ===")
        display(master.show_watersheds(radius=15))
        
        # === DETAIL VIEW ===
        print("\n=== DETAIL VIEW ===")
        display(detail.show_watersheds(radius=10, top_n=5))
        
    print(f"\n✓ Complete")


When this renders on detail it isn't creating a river svg. sonmething is off


##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

The issue is that `self.basins.draw_watersheds()` is returning empty because the detail basins created via `from_regions` don't have proper river trees built up.

The problem: `_build_basins_from_projection` creates watersheds with carved paths, but the BFS river-building isn't finding connected paths in the detail terrain.

**Quick fix**: Draw rivers by projecting from master instead of building new ones:

```python
@patch
def show_watersheds(self: DetailGeology, radius: float = 8, top_n: int = 5, max_width: float = 6):
    """Display detail terrain with watersheds projected from master."""
    self.terrain.hexGrid.adjustRadius(radius)
    self.terrain.colorMap()
    self.terrain.hexGrid.update()
    
    grid = self.terrain.hexGrid
    invalid = getattr(grid, 'invalidRegion', set())
    valid_indices = set(range(len(self.terrain.elevations))) - invalid
    
    # Get major watersheds from MASTER
    major = self.master.basins.get_major(top_n)
    ws_colors = StyleCSS.seaborn("tab10", len(major))
    
    # Map master watershed index -> color
    major_ids = {}
    for i, ws in enumerate(major):
        for full_id, full_ws in enumerate(self.master.basins.sheds):
            if full_ws is ws:
                major_ids[full_id] = ws_colors[i].properties.get('fill', '#888')
                break
    
    # Watershed overlay using projected IDs
    ws_overlay = ""
    for grid_idx in valid_indices:
        ws_id = int(self.terrain.fields.get('watershed_id', np.array([-1]))[grid_idx])
        if ws_id in major_ids:
            hex_obj = grid.hexes[grid_idx]
            ws_overlay += f'<polygon points="{" ".join(f"{v.x},{v.y}" for v in hex_obj.v)}" fill="{major_ids[ws_id]}" opacity="0.35"/>\n'
    
    grid.builder.adjust("watersheds", ws_overlay)
    
    # Project rivers from master onto detail grid
    river_svg = self._project_rivers_from_master(major, ws_colors, max_width)
    grid.builder.adjust("rivers", river_svg)
    
    # Title
    textStyle = StyleCSS("textStyle", fill="white", font_size="1.2em", 
                         filter="drop-shadow(2px 2px 1px rgba(0,0,0,.7))")
    grid.builder.add_style(textStyle)
    grid.builder.add_centered_text(
        f"Detail: {len(valid_indices)} hexes, top {top_n} watersheds", 
        y_offset=-grid.builder.height/2 + 30, 
        class_name=textStyle.name
    )
    
    return grid.builder.show()


@patch
def _project_rivers_from_master(self: DetailGeology, major_sheds: list, 
                                 ws_colors: list, max_width: float = 6) -> str:
    """Project master river paths onto detail terrain."""
    grid = self.terrain.hexGrid
    invalid = getattr(grid, 'invalidRegion', set())
    valid_indices = set(range(len(self.terrain.elevations))) - invalid
    
    if not valid_indices:
        return ""
    
    # Compute origin from chunk_set positions
    if self._chunk_set:
        positions = list(self._chunk_set.pos_to_chunk.keys())
        min_q = min(p.q for p in positions)
        min_r = min(p.r for p in positions)
        origin_world = HexPosition(min_q, min_r, -min_q - min_r)
    else:
        origin_world = HexPosition(0, 0, 0)
    
    river_svg = ""
    
    for i, ws in enumerate(major_sheds):
        color = ws_colors[i].properties.get('fill', '#1565c0')
        width = max(1, max_width * (1 - i * 0.1))
        
        # Get river hexes from master
        river_hexes = list(ws.tributary.hexes) if ws.tributary else []
        if not river_hexes:
            continue
        
        # Sort by elevation (high to low for drawing order)
        river_hexes.sort(key=lambda h: self.master.terrain.elevations[h], reverse=True)
        
        # Project each coarse river hex to fine coordinates
        points = []
        for coarse_idx in river_hexes:
            coarse_pos = self.master.terrain.hexGrid.index_to_hexposition(
                coarse_idx, origin_index=self.master.terrain.hexGrid.midpoint)
            
            # Scale to fine world coordinates
            fine_q = int(coarse_pos.q * self.scale)
            fine_r = int(coarse_pos.r * self.scale)
            fine_world = HexPosition(fine_q, fine_r, -fine_q - fine_r)
            
            # Convert to local grid coordinates
            local_pos = fine_world - origin_world
            grid_idx = grid.hexposition_to_index(local_pos, origin_index=0)
            
            if 0 <= grid_idx < len(grid.hexes) and grid_idx in valid_indices:
                hex_obj = grid.hexes[grid_idx]
                points.append((hex_obj.center.x, hex_obj.center.y))
        
        if len(points) >= 2:
            # Create smooth path
            path_d = f"M {points[0][0]},{points[0][1]}"
            for x, y in points[1:]:
                path_d += f" L {x},{y}"
            
            river_svg += f'<path d="{path_d}" fill="none" stroke="{color}" stroke-width="{width}" stroke-linecap="round" stroke-linejoin="round" opacity="0.8"/>\n'
    
    return river_svg
```

The key insight: **draw rivers from master, not from newly-built detail basins**. The watershed regions are projected correctly, and now the rivers are too.

Ahh this shows part of the problem. the river construction hexes are in the wrong order.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

**Note:** Some earlier messages were removed from context due to size limits.

---

Right! `ws.tributary.hexes` is a **set**, not an ordered path. Sorting by elevation doesn't give you the connected path order.

We need to traverse the river tree structure:

```python
@patch
def _project_rivers_from_master(self: DetailGeology, major_sheds: list, 
                                 ws_colors: list, max_width: float = 6) -> str:
    """Project master river paths onto detail terrain."""
    grid = self.terrain.hexGrid
    invalid = getattr(grid, 'invalidRegion', set())
    valid_indices = set(range(len(self.terrain.elevations))) - invalid
    
    if not valid_indices:
        return ""
    
    # Compute origin from chunk_set positions
    if self._chunk_set:
        positions = list(self._chunk_set.pos_to_chunk.keys())
        min_q = min(p.q for p in positions)
        min_r = min(p.r for p in positions)
        origin_world = HexPosition(min_q, min_r, -min_q - min_r)
    else:
        origin_world = HexPosition(0, 0, 0)
    
    river_svg = ""
    
    for i, ws in enumerate(major_sheds):
        color = ws_colors[i].properties.get('fill', '#1565c0')
        width = max(1, max_width * (1 - i * 0.1))
        
        # Get ordered segments from river tree
        for segment in ws.segments():
            points = []
            for coarse_idx in segment:
                if self.master.terrain.elevations[coarse_idx] <= 0:
                    continue  # Skip ocean hexes
                    
                coarse_pos = self.master.terrain.hexGrid.index_to_hexposition(
                    coarse_idx, origin_index=self.master.terrain.hexGrid.midpoint)
                
                # Scale to fine world coordinates
                fine_q = int(coarse_pos.q * self.scale)
                fine_r = int(coarse_pos.r * self.scale)
                fine_world = HexPosition(fine_q, fine_r, -fine_q - fine_r)
                
                # Convert to local grid coordinates
                local_pos = fine_world - origin_world
                grid_idx = grid.hexposition_to_index(local_pos, origin_index=0)
                
                if 0 <= grid_idx < len(grid.hexes) and grid_idx in valid_indices:
                    hex_obj = grid.hexes[grid_idx]
                    points.append((hex_obj.center.x, hex_obj.center.y))
            
            if len(points) >= 2:
                path_d = f"M {points[0][0]},{points[0][1]}"
                for x, y in points[1:]:
                    path_d += f" L {x},{y}"
                
                river_svg += f'<path d="{path_d}" fill="none" stroke="{color}" stroke-width="{width}" stroke-linecap="round" stroke-linejoin="round" opacity="0.8"/>\n'
    
    return river_svg
```

Key change: Use `ws.segments()` which returns the hexes in **path order** from the river tree, rather than iterating over the unordered `hexes` set.

In [ ]:
# === MASTER + DETAIL WITH REGION HIGHLIGHT ===
SVGBuilder.BUILDERHIDE = False

with GeoStorageDebugger(keep_on_error=False) as debugger:
    gen = ChunkedTerrainGenerator(
        storage=debugger.server,
        coarse_rings=15,
        coarse_radius=50,
        fine_radius=10,
        chunk_rings=8,
        halo_rings=3
    )
    
    coarse = gen.generate_coarse(
        num_plates=8, 
        ocean_fraction=0.35, 
        oceanic_sides=['W'], 
        seed=42
    )
    
    climate = TerrainPatterns(coarse).weatherPatterns()['mediterranean']
    master = MasterGeology.create(gen, num_lakes=3, climate=climate)
    
    print(f"=== MASTER: {len(master.basins.sheds)} watersheds ===")
    
    # Define detail region in COARSE coordinates
    detail_center = HexPosition(3, -2, -1)  # Pick interesting area
    detail_radius = 6  # In coarse hexes
    
    # Scale to fine coordinates for detail generation
    fine_center = HexPosition(
        int(detail_center.q * gen.scale),
        int(detail_center.r * gen.scale),
        int(detail_center.s * gen.scale)
    )
    fine_radius = int(detail_radius * gen.scale)
    
    # === COARSE MAP WITH REGION HIGHLIGHT ===
    master.terrain.hexGrid.adjustRadius(12)
    master.terrain.colorMap()
    master.terrain.hexGrid.update()
    
    # Add watershed colors
    top_n = 5
    major_sheds = master.basins.get_major(top_n)
    ws_colors = StyleCSS.seaborn("tab10", top_n)
    
    ws_overlay = ""
    for i, ws in enumerate(major_sheds):
        color = ws_colors[i].properties.get('fill', '#888')
        for hex_idx in ws.region.hexes:
            hex_obj = master.terrain.hexGrid.hexes[hex_idx]
            ws_overlay += f'<polygon points="{" ".join(f"{v.x},{v.y}" for v in hex_obj.v)}" fill="{color}" opacity="0.3"/>\n'
    
    master.terrain.hexGrid.builder.adjust("watersheds", ws_overlay)
    
    # Add rivers
    river_svg = master.basins.draw_watersheds(top_n=top_n, max_width=6)
    master.terrain.hexGrid.builder.adjust("rivers", river_svg)
    
    # === DRAW DETAIL REGION RECTANGLE ===
    # Find hexes in the detail region (coarse coords)
    region_hexes = []
    for ring in range(detail_radius + 1):
        if ring == 0:
            positions = [detail_center]
        else:
            positions = list(detail_center.ring(ring))
        
        for pos in positions:
            idx = master.terrain.hexGrid.hexposition_to_index(
                pos, origin_index=master.terrain.hexGrid.midpoint)
            if 0 <= idx < len(master.terrain.hexGrid.hexes):
                region_hexes.append(idx)
    
    # Draw region boundary
    if region_hexes:
        region = HexRegion(hexes=set(region_hexes), hexGrid=master.terrain.hexGrid)
        
        region_style = StyleCSS("detail_region", fill="none", 
                                stroke="#ff6600", stroke_width=4,
                                stroke_dasharray="10,5")
        master.terrain.hexGrid.builder.add_style(region_style)
        
        paths = region.trace_perimeter(style=region_style)
        region_overlay = ""
        for path in paths:
            curved = path.make_windy(iterations=1, offset_factor=0.05)
            region_overlay += curved.svg()
        
        # Add "DETAIL" label at region center
        center_idx = region.centroid_hex()
        if center_idx and 0 <= center_idx < len(master.terrain.hexGrid.hexes):
            cx = master.terrain.hexGrid.hexes[center_idx].center.x
            cy = master.terrain.hexGrid.hexes[center_idx].center.y
            region_overlay += f'<text x="{cx}" y="{cy}" text-anchor="middle" fill="#ff6600" font-size="16" font-weight="bold" style="filter: drop-shadow(1px 1px 1px rgba(0,0,0,.8))">DETAIL</text>\n'
        
        master.terrain.hexGrid.builder.adjust("detail_region", region_overlay)
    
    # Title
    textStyle = StyleCSS("textStyle", fill="white", font_size="1.2em", 
                         filter="drop-shadow(2px 2px 1px rgba(0,0,0,.7))")
    master.terrain.hexGrid.builder.add_style(textStyle)
    master.terrain.hexGrid.builder.add_centered_text(
        f"Master Geology - Orange box = detail region", 
        y_offset=-master.terrain.hexGrid.builder.height/2 + 30, 
        class_name=textStyle.name
    )
    
    print("\n=== COARSE MAP (orange = detail region) ===")
    display(master.terrain.hexGrid.builder.show())
    
    # === GENERATE AND SHOW DETAIL ===
    print(f"\n=== DETAIL VIEW (center {fine_center}, radius {fine_radius}) ===")
    detail = master.detail_for_center(fine_center, radius=fine_radius)
    
    detail.terrain.hexGrid.adjustRadius(20)
    detail.terrain.colorMap()
    detail.terrain.hexGrid.update()
    
    # Project watershed colors
    major_id_to_color = {}
    for i, ws in enumerate(major_sheds):
        for full_id, full_ws in enumerate(master.basins.sheds):
            if full_ws is ws:
                major_id_to_color[full_id] = ws_colors[i].properties.get('fill', '#888')
                break
    
    valid_indices = set(range(len(detail.terrain.elevations))) - detail.terrain.hexGrid.invalidRegion
    
    detail_ws_overlay = ""
    for grid_idx in valid_indices:
        ws_id = int(detail.terrain.fields.get('watershed_id', np.array([-1]))[grid_idx])
        if ws_id in major_id_to_color:
            hex_obj = detail.terrain.hexGrid.hexes[grid_idx]
            color = major_id_to_color[ws_id]
            detail_ws_overlay += f'<polygon points="{" ".join(f"{v.x},{v.y}" for v in hex_obj.v)}" fill="{color}" opacity="0.35"/>\n'
    
    detail.terrain.hexGrid.builder.adjust("watersheds", detail_ws_overlay)
    
    # Draw rivers from detail basins
    river_svg = detail.basins.draw_watersheds(top_n=top_n, max_width=4, lake_base_size=2)
    detail.terrain.hexGrid.builder.adjust("rivers", river_svg)
    
    # Add flow arrows for context
    flow_overlay = detail.terrain.flow_diagram_filtered(valid_indices)
    detail.terrain.hexGrid.builder.adjust("flow", flow_overlay)
    
    detail.terrain.hexGrid.builder.add_style(textStyle)
    detail.terrain.hexGrid.builder.add_centered_text(
        f"Detail: {len(valid_indices)} hexes, {len(detail.chunk_refs)} chunks, {len(detail.basins.sheds)} watersheds", 
        y_offset=-detail.terrain.hexGrid.builder.height/2 + 30, 
        class_name=textStyle.name
    )
    
    display(detail.terrain.hexGrid.builder.show())
    
    print(f"\n✓ Detail covers {len(valid_indices)} hexes from {len(detail.chunk_refs)} chunks")
    print(f"✓ {len(detail.basins.sheds)} watershed regions projected from master")


So this is where it gets really tricky. we don't want the drainage basin to be computed from the terrain since that is o n^2. we want to pass it a list of watersheds that we have previously computed.
```

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Ah yes! We need a factory method that builds `DrainageBasins` from pre-assigned watershed IDs, not by computing them:

```python
@dataclass  
class DetailGeology:
    """Fine-level geology projected from master."""
    terrain: Terrain
    master: MasterGeology
    scale: float
    chunk_refs: list[ChunkRef]
    _basins: DrainageBasins = None  # Pre-built, not computed
    
    @classmethod
    def _from_chunk_set(cls, chunk_set: ChunkSet, master: MasterGeology,
                        scale: float, chunk_refs: list) -> 'DetailGeology':
        """Internal: build from ChunkSet."""
        terrain, _ = chunk_set.to_weather_terrain(master.terrain, scale, fill_gaps=False)
        
        detail = cls(terrain=terrain, master=master, 
                     scale=scale, chunk_refs=chunk_refs)
        detail._project_watersheds(chunk_set)
        
        # Build basins from projected IDs, not computed
        detail._basins = detail._build_basins_from_projection()
        
        return detail
    
    def _build_basins_from_projection(self) -> DrainageBasins:
        """Build DrainageBasins from projected watershed_id, not by computation."""
        
        # Group hexes by watershed_id
        ws_hexes = {}  # ws_id -> list of hex indices
        for idx in range(len(self.terrain.elevations)):
            ws_id = int(self.terrain.fields.get('watershed_id', np.array([-1]))[idx])
            if ws_id >= 0:
                if ws_id not in ws_hexes:
                    ws_hexes[ws_id] = []
                ws_hexes[ws_id].append(idx)
        
        # Build Watershed objects from the grouped hexes
        # Use master's watershed metadata (tributaries, etc) scaled down
        sheds = []
        for ws_id, hex_indices in ws_hexes.items():
            if ws_id < len(self.master.basins.sheds):
                master_ws = self.master.basins.sheds[ws_id]
                
                # Create region from projected hexes
                region = HexRegion(hexes=set(hex_indices), hexGrid=self.terrain.hexGrid)
                
                # Create watershed with projected region but master's rivers/tributaries scaled
                ws = Watershed.from_projection(
                    terrain=self.terrain,
                    region=region,
                    master_watershed=master_ws,
                    scale=self.scale
                )
                sheds.append(ws)
        
        # Build DrainageBasins without computing
        return DrainageBasins.from_watersheds(self.terrain, sheds)
    
    @property
    def basins(self) -> DrainageBasins:
        return self._basins
    
    # ... rest of methods
```

This needs two new factory methods:
1. `Watershed.from_projection()` - builds a Watershed from pre-assigned hexes + master metadata
2. `DrainageBasins.from_watersheds()` - builds DrainageBasins from a list of Watersheds without computing

Do these exist in your codebase, or should I sketch them out?

In [ ]:
??DrainageBasins

So we now have from_regions api on DrainageBasins

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

`from_regions` is exactly what we need. So `_build_basins_from_projection` becomes:

```python
def _build_basins_from_projection(self) -> DrainageBasins:
    """Build DrainageBasins from projected watershed_id using from_regions."""
    
    # Group hexes by watershed_id
    ws_hexes = {}  # ws_id -> set of hex indices
    for idx in range(len(self.terrain.elevations)):
        if idx in self.terrain.hexGrid.invalidRegion:
            continue
        ws_id = int(self.terrain.fields.get('watershed_id', np.array([-1]))[idx])
        if ws_id >= 0:
            if ws_id not in ws_hexes:
                ws_hexes[ws_id] = set()
            ws_hexes[ws_id].add(idx)
    
    # Build regions list: (hex_set, mouth_idx)
    # Mouth is the lowest elevation hex in each watershed
    regions = []
    for ws_id, hex_set in ws_hexes.items():
        if not hex_set:
            continue
        # Find mouth (lowest elevation in the set)
        mouth_idx = min(hex_set, key=lambda i: self.terrain.elevations[i])
        regions.append((hex_set, mouth_idx))
    
    # Use from_regions - handles carving and river building
    return DrainageBasins.from_regions(self.terrain, regions)
```

This is O(n) for grouping + whatever `from_regions` does for carving (which is local to each watershed, not global n²).

Want me to write the full updated `DetailGeology` class?

please write the full updated DetailGeology. Remember that the collection in DrainageBasins is sheds. we need to carve to make the drainage work on the detail

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
@dataclass  
class DetailGeology:
    """Fine-level geology projected from master."""
    terrain: Terrain
    master: MasterGeology
    scale: float
    chunk_refs: list[ChunkRef]
    _basins: DrainageBasins = None
    _chunk_set: ChunkSet = None  # Keep reference for projection
    
    @classmethod
    def _from_chunk_set(cls, chunk_set: ChunkSet, master: MasterGeology,
                        scale: float, chunk_refs: list) -> 'DetailGeology':
        """Internal: build from ChunkSet."""
        terrain, _ = chunk_set.to_weather_terrain(master.terrain, scale, fill_gaps=False)
        
        detail = cls(terrain=terrain, master=master, 
                     scale=scale, chunk_refs=chunk_refs,
                     _chunk_set=chunk_set)
        
        # Project watershed IDs from master (O(n) lookup)
        detail._project_watersheds()
        
        # Build basins from projected IDs using from_regions (handles carving)
        detail._basins = detail._build_basins_from_projection()
        
        return detail
    
    def _project_watersheds(self):
        """Assign watershed_id by lookup from master."""
        lookup = self.master.watershed_lookup()
        
        positions = list(self._chunk_set.pos_to_chunk.keys())
        min_q = min(p.q for p in positions)
        min_r = min(p.r for p in positions)
        origin_world = HexPosition(min_q, min_r, -min_q - min_r)
        
        n = len(self.terrain.elevations)
        self.terrain.fields['watershed_id'] = np.full(n, -1, dtype=np.int32)
        
        for world_pos, (chunk, idx, is_core) in self._chunk_set.pos_to_chunk.items():
            if not is_core or chunk.elevations[idx] <= 0:
                continue
            
            # Map fine position to coarse
            coarse_q = int(round(world_pos.q / self.scale))
            coarse_r = int(round(world_pos.r / self.scale))
            coarse_s = -coarse_q - coarse_r
            
            ws_id = lookup.get((coarse_q, coarse_r, coarse_s), -1)
            
            local_pos = world_pos - origin_world
            grid_idx = self.terrain.hexGrid.hexposition_to_index(local_pos, origin_index=0)
            if 0 <= grid_idx < n:
                self.terrain.fields['watershed_id'][grid_idx] = ws_id
    
    def _build_basins_from_projection(self) -> DrainageBasins:
        """Build DrainageBasins from projected watershed_id using from_regions."""
        
        # Group hexes by watershed_id
        ws_hexes = {}  # ws_id -> set of hex indices
        invalid = getattr(self.terrain.hexGrid, 'invalidRegion', set())
        
        for idx in range(len(self.terrain.elevations)):
            if idx in invalid:
                continue
            ws_id = int(self.terrain.fields.get('watershed_id', np.array([-1]))[idx])
            if ws_id >= 0:
                if ws_id not in ws_hexes:
                    ws_hexes[ws_id] = set()
                ws_hexes[ws_id].add(idx)
        
        # Build regions list: (hex_set, mouth_idx)
        # Mouth is the lowest elevation hex in each watershed
        regions = []
        for ws_id in sorted(ws_hexes.keys()):  # Sort for consistent ordering
            hex_set = ws_hexes[ws_id]
            if not hex_set:
                continue
            
            # Find mouth (lowest elevation in the set)
            mouth_idx = min(hex_set, key=lambda i: self.terrain.elevations[i])
            regions.append((hex_set, mouth_idx))
        
        # Use from_regions - handles carving and river building
        return DrainageBasins.from_regions(self.terrain, regions)
    
    @property
    def basins(self) -> DrainageBasins:
        """Get drainage basins (pre-built from projection)."""
        return self._basins
    
    @property
    def soils(self):
        """Compute soils lazily (cheap operation)."""
        if not hasattr(self, '_soils'):
            self._soils = SoilSystem.from_terrain(self.terrain)
        return self._soils
    
    # === DISPLAY ===
    
    def show(self, radius: float = 8):
        """Display detail terrain."""
        self.terrain.hexGrid.adjustRadius(radius)
        self.terrain.colorMap()
        self.terrain.hexGrid.update()
        return self.terrain.hexGrid.builder.show()
    
    def show_watersheds(self, radius: float = 8, top_n: int = 5, max_width: float = 6):
        """Display detail terrain with watersheds and rivers."""
        self.terrain.hexGrid.adjustRadius(radius)
        self.terrain.colorMap()
        self.terrain.hexGrid.update()
        
        # Draw watershed regions + rivers using DrainageBasins
        river_svg = self.basins.draw_watersheds(top_n=top_n, max_width=max_width)
        self.terrain.hexGrid.builder.adjust("rivers", river_svg)
        
        # Add watershed color overlay
        ws_colors = StyleCSS.seaborn("tab10", min(top_n, len(self.basins.sheds)))
        major_sheds = self.basins.get_major(top_n) if hasattr(self.basins, 'get_major') else self.basins.sheds[:top_n]
        
        ws_overlay = ""
        for i, ws in enumerate(major_sheds):
            color = ws_colors[i % len(ws_colors)].properties.get('fill', '#888')
            for hex_idx in ws.region.hexes:
                if hex_idx < len(self.terrain.hexGrid.hexes):
                    hex_obj = self.terrain.hexGrid.hexes[hex_idx]
                    ws_overlay += f'<polygon points="{" ".join(f"{v.x},{v.y}" for v in hex_obj.v)}" fill="{color}" opacity="0.3"/>\n'
        
        self.terrain.hexGrid.builder.adjust("watersheds", ws_overlay)
        
        # Title
        textStyle = StyleCSS("textStyle", fill="white", font_size="1.2em", 
                             filter="drop-shadow(2px 2px 1px rgba(0,0,0,.7))")
        self.terrain.hexGrid.builder.add_style(textStyle)
        
        valid_count = len(self.terrain.elevations) - len(getattr(self.terrain.hexGrid, 'invalidRegion', set()))
        self.terrain.hexGrid.builder.add_centered_text(
            f"Detail: {valid_count} hexes, {len(self.basins.sheds)} watersheds", 
            y_offset=-self.terrain.hexGrid.builder.height/2 + 30, 
            class_name=textStyle.name
        )
        
        return self.terrain.hexGrid.builder.show()
    
    def show_temperature(self, radius: float = 8):
        """Display temperature map."""
        self.terrain.hexGrid.adjustRadius(radius)
        temp_overlay = self.terrain.render_icon_temperature()
        self.terrain.hexGrid.builder.adjust("hexes", temp_overlay)
        return self.terrain.hexGrid.builder.show()
    
    def show_precipitation(self, radius: float = 8):
        """Display precipitation map."""
        self.terrain.hexGrid.adjustRadius(radius)
        self.terrain.colorMap()
        self.terrain.hexGrid.update()
        self.terrain.add_rain_overlay(layer_name="precip_overlay")
        return self.terrain.hexGrid.builder.show()
```

Usage:

```python
# Create master once
master = MasterGeology.create(gen, num_lakes=3, climate=climate)

# Get detail - watersheds projected + carved automatically
detail = master.detail_for_center(HexPosition(0, 0, 0), radius=30)

# Display with watersheds (uses basins.draw_watersheds internally)
detail.show_watersheds(top_n=5)
```

Does this give us any insight

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes! It's **working now**. Compare:

**Before (broken):**
```
Start local: HexPosition(13, 5, -18)
Start fine idx: -1 (valid: 0-440)
```

**After (fixed):**
```
Start local pos: HexPosition(3, 0, -3)
Start local dist from center: 3
Start fine idx: 147 (valid: 0-288)
```

The chunk assignment is now correct—start hex distance (3) ≤ chunk_rings (6). The local A* found a 4-hex path with cost ~91.

The fix to `world_to_chunk` worked. Now you can test the full multi-chunk path:

```python
# Test full HPA pathfind
path, cost = hpa_pathfind(gen, start, goal, uphill_penalty=4.0)
print(f"Full path: {len(path)} hexes, cost={cost:.1f}")
```

This should now chain through chunks (1,-5,4) → ... → (5,-5,0) correctly.

In [ ]:
#| export
def hpa_pathfind_v2(gen: ChunkedTerrainGenerator,
                    start_world: HexPosition,
                    goal_world: HexPosition,
                    uphill_penalty: float = 4.0) -> tuple[list[HexPosition], float]:
    """Hierarchical pathfinding: coarse plan, then merge chunks for fine A*."""
    
    coarse = gen._coarse_terrain
    coarse_grid = coarse.hexGrid
    scale = gen.scale
    
    # 1. Map to coarse
    start_coarse = HexPosition(
        int(round(start_world.q / scale)),
        int(round(start_world.r / scale)),
        -int(round(start_world.q / scale)) - int(round(start_world.r / scale))
    )
    goal_coarse = HexPosition(
        int(round(goal_world.q / scale)),
        int(round(goal_world.r / scale)),
        -int(round(goal_world.q / scale)) - int(round(goal_world.r / scale))
    )
    
    start_coarse_idx = coarse_grid.hexposition_to_index(start_coarse, origin_index=coarse_grid.midpoint)
    goal_coarse_idx = coarse_grid.hexposition_to_index(goal_coarse, origin_index=coarse_grid.midpoint)
    
    # 2. Coarse A* to get waypoints
    coarse_path, _ = astar_terrain(coarse, start_coarse_idx, goal_coarse_idx, uphill_penalty)
    
    if not coarse_path:
        return [], float('inf')
    
    # 3. Collect all chunks touched by coarse path
    chunk_refs = set()
    for idx in coarse_path:
        pos = coarse_grid.index_to_hexposition(idx, origin_index=coarse_grid.midpoint)
        world_pos = HexPosition(int(pos.q * scale), int(pos.r * scale), int(pos.s * scale))
        chunk_ref = gen.world_to_chunk_ref(world_pos)
        chunk_refs.add(chunk_ref.key)
    
    # Add neighbors for safety
    expanded_chunks = set(chunk_refs)
    for key in list(chunk_refs):
        chunk_ref = ChunkRef.from_key(key, gen.chunk_rings)
        for neighbor_pos in chunk_ref.position.ring(1):
            expanded_chunks.add(ChunkRef(neighbor_pos, gen.chunk_rings).key)
    
    print(f"Path spans {len(chunk_refs)} chunks, expanded to {len(expanded_chunks)}")
    
    # 4. Check if merged terrain would be too big
    total_hexes = len(expanded_chunks) * (gen.chunk_rings * 2 + 1) ** 2
    if total_hexes > 6000:
        print(f"⚠ Merged terrain would have ~{total_hexes} hexes, too large")
        return [], float('inf')
    
    # 5. Generate/load all chunks
    chunks = {}
    for key in expanded_chunks:
        chunk_ref = ChunkRef.from_key(key, gen.chunk_rings)
        terrain = gen.load_chunk(chunk_ref)
        if terrain is None:
            terrain = gen.generate_chunk(chunk_ref)
            gen.save_chunk(terrain, chunk_ref)
        chunks[key] = (chunk_ref, terrain)
    
    # 6. Merge into single terrain
    merged = gen.merge_chunks(chunks)
    merged_grid = merged.hexGrid
    world_center = merged.world_center
    
    # 7. Find indices for start/goal in merged grid (world coord aware)
    start_idx = None
    goal_idx = None
    
    for idx in range(len(merged_grid.hexes)):
        local_pos = merged_grid.index_to_hexposition(idx, origin_index=merged_grid.midpoint)
        world_pos = local_pos + world_center
        
        if world_pos == start_world:
            start_idx = idx
        if world_pos == goal_world:
            goal_idx = idx
    
    if start_idx is None or goal_idx is None:
        print(f"⚠ Start/goal not found in merged grid. start={start_idx}, goal={goal_idx}")
        print(f"  start_world={start_world}, goal_world={goal_world}, center={world_center}")
        return [], float('inf')
    
    # 8. A* on merged terrain
    path_indices, cost = astar_terrain(merged, start_idx, goal_idx, uphill_penalty)
    
    if not path_indices:
        return [], float('inf')
    
    # 9. Convert to world positions
    path = []
    for idx in path_indices:
        local_pos = merged_grid.index_to_hexposition(idx, origin_index=merged_grid.midpoint)
        path.append(local_pos + world_center)
    
    return path, cost


In [ ]:
@patch
def test_hpa_pathfinding(self: GeoStorageDebugger,
                         coarse_rings: int = 15,
                         chunk_rings: int = 8,
                         num_tests: int = 3) -> dict:
    """Test hierarchical pathfinding across chunks."""
    
    gen = ChunkedTerrainGenerator(
        storage=self.server,
        coarse_rings=coarse_rings,
        coarse_radius=50,
        fine_radius=10,
        chunk_rings=chunk_rings,
        halo_rings=2
    )
    
    coarse = gen.generate_coarse(num_plates=8, ocean_fraction=0.3, oceanic_sides=['W'])
    
    # Find some land hexes for start/goal
    land_indices = [i for i, e in enumerate(coarse.elevations) if e > 0]
    if len(land_indices) < 2:
        return {'error': 'Not enough land hexes', 'land_count': len(land_indices)}
    
    results = []
    scale = gen.scale
    
    for test_num in range(min(num_tests, len(land_indices) // 2)):
        # Pick two land hexes
        start_idx = land_indices[test_num * 2]
        goal_idx = land_indices[min(test_num * 2 + 1 + test_num * 3, len(land_indices) - 1)]
        
        start_coarse = coarse.hexGrid.index_to_hexposition(start_idx, origin_index=coarse.hexGrid.midpoint)
        goal_coarse = coarse.hexGrid.index_to_hexposition(goal_idx, origin_index=coarse.hexGrid.midpoint)
        
        # Scale to fine world coords
        start_world = HexPosition(int(start_coarse.q * scale), int(start_coarse.r * scale), int(start_coarse.s * scale))
        goal_world = HexPosition(int(goal_coarse.q * scale), int(goal_coarse.r * scale), int(goal_coarse.s * scale))
        
        coarse_dist = abs(start_coarse - goal_coarse)
        
        try:
            path, cost = hpa_pathfind_v2(gen, start_world, goal_world)
            
            results.append({
                'test': test_num,
                'start': start_world,
                'goal': goal_world,
                'coarse_dist': coarse_dist,
                'path_len': len(path),
                'cost': cost,
                'chunks_crossed': len(set(gen.world_to_chunk_ref(p).key for p in path)) if path else 0,
                'ok': len(path) > 0
            })
        except Exception as e:
            results.append({
                'test': test_num,
                'start': start_world,
                'goal': goal_world,
                'error': str(e),
                'ok': False
            })
    
    # Summary
    passed = sum(1 for r in results if r.get('ok'))
    print(f"HPA Pathfinding: {passed}/{len(results)} tests passed")
    print(f"{'Test':<6} {'Coarse Dist':>12} {'Path Len':>10} {'Cost':>10} {'Chunks':>8}")
    print("-" * 50)
    for r in results:
        if r.get('ok'):
            print(f"{r['test']:<6} {r['coarse_dist']:>12} {r['path_len']:>10} {r['cost']:>10.1f} {r['chunks_crossed']:>8}")
        else:
            print(f"{r['test']:<6} ERROR: {r.get('error', 'no path')}")
    
    return {'results': results, 'passed': passed, 'total': len(results), 'gen': gen}


In [ ]:
with GeoStorageDebugger(keep_on_error=True) as dbg:
    result = dbg.test_hpa1_pathfinding(num_tests=5)
    result = dbg.test_hpa_pathfinding(num_tests=5)


ocean crossing?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes, exactly. Look at the results:
- Path found (path_len > 0) but cost = `inf`

The coarse A* probably routes through ocean hexes (which have elevation ≤ 0), and then:

1. In `_add_detail`, ocean gets forced to ≤ -50
2. Stepping from ocean (-50) to land (+100) = +150m elevation
3. With `uphill_penalty=4.0`: 150 × 4 = 600+ cost **per hex**

But `inf` specifically suggests some segment returned no path (A* failed), then `_refine_segment` fell back to the direct connection returning `inf`:

```python
return [start, goal], abs(start - goal) * 10  # or inf from failed A*
```

**Fix options:**

1. **Block ocean in coarse A*** — add to `astar_terrain`:
```python
if terrain.elevations[neighbor_idx] < -10:  # Ocean
    continue  # Skip this neighbor
```

2. **High ocean cost instead of blocking:**
```python
def hex_move_cost(elev_from, elev_to, uphill_penalty=4.0, ocean_penalty=100.0):
    if elev_to < 0:  # Entering ocean
        return ocean_penalty
    # ... rest
```

3. **Ensure coarse path stays on land** — filter `land_indices` for start/goal selection is good, but coarse A* still routes through ocean between them.

Option 1 is simplest if you want land-only paths. Want me to show the modified `astar_terrain`?

does it make sense to downsample the merge grid

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good question. It's a tradeoff:

**When downsampling helps:**
- Merged terrain approaching the 6000 hex limit
- Very long paths spanning many chunks
- "Good enough" paths are acceptable

**When it doesn't:**
- You're already at coarse→fine hierarchy (adding a third level might overcomplicate)
- Narrow passages or detailed coastlines could get lost
- Your test results show merged terrains of 7-10 chunks (~3000-5000 hexes) — A* handles that fine

**Alternative approach for large paths:**

Instead of downsampling the merged grid, you could use a **corridor strategy**:

```python
# Only load chunks within N hexes of coarse path, not full expansion
corridor_width = 1  # Just the chunks the path touches + immediate neighbors
```

Or **fall back to segment-by-segment** when merged would be too big:

```python
if total_hexes > 6000:
    return hpa_pathfind_v1(...)  # Original segment approach
else:
    return hpa_pathfind_v2(...)  # Merged approach
```

Given your current results work well, I'd hold off on downsampling unless you hit performance issues with longer cross-map paths. The coarse A* is already doing that job.

What are the speed difference betwen the two ways of computing paths. can you test?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
import time

def benchmark_pathfinding(num_tests: int = 5):
    """Compare v1 (segment) vs v2 (merge) pathfinding speed."""
    
    with GeoStorageDebugger(keep_on_error=False) as dbg:
        gen = ChunkedTerrainGenerator(
            storage=dbg.server,
            coarse_rings=15,
            coarse_radius=50,
            fine_radius=10,
            chunk_rings=8,
            halo_rings=2
        )
        
        coarse = gen.generate_coarse(num_plates=8, ocean_fraction=0.3, oceanic_sides=['W'])
        
        # Find land hexes
        land_indices = [i for i, e in enumerate(coarse.elevations) if e > 0]
        scale = gen.scale
        
        results = []
        
        for test_num in range(min(num_tests, len(land_indices) // 2)):
            start_idx = land_indices[test_num * 2]
            goal_idx = land_indices[min(test_num * 2 + 5, len(land_indices) - 1)]
            
            start_coarse = coarse.hexGrid.index_to_hexposition(start_idx, origin_index=coarse.hexGrid.midpoint)
            goal_coarse = coarse.hexGrid.index_to_hexposition(goal_idx, origin_index=coarse.hexGrid.midpoint)
            
            start_world = HexPosition(int(start_coarse.q * scale), int(start_coarse.r * scale), int(start_coarse.s * scale))
            goal_world = HexPosition(int(goal_coarse.q * scale), int(goal_coarse.r * scale), int(goal_coarse.s * scale))
            
            coarse_dist = abs(start_coarse - goal_coarse)
            
            # Time v1 (segment)
            t0 = time.perf_counter()
            path1, cost1 = hpa_pathfind(gen, start_world, goal_world)
            t1 = time.perf_counter()
            time_v1 = (t1 - t0) * 1000
            
            # Time v2 (merge)
            t0 = time.perf_counter()
            path2, cost2 = hpa_pathfind_v2(gen, start_world, goal_world)
            t1 = time.perf_counter()
            time_v2 = (t1 - t0) * 1000
            
            results.append({
                'dist': coarse_dist,
                'v1_ms': time_v1,
                'v1_len': len(path1),
                'v1_cost': cost1,
                'v2_ms': time_v2,
                'v2_len': len(path2),
                'v2_cost': cost2,
            })
        
        print(f"{'Dist':>5} {'V1 (ms)':>10} {'V1 Len':>8} {'V2 (ms)':>10} {'V2 Len':>8} {'Speedup':>10}")
        print("-" * 60)
        for r in results:
            speedup = r['v1_ms'] / r['v2_ms'] if r['v2_ms'] > 0 else 0
            print(f"{r['dist']:>5} {r['v1_ms']:>10.2f} {r['v1_len']:>8} {r['v2_ms']:>10.2f} {r['v2_len']:>8} {speedup:>10.2f}x")
        
        return results

benchmark_pathfinding(num_tests=6)
```

This will show you:
- Time in milliseconds for each approach
- Path length (should be similar, v2 might find slightly better paths)
- Speedup ratio

My expectation: v2 (merge) should be faster for multi-chunk paths because it avoids repeated chunk generation and runs a single A* instead of multiple segment A*s. But v1 might win for very long paths where merging becomes expensive.

In [ ]:
import time

def benchmark_pathfinding(num_tests: int = 5):
    """Compare v1 (segment) vs v2 (merge) pathfinding speed."""
    
    with GeoStorageDebugger(keep_on_error=False) as dbg:
        gen = ChunkedTerrainGenerator(
            storage=dbg.server,
            coarse_rings=15,
            coarse_radius=50,
            fine_radius=10,
            chunk_rings=8,
            halo_rings=2
        )
        
        coarse = gen.generate_coarse(num_plates=8, ocean_fraction=0.3, oceanic_sides=['W'])
        
        # Find land hexes
        land_indices = [i for i, e in enumerate(coarse.elevations) if e > 0]
        scale = gen.scale
        
        results = []
        
        for test_num in range(min(num_tests, len(land_indices) // 2)):
            start_idx = land_indices[test_num * 2]
            goal_idx = land_indices[min(test_num * 2 + 5, len(land_indices) - 1)]
            
            start_coarse = coarse.hexGrid.index_to_hexposition(start_idx, origin_index=coarse.hexGrid.midpoint)
            goal_coarse = coarse.hexGrid.index_to_hexposition(goal_idx, origin_index=coarse.hexGrid.midpoint)
            
            start_world = HexPosition(int(start_coarse.q * scale), int(start_coarse.r * scale), int(start_coarse.s * scale))
            goal_world = HexPosition(int(goal_coarse.q * scale), int(goal_coarse.r * scale), int(goal_coarse.s * scale))
            
            coarse_dist = abs(start_coarse - goal_coarse)
            
            # Time v1 (segment)
            t0 = time.perf_counter()
            path1, cost1 = hpa_pathfind(gen, start_world, goal_world)
            t1 = time.perf_counter()
            time_v1 = (t1 - t0) * 1000
            
            # Time v2 (merge)
            t0 = time.perf_counter()
            path2, cost2 = hpa_pathfind_v2(gen, start_world, goal_world)
            t1 = time.perf_counter()
            time_v2 = (t1 - t0) * 1000
            
            results.append({
                'dist': coarse_dist,
                'v1_ms': time_v1,
                'v1_len': len(path1),
                'v1_cost': cost1,
                'v2_ms': time_v2,
                'v2_len': len(path2),
                'v2_cost': cost2,
            })
        
        print(f"{'Dist':>5} {'V1 (ms)':>10} {'V1 Len':>8} {'V2 (ms)':>10} {'V2 Len':>8} {'Speedup':>10}")
        print("-" * 60)
        for r in results:
            speedup = r['v1_ms'] / r['v2_ms'] if r['v2_ms'] > 0 else 0
            print(f"{r['dist']:>5} {r['v1_ms']:>10.2f} {r['v1_len']:>8} {r['v2_ms']:>10.2f} {r['v2_len']:>8} {speedup:>10.2f}x")
        
        return results

benchmark_pathfinding(num_tests=6)


Thoughts

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Key observations:

| Pattern | Winner | Why |
|---------|--------|-----|
| Short paths (dist ≤ 2) | **V1** (20-40ms vs 500ms) | Merge overhead not worth it |
| Long paths (dist ≥ 4) | **V2** (3-5x faster) | Single A* beats repeated segment A* |
| Path quality | **V2** | V1 returns `inf` cost often, V2 finds real paths |

**V1 is broken for multi-chunk paths** — those `inf` costs mean it's failing to stitch segments through ocean or difficult terrain. V2's merged approach handles this because A* can route around obstacles globally.

**Hybrid suggestion:**

```python
def hpa_pathfind_hybrid(gen, start, goal, uphill_penalty=4.0):
    coarse_dist = estimate_coarse_distance(gen, start, goal)
    
    if coarse_dist <= 2:
        return hpa_pathfind(gen, start, goal, uphill_penalty)  # V1
    else:
        return hpa_pathfind_v2(gen, start, goal, uphill_penalty)  # V2
```

But honestly, V2's 500ms for short paths is probably fine for most uses. The real win is that V2 actually **works** for longer paths where V1 fails.

In [ ]:
# Debug the merge process
with GeoStorageDebugger(keep_on_error=True) as dbg:
    gen = ChunkedTerrainGenerator(
        storage=dbg.server,
        coarse_rings=15,
        coarse_radius=50,
        fine_radius=10,
        chunk_rings=8,
        halo_rings=2
    )
    
    coarse = gen.generate_coarse(num_plates=8, ocean_fraction=0.3, oceanic_sides=['W'])
    
    # Pick a simple path
    land_indices = [i for i, e in enumerate(coarse.elevations) if e > 0]
    start_idx = land_indices[0]
    goal_idx = land_indices[5]
    
    scale = gen.scale
    start_coarse = coarse.hexGrid.index_to_hexposition(start_idx, origin_index=coarse.hexGrid.midpoint)
    goal_coarse = coarse.hexGrid.index_to_hexposition(goal_idx, origin_index=coarse.hexGrid.midpoint)
    
    start_world = HexPosition(int(start_coarse.q * scale), int(start_coarse.r * scale), int(start_coarse.s * scale))
    goal_world = HexPosition(int(goal_coarse.q * scale), int(goal_coarse.r * scale), int(goal_coarse.s * scale))
    
    print(f"Start world: {start_world}")
    print(f"Goal world: {goal_world}")
    
    # Generate just the start chunk
    start_chunk = gen.world_to_chunk_ref(start_world)
    start_terrain = gen.generate_chunk(start_chunk)
    
    print(f"\nStart chunk center: {start_chunk.center_hex}")
    print(f"Start terrain hexes: {len(start_terrain.elevations)}")
    print(f"Start terrain elev range: {start_terrain.elevations.min():.0f} to {start_terrain.elevations.max():.0f}")
    
    # Try to find start_world in the chunk
    start_grid = start_terrain.hexGrid
    start_local = start_world - start_chunk.center_hex
    print(f"Start local in chunk: {start_local}")
    
    start_idx_in_chunk = start_grid.hexposition_to_index(start_local, origin_index=start_grid.midpoint)
    print(f"Start index in chunk: {start_idx_in_chunk} (valid: {0 <= start_idx_in_chunk < len(start_terrain.elevations)})")
    
    if 0 <= start_idx_in_chunk < len(start_terrain.elevations):
        print(f"Start elevation: {start_terrain.elevations[start_idx_in_chunk]:.1f}")


Are we aligned?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes! The chunk generation and coordinate mapping are working correctly:
- Start world position maps to the right chunk
- Local coordinates within the chunk are valid
- We can find the hex and get its elevation

So the problem must be in `_merge_chunks()` or the merged terrain's coordinate system. The merged grid probably isn't centered correctly, so when we try to look up `start_world` and `goal_world` in the merged terrain, we get invalid indices.

Let's debug the merge:

```python
with GeoStorageDebugger(keep_on_error=True) as dbg:
    gen = ChunkedTerrainGenerator(
        storage=dbg.server,
        coarse_rings=15,
        coarse_radius=50,
        fine_radius=10,
        chunk_rings=8,
        halo_rings=2
    )
    
    coarse = gen.generate_coarse(num_plates=8, ocean_fraction=0.3, oceanic_sides=['W'])
    
    # Same start/goal
    land_indices = [i for i, e in enumerate(coarse.elevations) if e > 0]
    start_idx, goal_idx = land_indices[0], land_indices[5]
    scale = gen.scale
    
    start_coarse = coarse.hexGrid.index_to_hexposition(start_idx, origin_index=coarse.hexGrid.midpoint)
    goal_coarse = coarse.hexGrid.index_to_hexposition(goal_idx, origin_index=coarse.hexGrid.midpoint)
    
    start_world = HexPosition(int(start_coarse.q * scale), int(start_coarse.r * scale), int(start_coarse.s * scale))
    goal_world = HexPosition(int(goal_coarse.q * scale), int(goal_coarse.r * scale), int(goal_coarse.s * scale))
    
    # Generate chunks
    chunk_refs = {gen.world_to_chunk_ref(start_world).key, gen.world_to_chunk_ref(goal_world).key}
    chunks = {}
    for key in chunk_refs:
        chunk_ref = ChunkRef.from_key(key, gen.chunk_rings)
        terrain = gen.generate_chunk(chunk_ref)
        chunks[key] = (chunk_ref, terrain)
    
    print(f"Merging {len(chunks)} chunks...")
    merged = _merge_chunks(gen, chunks)
    
    print(f"Merged terrain: {len(merged.elevations)} hexes")
    print(f"Merged grid midpoint: {merged.hexGrid.midpoint}")
    print(f"Merged elev range: {merged.elevations.min():.0f} to {merged.elevations.max():.0f}")
    print(f"Unmapped hexes (=-999): {sum(merged.elevations == -999.0)}")
    
    # Try to find start in merged
    start_idx_merged = merged.hexGrid.hexposition_to_index(start_world, origin_index=merged.hexGrid.midpoint)
    print(f"\nStart {start_world} -> merged index {start_idx_merged}")
    print(f"Valid: {0 <= start_idx_merged < len(merged.elevations)}")
    
    if 0 <= start_idx_merged < len(merged.elevations):
        print(f"Elevation: {merged.elevations[start_idx_merged]:.1f}")
```

This will show if the merged grid's coordinate system is broken.

In [ ]:
with GeoStorageDebugger(keep_on_error=True) as dbg:
    gen = ChunkedTerrainGenerator(
        storage=dbg.server,
        coarse_rings=15,
        coarse_radius=50,
        fine_radius=10,
        chunk_rings=8,
        halo_rings=2
    )
    
    coarse = gen.generate_coarse(num_plates=8, ocean_fraction=0.3, oceanic_sides=['W'])
    
    # Same start/goal
    land_indices = [i for i, e in enumerate(coarse.elevations) if e > 0]
    start_idx, goal_idx = land_indices[0], land_indices[5]
    scale = gen.scale
    
    start_coarse = coarse.hexGrid.index_to_hexposition(start_idx, origin_index=coarse.hexGrid.midpoint)
    goal_coarse = coarse.hexGrid.index_to_hexposition(goal_idx, origin_index=coarse.hexGrid.midpoint)
    
    start_world = HexPosition(int(start_coarse.q * scale), int(start_coarse.r * scale), int(start_coarse.s * scale))
    goal_world = HexPosition(int(goal_coarse.q * scale), int(goal_coarse.r * scale), int(goal_coarse.s * scale))
    
    # Generate chunks
    chunk_refs = {gen.world_to_chunk_ref(start_world).key, gen.world_to_chunk_ref(goal_world).key}
    chunks = {}
    for key in chunk_refs:
        chunk_ref = ChunkRef.from_key(key, gen.chunk_rings)
        terrain = gen.generate_chunk(chunk_ref)
        chunks[key] = (chunk_ref, terrain)
    
    print(f"Merging {len(chunks)} chunks...")
    merged = gen.merge_chunks( chunks)
    
    print(f"Merged terrain: {len(merged.elevations)} hexes")
    print(f"Merged grid midpoint: {merged.hexGrid.midpoint}")
    print(f"Merged elev range: {merged.elevations.min():.0f} to {merged.elevations.max():.0f}")
    print(f"Unmapped hexes (=-999): {sum(merged.elevations == -999.0)}")
    
    # Try to find start in merged
    start_idx_merged = merged.hexGrid.hexposition_to_index(start_world, origin_index=merged.hexGrid.midpoint)
    print(f"\nStart {start_world} -> merged index {start_idx_merged}")
    print(f"Valid: {0 <= start_idx_merged < len(merged.elevations)}")
    
    if 0 <= start_idx_merged < len(merged.elevations):
        print(f"Elevation: {merged.elevations[start_idx_merged]:.1f}")


### I think we can kill this

In [ ]:
#| export
from HexMagic.core import *
from HexMagic.primitives import HexPosition

In [ ]:
#| export
import heapq
from typing import Optional

def hex_distance(a: HexPosition, b: HexPosition) -> int:
    """Manhattan distance in hex coordinates."""
    return (abs(a.q - b.q) + abs(a.r - b.r) + abs(a.s - b.s)) // 2


def astar_terrain(terrain: Terrain, start_idx: int, goal_idx: int, 
                  uphill_penalty: float = 4.0) -> tuple[list[int], float]:
    """A* pathfinding on a Terrain. Returns (path_indices, total_cost)."""
    grid = terrain.hexGrid
    elevations = terrain.elevations
    
    start_pos = grid.index_to_hexposition(start_idx)
    goal_pos = grid.index_to_hexposition(goal_idx)
    
    # Priority queue: (f_score, counter, index)
    counter = 0
    frontier = [(0, counter, start_idx)]
    
    came_from = {start_idx: None}
    g_score = {start_idx: 0.0}
    
    while frontier:
        _, _, current = heapq.heappop(frontier)
        
        if current == goal_idx:
            # Reconstruct path
            path = []
            while current is not None:
                path.append(current)
                current = came_from[current]
            return path[::-1], g_score[goal_idx]
        
        current_pos = grid.index_to_hexposition(current)
        current_elev = elevations[current]
        
        # Check all 6 neighbors
        for neighbor_pos in current_pos.neighbors():
            neighbor_idx = grid.hexposition_to_index(neighbor_pos)
            
            # Skip if out of bounds
            if neighbor_idx < 0 or neighbor_idx >= len(elevations):
                continue
            
            neighbor_elev = elevations[neighbor_idx]
            
            # Skip water (below sea level)
            if neighbor_elev < terrain.seaLevel:
                continue
            
            cost = hex_move_cost(current_elev, neighbor_elev, uphill_penalty)
            tentative_g = g_score[current] + cost
            
            if neighbor_idx not in g_score or tentative_g < g_score[neighbor_idx]:
                g_score[neighbor_idx] = tentative_g
                f = tentative_g + hex_distance(neighbor_pos, goal_pos)
                counter += 1
                heapq.heappush(frontier, (f, counter, neighbor_idx))
                came_from[neighbor_idx] = current
    
    return [], float('inf')  # No path found


def visualize_path(terrain: Terrain, path: list[int], 
                   start_idx: int, goal_idx: int) -> str:
    """Add path visualization to terrain and return SVG."""
    grid = terrain.hexGrid
    
    # Color the path
    for idx in path:
        hex_obj = grid.hexes[idx]
        if idx == start_idx:
            hex_obj.fill = "#00ff00"  # Green start
        elif idx == goal_idx:
            hex_obj.fill = "#ff0000"  # Red goal
        else:
            hex_obj.fill = "#ffff00"  # Yellow path
    
    terrain.hexGrid.update()
    return terrain.builder


In [ ]:
#| export
def astar_terrain(terrain: Terrain, start_idx: int, goal_idx: int, 
                  uphill_penalty: float = 4.0) -> tuple[list[int], float]:
    """A* pathfinding on a Terrain. Returns (path_indices, total_cost)."""
    grid = terrain.hexGrid
    elevations = terrain.elevations
    
    start_pos = grid.index_to_hexposition(start_idx)
    goal_pos = grid.index_to_hexposition(goal_idx)
    
    # Priority queue: (f_score, counter, index)
    counter = 0
    frontier = [(0, counter, start_idx)]
    
    came_from = {start_idx: None}
    g_score = {start_idx: 0.0}
    
    while frontier:
        _, _, current = heapq.heappop(frontier)
        
        if current == goal_idx:
            # Reconstruct path
            path = []
            while current is not None:
                path.append(current)
                current = came_from[current]
            return path[::-1], g_score[goal_idx]
        
        current_elev = elevations[current]
        
        # Use terrain.ring to get neighbor indices
        for neighbor_idx in terrain.ring(current, 1):
            # Skip if out of bounds
            if neighbor_idx < 0 or neighbor_idx >= len(elevations):
                continue
            
            # Skip water (elevationLevel returns -1 for ocean)
            if terrain.elevationLevel(neighbor_idx) < 0:
                cost = 100  # Boats!
            else:
                neighbor_elev = elevations[neighbor_idx]
                cost = hex_move_cost(current_elev, neighbor_elev, uphill_penalty)

            
            #neighbor_elev = elevations[neighbor_idx]
            #cost = hex_move_cost(current_elev, neighbor_elev, uphill_penalty)
            tentative_g = g_score[current] + cost
            
            if neighbor_idx not in g_score or tentative_g < g_score[neighbor_idx]:
                g_score[neighbor_idx] = tentative_g
                neighbor_pos = grid.index_to_hexposition(neighbor_idx)
                f = tentative_g + hex_distance(neighbor_pos, goal_pos)
                counter += 1
                heapq.heappush(frontier, (f, counter, neighbor_idx))
                came_from[neighbor_idx] = current
    
    return [], float('inf')  # No path found


In [ ]:
# Find coastal and mountain hexes using terrain methods
# Coastal = elevation level 0 (just above sea)
# Mountain = highest peaks

# Find a low elevation hex (level 0 or 1)
coastal_idx = None
for i in range(len(sfTerr.elevations)):
    level = sfTerr.elevationLevel(i)
    if level == 0:  # Just above sea level
        coastal_idx = i
        break

# Find a high peak
peaks = sfTerr.find_peaks(k=1, min_height=50, exclusion_radius=3)
mountain_idx = peaks[0] if peaks else None

print(f"Start (coastal): idx={coastal_idx}, level={sfTerr.elevationLevel(coastal_idx)}, elev={sfTerr.elevations[coastal_idx]:.1f}")
print(f"Goal (mountain): idx={mountain_idx}, level={sfTerr.elevationLevel(mountain_idx)}, elev={sfTerr.elevations[mountain_idx]:.1f}")

# Pick hexes on opposite coasts
coastal_idx = 50    # West side
mountain_idx = 150  # East side (adjust based on your grid)

# Run A*
path, cost = astar_terrain(sfTerr, coastal_idx, mountain_idx)
print(f"Path found: {len(path)} hexes, total cost: {cost:.1f}")

# Visualize
sfTerr.colorMap()
builder = visualize_path(sfTerr, path, coastal_idx, mountain_idx)
len(builder.xml())


In [ ]:
#| export
def refine_coarse_path(generator: ChunkedTerrainGenerator, 
                       coarse: Terrain,
                       coarse_path: list[int],
                       uphill_penalty: float = 4.0) -> tuple[list[HexPosition], float]:
    """Refine a coarse path through fine chunks.
    
    Returns (world_positions, total_cost)
    """
    if len(coarse_path) < 2:
        return [], 0.0
    
    coarse_grid = coarse.hexGrid
    scale = generator.scale
    
    fine_path = []
    total_cost = 0.0
    
    for i in range(len(coarse_path) - 1):
        coarse_idx = coarse_path[i]
        next_coarse_idx = coarse_path[i + 1]
        
        # Get coarse positions
        coarse_pos = coarse_grid.index_to_hexposition(coarse_idx)
        next_coarse_pos = coarse_grid.index_to_hexposition(next_coarse_idx)
        
        # Map to fine world positions (centers of coarse hexes)
        fine_start = HexPosition(
            int(coarse_pos.q * scale),
            int(coarse_pos.r * scale),
            int(coarse_pos.s * scale)
        )
        fine_goal = HexPosition(
            int(next_coarse_pos.q * scale),
            int(next_coarse_pos.r * scale),
            int(next_coarse_pos.s * scale)
        )
        
        # Which chunk contains the start?
        chunk_ref = generator.world_to_chunk_ref(fine_start)
        chunk_terrain = generator.load_chunk(chunk_ref)
        
        if chunk_terrain is None:
            # Chunk not generated - fall back to coarse position
            fine_path.append(fine_start)
            total_cost += 1.0
            continue
        
        # Map world positions to local chunk indices
        chunk_center = chunk_ref.center_hex
        local_start = fine_start - chunk_center
        local_goal = fine_goal - chunk_center
        
        chunk_grid = chunk_terrain.hexGrid
        start_idx = chunk_grid.hexposition_to_index(local_start, origin_index=chunk_grid.middle)
        goal_idx = chunk_grid.hexposition_to_index(local_goal, origin_index=chunk_grid.middle)
        
        # Bounds check
        if start_idx < 0 or goal_idx < 0:
            fine_path.append(fine_start)
            total_cost += 1.0
            continue
        if start_idx >= len(chunk_terrain.elevations) or goal_idx >= len(chunk_terrain.elevations):
            fine_path.append(fine_start)
            total_cost += 1.0
            continue
        
        # Local A* within chunk
        local_path, local_cost = astar_terrain(chunk_terrain, start_idx, goal_idx, uphill_penalty)
        
        if local_path:
            # Convert local indices back to world positions
            for idx in local_path[:-1]:  # Skip last to avoid duplicates
                local_pos = chunk_grid.index_to_hexposition(idx, origin_index=chunk_grid.middle)
                world_pos = local_pos + chunk_center
                fine_path.append(world_pos)
            total_cost += local_cost
        else:
            # No local path - just add the coarse waypoint
            fine_path.append(fine_start)
            total_cost += hex_distance(fine_start, fine_goal)
    
    # Add final goal
    last_coarse_pos = coarse_grid.index_to_hexposition(coarse_path[-1])
    final_pos = HexPosition(
        int(last_coarse_pos.q * scale),
        int(last_coarse_pos.r * scale),
        int(last_coarse_pos.s * scale)
    )
    fine_path.append(final_pos)
    
    return fine_path, total_cost


#| export
I redid the ChunkedTerrainGenerator and the complete chunk and terrain gneration. How should the path finding algorihm be changed. I know the theory stays the same - look at higher levels, work in lower ones.


##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Looking at your updated system, here's how pathfinding should adapt:

```python
def hpa_pathfind(gen: ChunkedTerrainGenerator,
                 start_world: HexPosition,
                 goal_world: HexPosition,
                 uphill_penalty: float = 4.0) -> tuple[list[HexPosition], float]:
    """Hierarchical pathfinding: coarse plan, fine execution."""
    
    coarse = gen._coarse_terrain
    coarse_grid = coarse.hexGrid
    scale = gen.scale
    
    # 1. Map world positions to coarse indices
    start_coarse = HexPosition(
        int(round(start_world.q / scale)),
        int(round(start_world.r / scale)),
        -int(round(start_world.q / scale)) - int(round(start_world.r / scale))
    )
    goal_coarse = HexPosition(
        int(round(goal_world.q / scale)),
        int(round(goal_world.r / scale)),
        -int(round(goal_world.q / scale)) - int(round(goal_world.r / scale))
    )
    
    start_coarse_idx = coarse_grid.hexposition_to_index(start_coarse, origin_index=coarse_grid.midpoint)
    goal_coarse_idx = coarse_grid.hexposition_to_index(goal_coarse, origin_index=coarse_grid.midpoint)
    
    # 2. Coarse A* to get waypoints
    coarse_path, _ = astar_terrain(coarse, start_coarse_idx, goal_coarse_idx, uphill_penalty)
    
    if not coarse_path:
        return [], float('inf')
    
    # 3. Convert coarse path to world waypoints
    waypoints = []
    for idx in coarse_path:
        pos = coarse_grid.index_to_hexposition(idx, origin_index=coarse_grid.midpoint)
        world_pos = HexPosition(int(pos.q * scale), int(pos.r * scale), int(pos.s * scale))
        waypoints.append(world_pos)
    
    # Override first/last with exact positions
    waypoints[0] = start_world
    waypoints[-1] = goal_world
    
    # 4. Refine each segment through fine chunks
    fine_path = [start_world]
    total_cost = 0.0
    
    for i in range(len(waypoints) - 1):
        segment_path, segment_cost = _refine_segment(
            gen, waypoints[i], waypoints[i+1], uphill_penalty
        )
        fine_path.extend(segment_path[1:])  # Skip duplicate start
        total_cost += segment_cost
    
    return fine_path, total_cost


def _refine_segment(gen: ChunkedTerrainGenerator,
                    start: HexPosition, 
                    goal: HexPosition,
                    uphill_penalty: float) -> tuple[list[HexPosition], float]:
    """Refine one coarse segment through fine terrain."""
    
    # Get chunk(s) involved
    start_chunk = gen.world_to_chunk_ref(start)
    goal_chunk = gen.world_to_chunk_ref(goal)
    
    # Same chunk? Simple local A*
    if start_chunk.key == goal_chunk.key:
        return _local_astar(gen, start_chunk, start, goal, uphill_penalty)
    
    # Different chunks: path to border, then recurse
    # Find border hex closest to goal direction
    border_hex = _find_border_toward(gen, start_chunk, goal)
    
    path1, cost1 = _local_astar(gen, start_chunk, start, border_hex, uphill_penalty)
    path2, cost2 = _refine_segment(gen, border_hex, goal, uphill_penalty)
    
    return path1 + path2[1:], cost1 + cost2


def _local_astar(gen: ChunkedTerrainGenerator,
                 chunk_ref: ChunkRef,
                 start_world: HexPosition,
                 goal_world: HexPosition,
                 uphill_penalty: float) -> tuple[list[HexPosition], float]:
    """A* within a single chunk, returning world positions."""
    
    # Load or generate chunk
    chunk_terrain = gen.load_chunk(chunk_ref)
    if chunk_terrain is None:
        chunk_terrain = gen.generate_chunk(chunk_ref)
        gen.save_chunk(chunk_terrain, chunk_ref)
    
    chunk_grid = chunk_terrain.hexGrid
    center = chunk_ref.center_hex
    
    # World -> local
    start_local = start_world - center
    goal_local = goal_world - center
    
    start_idx = chunk_grid.hexposition_to_index(start_local, origin_index=chunk_grid.midpoint)
    goal_idx = chunk_grid.hexposition_to_index(goal_local, origin_index=chunk_grid.midpoint)
    
    # Run A*
    local_path, cost = astar_terrain(chunk_terrain, start_idx, goal_idx, uphill_penalty)
    
    # Local -> world
    world_path = []
    for idx in local_path:
        local_pos = chunk_grid.index_to_hexposition(idx, origin_index=chunk_grid.midpoint)
        world_path.append(local_pos + center)
    
    return world_path, cost


def _find_border_toward(gen: ChunkedTerrainGenerator,
                        chunk_ref: ChunkRef,
                        goal: HexPosition) -> HexPosition:
    """Find the border hex of chunk closest to goal direction."""
    center = chunk_ref.center_hex
    direction = HexPosition(
        1 if goal.q > center.q else (-1 if goal.q < center.q else 0),
        1 if goal.r > center.r else (-1 if goal.r < center.r else 0),
        0
    )
    direction = HexPosition(direction.q, direction.r, -direction.q - direction.r)
    
    # Border is at chunk_rings distance from center
    border = center + HexPosition(
        direction.q * gen.chunk_rings,
        direction.r * gen.chunk_rings,
        direction.s * gen.chunk_rings
    )
    return border
```

The key changes from before:

1. **Coordinate transforms** use `gen.scale` consistently
2. **Chunk loading** uses your `load_chunk`/`save_chunk` pattern
3. **Border crossing** handled recursively via `_refine_segment`
4. **World positions** throughout (not mixing local/world)

Want me to add a test on SF terrain first before trying the generated chunks?

In [ ]:
# Create a larger chunked world
with GeoStorageDebugger(keep_on_error=False) as dbg:
    generator = ChunkedTerrainGenerator(
        storage=dbg.server,
        coarse_radius=50,
        fine_radius=10,
        chunk_rings=10,
        halo_rings=2
    )
    
    bounds = MapRect(MapCord(0, 0), MapSize(1200, 900))
    coarse = generator.generate_coarse(
        bounds,
        num_plates=8,
        ocean_fraction=0.3,
        oceanic_sides=['W'],
        name="pathfinding_test"
    )
    
    # Generate chunks covering the area
    chunks_to_generate = list(generator.chunk_refs(max_ring=3))
    print(f"Generating {len(chunks_to_generate)} chunks...")
    for chunk in chunks_to_generate:
        chunk_terrain = generator.generate_chunk(chunk, coarse)
        generator.save_chunk(chunk_terrain, chunk)
    
    # Find coarse path
    land_hexes = [i for i in range(len(coarse.elevations)) 
                  if coarse.elevationLevel(i) >= 0]
    start_coarse = land_hexes[0]
    goal_coarse = land_hexes[-1]
    
    coarse_path, coarse_cost = astar_terrain(coarse, start_coarse, goal_coarse)
    print(f"Coarse path: {len(coarse_path)} hexes, cost: {coarse_cost:.1f}")
    
    # Refine through chunks
    fine_path, fine_cost = refine_coarse_path(generator, coarse, coarse_path)
    print(f"Fine path: {len(fine_path)} positions, cost: {fine_cost:.1f}")
    
    # Show coarse with path
    coarse.colorMap()
    visualize_path(coarse, coarse_path, start_coarse, goal_coarse)
    display(coarse.builder.show())
    
    print(f"\nRefinement ratio: {len(fine_path)} fine / {len(coarse_path)} coarse = {len(fine_path)/len(coarse_path):.1f}x detail")


I messed something up

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

The issue is the `@patch` for `query_hexes_in_radius` in the note block isn't being executed (notes don't run). The code cell version tries to access `cursor.description` after the query completes.

Quick fix—replace the code cell version with the hardcoded columns approach from your note:

```python
@patch
def query_hexes_in_radius(self: GeoStorage, world_id: int,
                          center_q: int, center_r: int, center_s: int,
                          radius: int, as_of: int = None) -> list:
    """Query latest hexes within radius of a center position."""
    subq = self._latest_hex_subquery(world_id, as_of)
    
    rows = list(self.db.execute(f"""
        SELECT h.* FROM hex_data h
        INNER JOIN ({subq}) latest 
            ON h.world_id = latest.world_id 
            AND h.q = latest.q AND h.r = latest.r AND h.s = latest.s
            AND h.modified = latest.max_mod
        WHERE h.q BETWEEN ? AND ? AND h.r BETWEEN ? AND ?
    """, [
        center_q - radius, center_q + radius,
        center_r - radius, center_r + radius
    ]).fetchall())
    
    if not rows:
        return []
    
    # Hardcode column order to avoid cursor.description issue
    cols = ['id', 'world_id', 'q', 'r', 's', 'grid_index', 'elevation', 
            'latitude', 'longitude', 'distance_from_coast', 'watershed_id', 'modified']
    
    rows = [dict(zip(cols, row)) for row in rows]
    
    # Filter to actual hex distance
    return [row for row in rows
            if max(abs(row['q'] - center_q), abs(row['r'] - center_r), abs(row['s'] - center_s)) <= radius]
```

Run that as a code cell before the chunked world test.

I guess it is working

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes! It's working nicely:

- **Coarse path**: 31 hexes, cost 1012
- **Fine path**: 56 positions, cost 350

The fine cost being *lower* is actually correct—the detailed search found better routes around elevation changes that the coarse map couldn't see.

Next steps if you want to push further:

1. **Visualize the fine path** on a stitched chunk view
2. **Cache chunk border connectivity** in the database (so you don't need to load chunks just for pathfinding)
3. **Compare against full A*** on the fine grid to measure how close to optimal HPA* gets
4. **Add terrain types** (roads = 0.5 cost, forest = 2.0, etc.)

Or is this enough for now and you want to move on to something else?

In [ ]:
WatershedMeta??

In [ ]:
#!cat ../HexMagic/water/*.py

In [ ]:
#!cat ../HexMagic/database.py

I am wondering about watershed can we do some of our plates and chunks to make a nixe algorithm. could we also use these thing for weather

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes! Both watersheds and weather can benefit from the hierarchical approach:

## Watersheds Across Chunks

The key insight: water flows downhill, so we can trace drainage at coarse level first.

```python
def chunked_watersheds(gen: ChunkedTerrainGenerator):
    """Compute watersheds using coarse-to-fine strategy."""
    coarse = gen._coarse_terrain
    
    # 1. Find drainage directions on coarse map
    #    Each coarse hex points to its lowest neighbor
    coarse_flow = compute_flow_directions(coarse)
    
    # 2. Identify pour points - where water exits each chunk
    #    These are coarse hexes at chunk borders that flow outward
    pour_points = {}  # chunk_key -> list of (border_hex, downstream_chunk)
    
    # 3. For each chunk, compute local watersheds
    #    But mark border hexes that drain to pour points
    for chunk_ref in gen.chunk_refs(max_ring=3):
        chunk_terrain = gen.load_chunk(chunk_ref)
        
        # Local watershed computation
        local_basins = DrainageBasins(chunk_terrain, 
                                       border_drains=pour_points.get(chunk_ref.key))
        
        # Rivers that hit the border continue in the next chunk
        for river in local_basins.border_rivers:
            # Hand off to downstream chunk
            downstream_chunk = pour_points[chunk_ref.key][river.exit_hex]
            connect_river_across_boundary(river, downstream_chunk)
```

## Weather Across Chunks

Weather is simpler - mostly local with some regional effects:

```python
def chunked_weather(gen: ChunkedTerrainGenerator, season: str = "annual"):
    """Compute weather using coarse patterns + fine detail."""
    coarse = gen._coarse_terrain
    
    # 1. Compute coarse weather patterns (wind, moisture sources)
    coarse_patterns = TerrainPatterns(coarse)
    coarse_weather = coarse_patterns.weatherPatterns()[gen.climate_preset]
    
    # 2. For each chunk, interpolate coarse + add local effects
    for chunk_ref in gen.chunk_refs(max_ring=3):
        chunk_terrain = gen.load_chunk(chunk_ref)
        
        # Inherit regional patterns from coarse
        apply_regional_weather(chunk_terrain, coarse_weather, chunk_ref, gen.scale)
        
        # Add local effects (rain shadows, valley inversions)
        apply_local_weather_effects(chunk_terrain)
        
        # Save weather data
        gen.storage.save_weather(chunk_terrain, chunk_world_id, season)
```

## Database Support

Add these to help:

```python
@dataclass
class ChunkBorder:
    """Track drainage across chunk boundaries."""
    id: int = None
    world_id: int = 0
    chunk_q: int = 0
    chunk_r: int = 0  
    chunk_s: int = 0
    border_hex_q: int = 0  # Local hex at border
    border_hex_r: int = 0
    border_hex_s: int = 0
    downstream_chunk_q: int = 0  # Which chunk it drains to
    downstream_chunk_r: int = 0
    downstream_chunk_s: int = 0
    flow_volume: float = 0.0  # Accumulated upstream area
```

This lets you:
1. Query "which chunks drain into this one?" for river assembly
2. Cache watershed boundaries so you don't recompute
3. Track major rivers across the whole world at coarse level

Want me to flesh out any of these pieces?

## building weather

I just added chunked boarders. what would be something that would be great at computing weather. perhaps something we could patch on ChunkedTerrainGenerator

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Here's a weather computation system for chunks:

```python
@patch
def compute_coarse_weather(self: ChunkedTerrainGenerator, 
                           climate_preset: str = "temperate",
                           prevailing_wind: str = "W") -> dict:
    """Compute weather patterns on coarse map."""
    coarse = self._coarse_terrain
    
    # Use existing TerrainPatterns for base computation
    patterns = TerrainPatterns(coarse)
    weather = patterns.weatherPatterns().get(climate_preset, patterns.weatherPatterns()["temperate"])
    
    # Store on coarse terrain
    coarse.fields['temperature'] = weather['temperature']
    coarse.fields['precipitation'] = weather['precipitation']
    coarse.fields['wind_direction'] = np.full(len(coarse.elevations), prevailing_wind)
    
    self._coarse_weather = weather
    self._climate_preset = climate_preset
    self._prevailing_wind = prevailing_wind
    
    return weather


@patch
def compute_chunk_weather(self: ChunkedTerrainGenerator, 
                          chunk: ChunkRef,
                          chunk_terrain: Terrain = None) -> Terrain:
    """Compute weather for a chunk by interpolating coarse + local effects."""
    
    if chunk_terrain is None:
        chunk_terrain = self.load_chunk(chunk)
        if chunk_terrain is None:
            chunk_terrain = self.generate_chunk(chunk)
    
    coarse = self._coarse_terrain
    coarse_grid = coarse.hexGrid
    fine_grid = chunk_terrain.hexGrid
    n_hexes = len(chunk_terrain.elevations)
    
    # Initialize weather fields
    chunk_terrain.fields['temperature'] = np.zeros(n_hexes)
    chunk_terrain.fields['precipitation'] = np.zeros(n_hexes)
    
    # Interpolate from coarse
    for fine_idx in range(n_hexes):
        fine_local = fine_grid.index_to_hexposition(fine_idx, origin_index=fine_grid.midpoint)
        world_pos = chunk.center_hex + fine_local
        
        # Map to coarse
        coarse_q = int(round(world_pos.q / self.scale))
        coarse_r = int(round(world_pos.r / self.scale))
        coarse_s = -coarse_q - coarse_r
        coarse_pos = HexPosition(coarse_q, coarse_r, coarse_s)
        coarse_idx = coarse_grid.hexposition_to_index(coarse_pos, origin_index=coarse_grid.midpoint)
        
        if 0 <= coarse_idx < len(coarse.elevations):
            # Base values from coarse
            base_temp = coarse.fields['temperature'][coarse_idx]
            base_precip = coarse.fields['precipitation'][coarse_idx]
            
            # Local elevation adjustment
            coarse_elev = coarse.elevations[coarse_idx]
            fine_elev = chunk_terrain.elevations[fine_idx]
            elev_diff = fine_elev - coarse_elev
            
            # Lapse rate: ~6.5°C per 1000m
            temp_adjust = -elev_diff * 0.0065
            
            # Orographic precipitation (higher = more on windward)
            if fine_elev > coarse_elev and fine_elev > 0:
                precip_adjust = 1.0 + (elev_diff / 500) * 0.3  # 30% more per 500m rise
            else:
                precip_adjust = 1.0
            
            chunk_terrain.fields['temperature'][fine_idx] = base_temp + temp_adjust
            chunk_terrain.fields['precipitation'][fine_idx] = base_precip * precip_adjust
        else:
            # Fallback
            chunk_terrain.fields['temperature'][fine_idx] = 15.0
            chunk_terrain.fields['precipitation'][fine_idx] = 500.0
    
    # Apply rain shadow effect using chunk borders
    self._apply_rain_shadow(chunk_terrain, chunk)
    
    return chunk_terrain


@patch  
def _apply_rain_shadow(self: ChunkedTerrainGenerator,
                       chunk_terrain: Terrain,
                       chunk: ChunkRef):
    """Reduce precipitation on lee side of mountains."""
    
    # Get prevailing wind direction
    wind_dir = getattr(self, '_prevailing_wind', 'W')
    wind_vector = {
        'W': HexPosition(1, 0, -1),   # Wind from west = flows east
        'E': HexPosition(-1, 0, 1),
        'N': HexPosition(0, 1, -1),
        'S': HexPosition(0, -1, 1),
    }.get(wind_dir, HexPosition(1, 0, -1))
    
    grid = chunk_terrain.hexGrid
    elevations = chunk_terrain.elevations
    
    # For each hex, check upwind for mountains
    for idx in range(len(elevations)):
        if elevations[idx] < 0:  # Skip ocean
            continue
            
        pos = grid.index_to_hexposition(idx, origin_index=grid.midpoint)
        
        # Look upwind (opposite of wind direction)
        upwind_pos = pos - wind_vector
        upwind_idx = grid.hexposition_to_index(upwind_pos, origin_index=grid.midpoint)
        
        if 0 <= upwind_idx < len(elevations):
            upwind_elev = elevations[upwind_idx]
            current_elev = elevations[idx]
            
            # If upwind is higher, we're in rain shadow
            if upwind_elev > current_elev + 50:  # Significant drop
                shadow_factor = min(0.7, (upwind_elev - current_elev) / 500)
                chunk_terrain.fields['precipitation'][idx] *= (1.0 - shadow_factor)


@patch
def generate_all_weather(self: ChunkedTerrainGenerator,
                         climate_preset: str = "temperate",
                         max_ring: int = 3) -> dict:
    """Generate weather for coarse map and all chunks."""
    
    # Coarse first
    self.compute_coarse_weather(climate_preset)
    print(f"✓ Coarse weather computed")
    
    # Each chunk
    results = {}
    for chunk_ref in self.chunk_refs(max_ring):
        chunk_terrain = self.load_chunk(chunk_ref)
        if chunk_terrain is None:
            chunk_terrain = self.generate_chunk(chunk_ref)
        
        self.compute_chunk_weather(chunk_ref, chunk_terrain)
        results[chunk_ref.key] = chunk_terrain
        
        # Optionally save weather to DB
        if chunk_ref.key in self.chunk_world_ids:
            world_id = self.chunk_world_ids[chunk_ref.key]
            self.storage.save_weather(chunk_terrain, world_id, season="annual")
    
    print(f"✓ Weather computed for {len(results)} chunks")
    return results
```

Test it:

```python
with GeoStorageDebugger(keep_on_error=False) as dbg:
    gen = ChunkedTerrainGenerator(
        storage=dbg.server,
        coarse_rings=15,
        coarse_radius=50,
        fine_radius=10,
        chunk_rings=8,
        halo_rings=2
    )
    
    coarse = gen.generate_coarse(num_plates=8, ocean_fraction=0.3, oceanic_sides=['W'])
    
    # Generate and save some chunks first
    for chunk_ref in gen.chunk_refs(max_ring=1):
        terrain = gen.generate_chunk(chunk_ref)
        gen.save_chunk(terrain, chunk_ref)
    
    # Now compute weather
    weather_results = gen.generate_all_weather(climate_preset="temperate", max_ring=1)
    
    # Check a chunk
    sample = list(weather_results.values())[0]
    print(f"Temp range: {sample.fields['temperature'].min():.1f} to {sample.fields['temperature'].max():.1f}°C")
    print(f"Precip range: {sample.fields['precipitation'].min():.0f} to {sample.fields['precipitation'].max():.0f} mm")
```

In [ ]:
#!cat ../HexMagic/weather.py

So we had the existing weather system which had nice climate settings. do these work with our model? 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes! The existing system integrates well. Here's how to connect them:

```python
@patch
def compute_coarse_weather(self: ChunkedTerrainGenerator, 
                           climate_preset: str = "temperate") -> Terrain:
    """Compute weather on coarse map using existing climate system."""
    coarse = self._coarse_terrain
    
    # Get preset from existing patterns
    patterns = TerrainPatterns(coarse)
    preset = patterns.weatherPatterns().get(climate_preset)
    
    if preset is None:
        raise ValueError(f"Unknown climate preset: {climate_preset}")
    
    # Apply to coarse terrain using existing method
    coarse.climate = preset
    coarse.compute_weather(force_recompute=True)
    
    self._climate_preset = preset
    return coarse


@patch
def compute_chunk_weather(self: ChunkedTerrainGenerator, 
                          chunk: ChunkRef,
                          chunk_terrain: Terrain = None) -> Terrain:
    """Refine coarse weather to chunk with local elevation effects."""
    
    if not hasattr(self, '_climate_preset'):
        raise ValueError("Call compute_coarse_weather() first")
    
    if chunk_terrain is None:
        chunk_terrain = self.load_chunk(chunk)
    
    coarse = self._coarse_terrain
    preset = self._climate_preset
    fine_grid = chunk_terrain.hexGrid
    coarse_grid = coarse.hexGrid
    n_hexes = len(chunk_terrain.elevations)
    
    # Initialize from coarse interpolation
    chunk_terrain.fields['temperature'] = np.zeros(n_hexes)
    chunk_terrain.fields['precipitation'] = np.zeros(n_hexes)
    
    for idx in range(n_hexes):
        local = fine_grid.index_to_hexposition(idx, origin_index=fine_grid.midpoint)
        world = chunk.center_hex + local
        
        # Map to coarse
        coarse_q = int(round(world.q / self.scale))
        coarse_r = int(round(world.r / self.scale))
        coarse_pos = HexPosition(coarse_q, coarse_r, -coarse_q - coarse_r)
        coarse_idx = coarse_grid.hexposition_to_index(coarse_pos, origin_index=coarse_grid.midpoint)
        
        if 0 <= coarse_idx < len(coarse.elevations):
            # Base from coarse
            base_temp = coarse.fields['temperature'][coarse_idx]
            base_precip = coarse.fields['precipitation'][coarse_idx]
            coarse_elev = coarse.elevations[coarse_idx]
            fine_elev = chunk_terrain.elevations[idx]
            
            # Lapse rate adjustment (use preset value)
            elev_diff = fine_elev - coarse_elev
            temp_adjust = -elev_diff * (preset.lapse_rate / 1000.0)
            
            # Orographic precipitation adjustment
            if fine_elev > coarse_elev and fine_elev > 0:
                # Enhanced uplift on windward slopes
                precip_factor = 1.0 + (elev_diff / 500) * preset.cw * 100
            elif fine_elev < coarse_elev and fine_elev > 0:
                # Rain shadow on lee side
                precip_factor = max(0.5, 1.0 - (coarse_elev - fine_elev) / 1000 * 0.3)
            else:
                precip_factor = 1.0
            
            chunk_terrain.fields['temperature'][idx] = base_temp + temp_adjust
            chunk_terrain.fields['precipitation'][idx] = base_precip * precip_factor
        else:
            # Fallback to preset defaults
            chunk_terrain.fields['temperature'][idx] = np.mean(preset.base_temp_range)
            chunk_terrain.fields['precipitation'][idx] = preset.precip_base * 8760
    
    # Copy climate preset for rendering
    chunk_terrain.climate = preset
    
    return chunk_terrain
```

Test it:

```python
with GeoStorageDebugger(keep_on_error=False) as dbg:
    gen = ChunkedTerrainGenerator(
        storage=dbg.server,
        coarse_rings=15,
        coarse_radius=50,
        fine_radius=10,
        chunk_rings=8,
        halo_rings=2
    )
    
    coarse = gen.generate_coarse(num_plates=8, ocean_fraction=0.3, oceanic_sides=['W'])
    
    # Use existing climate system
    gen.compute_coarse_weather(climate_preset="mediterranean")
    
    print(f"Coarse temp: {coarse.fields['temperature'].min():.1f} to {coarse.fields['temperature'].max():.1f}°C")
    print(f"Coarse precip: {coarse.fields['precipitation'].min():.0f} to {coarse.fields['precipitation'].max():.0f} mm")
    
    # Generate a chunk with weather
    chunk_ref = ChunkRef(HexPosition(0, 0, 0), gen.chunk_rings)
    chunk = gen.generate_chunk(chunk_ref)
    chunk = gen.compute_chunk_weather(chunk_ref, chunk)
    
    print(f"\nChunk temp: {chunk.fields['temperature'].min():.1f} to {chunk.fields['temperature'].max():.1f}°C")
    print(f"Chunk precip: {chunk.fields['precipitation'].min():.0f} to {chunk.fields['precipitation'].max():.0f} mm")
```

The key integration points:
- Uses `TerrainPatterns.weatherPatterns()` to get existing presets
- Applies `terrain.compute_weather()` on coarse map (Smith-Barstad precipitation)
- Inherits `preset.lapse_rate` and `preset.cw` for fine-scale adjustments
- Chunks get `climate` attribute for rendering with `render_icon_temperature()`

In [ ]:
#| export
@patch
def compute_coarse_weather(self: ChunkedTerrainGenerator, 
                           climate_preset: str = "temperate") -> Terrain:
    """Compute weather on coarse map using existing climate system."""
    coarse = self._coarse_terrain
    
    # Get preset from existing patterns
    patterns = TerrainPatterns(coarse)
    preset = patterns.weatherPatterns().get(climate_preset)
    
    if preset is None:
        raise ValueError(f"Unknown climate preset: {climate_preset}")
    
    # Apply to coarse terrain using existing method
    coarse.climate = preset
    coarse.compute_weather(force_recompute=True)
    
    self._climate_preset = preset
    return coarse


@patch
def compute_chunk_weather(self: ChunkedTerrainGenerator, 
                          chunk: ChunkRef,
                          chunk_terrain: Terrain = None) -> Terrain:
    """Refine coarse weather to chunk with local elevation effects."""
    
    if not hasattr(self, '_climate_preset'):
        raise ValueError("Call compute_coarse_weather() first")
    
    if chunk_terrain is None:
        chunk_terrain = self.load_chunk(chunk)
    
    coarse = self._coarse_terrain
    preset = self._climate_preset
    fine_grid = chunk_terrain.hexGrid
    coarse_grid = coarse.hexGrid
    n_hexes = len(chunk_terrain.elevations)
    
    # Initialize from coarse interpolation
    chunk_terrain.fields['temperature'] = np.zeros(n_hexes)
    chunk_terrain.fields['precipitation'] = np.zeros(n_hexes)
    
    for idx in range(n_hexes):
        local = fine_grid.index_to_hexposition(idx, origin_index=fine_grid.midpoint)
        world = chunk.center_hex + local
        
        # Map to coarse
        coarse_q = int(round(world.q / self.scale))
        coarse_r = int(round(world.r / self.scale))
        coarse_pos = HexPosition(coarse_q, coarse_r, -coarse_q - coarse_r)
        coarse_idx = coarse_grid.hexposition_to_index(coarse_pos, origin_index=coarse_grid.midpoint)
        
        if 0 <= coarse_idx < len(coarse.elevations):
            # Base from coarse
            base_temp = coarse.fields['temperature'][coarse_idx]
            base_precip = coarse.fields['precipitation'][coarse_idx]
            coarse_elev = coarse.elevations[coarse_idx]
            fine_elev = chunk_terrain.elevations[idx]
            
            # Lapse rate adjustment (use preset value)
            elev_diff = fine_elev - coarse_elev
            temp_adjust = -elev_diff * (preset.lapse_rate / 1000.0)
            
            # Orographic precipitation adjustment
            if fine_elev > coarse_elev and fine_elev > 0:
                # Enhanced uplift on windward slopes
                precip_factor = 1.0 + (elev_diff / 500) * preset.cw * 100
            elif fine_elev < coarse_elev and fine_elev > 0:
                # Rain shadow on lee side
                precip_factor = max(0.5, 1.0 - (coarse_elev - fine_elev) / 1000 * 0.3)
            else:
                precip_factor = 1.0
            
            chunk_terrain.fields['temperature'][idx] = base_temp + temp_adjust
            chunk_terrain.fields['precipitation'][idx] = base_precip * precip_factor
        else:
            # Fallback to preset defaults
            chunk_terrain.fields['temperature'][idx] = np.mean(preset.base_temp_range)
            chunk_terrain.fields['precipitation'][idx] = preset.precip_base * 8760
    
    # Copy climate preset for rendering
    chunk_terrain.climate = preset
    
    return chunk_terrain


In [ ]:
with GeoStorageDebugger(keep_on_error=False) as dbg:
    gen = ChunkedTerrainGenerator(
        storage=dbg.server,
        coarse_rings=15,
        coarse_radius=50,
        fine_radius=10,
        chunk_rings=8,
        halo_rings=2
    )
    
    coarse = gen.generate_coarse(num_plates=8, ocean_fraction=0.3, oceanic_sides=['W'])
    coarse.geo = GeoBounds(
            lat_min=20.57,   # Southern tip (near Makena)
            lat_max=21.03,   # Northern tip (near Kahakuloa)
            lon_min=-156.69, # Western tip (West Maui)
            lon_max=-155.97  # Eastern tip (Haleakalā/Hāna)
        )
    
    # Use existing climate system
    gen.compute_coarse_weather(climate_preset="mediterranean")
    
    print(f"Coarse temp: {coarse.fields['temperature'].min():.1f} to {coarse.fields['temperature'].max():.1f}°C")
    print(f"Coarse precip: {coarse.fields['precipitation'].min():.0f} to {coarse.fields['precipitation'].max():.0f} mm")
    
    # Generate a chunk with weather
    chunk_ref = ChunkRef(HexPosition(0, 0, 0), gen.chunk_rings)
    chunk = gen.generate_chunk(chunk_ref)
    chunk = gen.compute_chunk_weather(chunk_ref, chunk)
    
    print(f"\nChunk temp: {chunk.fields['temperature'].min():.1f} to {chunk.fields['temperature'].max():.1f}°C")
    print(f"Chunk precip: {chunk.fields['precipitation'].min():.0f} to {chunk.fields['precipitation'].max():.0f} mm")


In [ ]:
coarse.visualize_precipitation()
coarse.hexGrid.builder.show()

Because we run the model at the top we don't need it for the finer calculations and can just approximate

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Exactly. The expensive Smith-Barstad computation happens once on the coarse map where:
- Grid is small (~1000 hexes vs ~400 per chunk × many chunks)
- Regional patterns are what matter (wind direction, moisture sources, major orography)

For chunks, we just:
1. **Interpolate** temperature and precipitation from the nearest coarse hex
2. **Adjust temperature** using lapse rate for elevation difference
3. **Tweak precipitation** with simple multipliers for local slopes

This is fast (O(n) per chunk, no FFT) and accurate enough because the coarse map already captured the big picture - windward/leeward effects, distance from coast, latitude gradients.

The fine-scale "detail" is mostly just:
- Higher peaks → colder (lapse rate)
- Local valleys → slightly drier (rain shadow approximation)
- Variation within a coarse cell → interpolated noise

You could even skip the orographic adjustment entirely and just use elevation-based lapse rate - the visual difference would be subtle since the coarse already has the major patterns baked in.

Watersheds need to be computed after we have weather

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Right. The flow volume depends on precipitation. The order should be:

1. **Terrain** (elevation, plates)
2. **Weather** (precipitation, temperature)
3. **Watersheds** (drainage using precip as input)

For chunks:

```python
@patch
def compute_chunk_watersheds(self: ChunkedTerrainGenerator,
                              chunk: ChunkRef,
                              chunk_terrain: Terrain) -> list:
    """Compute watersheds for a chunk using precipitation data."""
    
    if 'precipitation' not in chunk_terrain.fields:
        raise ValueError("Compute weather first")
    
    # DrainageBasins uses elevation for flow direction
    # but precipitation determines flow volume
    basins = DrainageBasins(chunk_terrain)
    
    # Scale flow by local precipitation
    precip = chunk_terrain.fields['precipitation']
    for watershed in basins.sheds:
        flows = watershed.calculate_flow()
        for idx, flow in flows.items():
            # Flow = upstream area × precipitation
            flows[idx] = flow * precip[idx] / 1000  # mm → m³ish
    
    # Record border crossings for cross-chunk rivers
    borders = self.detect_chunk_borders(chunk_terrain, chunk)
    for border in borders:
        # Find which watershed this border hex belongs to
        border_local = HexPosition(border['border_hex_q'], 
                                    border['border_hex_r'],
                                    border['border_hex_s'])
        border_idx = chunk_terrain.hexGrid.hexposition_to_index(
            border_local, origin_index=chunk_terrain.hexGrid.midpoint)
        
        # Get accumulated flow at border
        for watershed in basins.sheds:
            flows = watershed.calculate_flow()
            if border_idx in flows:
                border['flow_volume'] = flows[border_idx]
                break
    
    # Save borders with flow data
    if borders and chunk.key in self.chunk_world_ids:
        self.storage.save_chunk_borders(self.chunk_world_ids[chunk.key], borders)
    
    return basins.sheds
```

Then for assembling rivers across chunks:

```python
@patch  
def assemble_cross_chunk_rivers(self: ChunkedTerrainGenerator) -> list:
    """Connect rivers that cross chunk boundaries."""
    
    # Query all borders that have downstream flow
    all_borders = list(self.storage.borders.rows)
    
    # Group by downstream chunk
    inflows = {}  # chunk_key -> list of (border_hex, flow_volume, source_chunk)
    for b in all_borders:
        downstream_key = (b['downstream_chunk_q'], b['downstream_chunk_r'], b['downstream_chunk_s'])
        if downstream_key not in inflows:
            inflows[downstream_key] = []
        inflows[downstream_key].append({
            'hex': HexPosition(b['border_hex_q'], b['border_hex_r'], b['border_hex_s']),
            'flow': b['flow_volume'],
            'source': (b['chunk_q'], b['chunk_r'], b['chunk_s'])
        })
    
    # Major rivers = high flow borders
    major_rivers = [b for b in all_borders if b['flow_volume'] > 100]
    
    return major_rivers, inflows
```

The coarse map could also have its own watershed computation to identify major drainage basins before chunking.

In [ ]:
??DrainageBasins

so I want to build a test of the watershed. The goal would be a large and small map and then we compute the drainagebasins for each level to see if they work.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
@patch
def test_multiscale_watersheds(self: GeoStorageDebugger):
    """Test watershed computation at coarse and fine scales."""
    
    # Set up chunked generator
    generator = ChunkedTerrainGenerator(
        storage=self.server,
        coarse_radius=50,
        fine_radius=10,
        chunk_rings=15,
        halo_rings=3
    )
    
    # Generate coarse world with weather
    bounds = MapRect(MapCord(0, 0), MapSize(1200, 900))
    coarse = generator.generate_coarse(
        bounds,
        num_plates=8,
        ocean_fraction=0.4,
        oceanic_sides=['W'],
        name="watershed_scale_test"
    )
    
    # Compute weather on coarse
    coarse.compute_distance_from_coast()
    coarse.compute_weather()
    
    print(f"Coarse map: {len(coarse.hexGrid.hexes)} hexes")
    print(f"  Precip range: {coarse.fields['precipitation'].min():.0f} - {coarse.fields['precipitation'].max():.0f} mm")
    
    # Compute watersheds on coarse
    coarse_basins = DrainageBasins(coarse)
    print(f"  Coarse watersheds: {len(coarse_basins.sheds)}")
    for i, ws in enumerate(coarse_basins.sheds[:5]):
        print(f"    WS {i}: {len(ws.region.hexes)} hexes, ocean={ws.is_ocean}")
    
    # Save coarse watersheds
    save_ws = self.server.save_watersheds(coarse, coarse_basins.sheds, generator.coarse_world_id)
    print(f"  Saved: {save_ws.context}")
    
    # Generate a detail chunk with weather
    origin_chunk = generator.world_to_chunk_ref(HexPosition(0, 0, 0))
    detail = generator.generate_chunk(origin_chunk, coarse)
    generator.compute_chunk_weather(detail)
    
    print(f"\nDetail chunk: {len(detail.hexGrid.hexes)} hexes")
    print(f"  Precip range: {detail.fields['precipitation'].min():.0f} - {detail.fields['precipitation'].max():.0f} mm")
    
    # Compute watersheds on detail
    detail_basins = DrainageBasins(detail)
    print(f"  Detail watersheds: {len(detail_basins.sheds)}")
    for i, ws in enumerate(detail_basins.sheds[:5]):
        print(f"    WS {i}: {len(ws.region.hexes)} hexes, ocean={ws.is_ocean}")
    
    # Save detail chunk and its watersheds
    chunk_id = generator.save_chunk(detail, origin_chunk)
    save_detail_ws = self.server.save_watersheds(detail, detail_basins.sheds, chunk_id)
    print(f"  Saved: {save_detail_ws.context}")
    
    # Visualize both
    print("\n--- Coarse Map ---")
    coarse.hexGrid.adjustRadius(8)
    coarse.colorMap()
    coarse.hexGrid.update()
    coarse.builder.adjust("rivers", coarse_basins.draw_watersheds(top_n=5))
    display(coarse.builder.show())
    
    print("\n--- Detail Chunk ---")
    detail.hexGrid.adjustRadius(8)
    detail.colorLevels = coarse.colorLevels
    detail.seaLevel = coarse.seaLevel
    detail.colorMap()
    detail.hexGrid.update()
    detail.builder.adjust("rivers", detail_basins.draw_watersheds(top_n=5))
    display(detail.builder.show())
    
    # Verify we can load them back
    load_coarse_ws = self.server.load_watersheds(generator.coarse_world_id, coarse)
    load_detail_ws = self.server.load_watersheds(chunk_id, detail)
    
    print(f"\n✓ Loaded coarse watersheds: {len(load_coarse_ws.data)}")
    print(f"✓ Loaded detail watersheds: {len(load_detail_ws.data)}")
```

Run it:

```python
with GeoStorageDebugger(keep_on_error=False) as dbg:
    dbg.test_multiscale_watersheds()
```

This tests:
1. Coarse watershed computation (global drainage patterns)
2. Fine chunk watershed computation (local detail)
3. Save/load roundtrip for both scales
4. Visual comparison of rivers at each scale

I would love to do it. and I would also like to make it so that we don't chunk ocean (that should just stay at elevation 0 everywhere - I don't want tiny island popping up'


##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good idea! Update `_add_detail` to skip ocean hexes entirely:

```python
def _add_detail(self, terrain: Terrain, chunk: ChunkRef):
    """Add deterministic noise for local variation - land only."""
    seed = hash(chunk.key) % (2**31)
    np.random.seed(seed)
    
    noise_scale = terrain.elevationDelta * 0.1
    
    # Only add noise to land hexes (elevation > 0)
    land_mask = terrain.elevations > 0
    terrain.elevations[land_mask] += np.random.uniform(
        -noise_scale, noise_scale, land_mask.sum()
    )
    
    # Ensure ocean stays exactly at coarse value (no tiny islands)
    ocean_mask = terrain.elevations <= 0
    terrain.elevations[ocean_mask] = np.minimum(terrain.elevations[ocean_mask], -10)
```

Or even simpler—clamp ocean to a fixed depth:

```python
def _add_detail(self, terrain: Terrain, chunk: ChunkRef):
    """Add deterministic noise for local variation - land only."""
    seed = hash(chunk.key) % (2**31)
    np.random.seed(seed)
    
    noise_scale = terrain.elevationDelta * 0.1
    land_mask = terrain.elevations > 0
    
    # Add noise only to land
    terrain.elevations[land_mask] += np.random.uniform(
        -noise_scale, noise_scale, land_mask.sum()
    )
    
    # Force ocean to stay ocean (prevent noise from creating islands)
    terrain.elevations[~land_mask] = -100
```

Now run the test:

```python
with GeoStorageDebugger(keep_on_error=False) as dbg:
    dbg.test_multiscale_watersheds()
```

Can you 

The real question is how to we handle algorithims in those purple triangles. like computing drainage or shortest path.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

The purple triangles are covered by **all 3 adjacent halos** but no core. For algorithms:

**Option 1: Fill from coarse (simplest)**
```python
def stitch_chunks_with_fill(gen, chunks):
    terrain = stitch_chunks(gen, chunks)
    
    # Fill invalid hexes from coarse
    for idx in terrain.hexGrid.invalid_region:
        world_pos = idx_to_world(terrain, idx)
        coarse_elev = gen.interpolate_from_coarse(world_pos)
        terrain.elevations[idx] = coarse_elev
    
    terrain.hexGrid.invalid_region = set()  # Now all valid
    return terrain
```

**Option 2: Query overlapping halos**
Since each of the 3 adjacent chunks has this position in its halo, pick one:
```python
def fill_from_halos(gen, terrain, chunks, idx):
    world_pos = idx_to_world(terrain, idx)
    
    # Find which chunks have this in their halo
    for chunk in chunks:
        if chunk.halo_contains(world_pos):
            return chunk.get_halo_elevation(world_pos)
    
    # Fallback to coarse
    return gen.interpolate_from_coarse(world_pos)
```

**Option 3: Design away the gaps**
If `halo_rings >= chunk_spacing / 3`, the halos cover everything and there are no gaps. But this wastes memory.

**For drainage specifically:**
Water flows downhill, so the triangles just need consistent elevation. Coarse interpolation is fine—rivers will route through.

**For pathfinding:**
Same logic—A* just needs traversable terrain. Fill from coarse, run A*.

The pragmatic answer: **fill from coarse** for the gaps, since those hexes are small transition zones and coarse data is "good enough" for continuity.

In [ ]:
# Modify the test to mark invalid hexes
with GeoStorageDebugger(keep_on_error=False) as debugger:
    gen = ChunkedTerrainGenerator(
        storage=debugger.server,
        coarse_rings=15, coarse_radius=50, fine_radius=10,
        chunk_rings=8, halo_rings=3
    )
    
    coarse = gen.generate_coarse(num_plates=8, ocean_fraction=0.35, oceanic_sides=['W'], seed=42)
    
    # Get processing order
    _, rivers, chunk_info = visualize_coarse_drainage(gen, coarse)
    flow_dir = coarse.flow_directions()
    deps = compute_chunk_dependencies(chunk_info, flow_dir, coarse)
    
    def topo_sort(deps):
        order, remaining, satisfied = [], set(deps.keys()), set()
        while remaining:
            ready = [c for c in remaining if deps[c] <= satisfied] or [min(remaining)]
            for c in sorted(ready):
                order.append(c)
                satisfied.add(c)
                remaining.remove(c)
        return order
    
    process_order = topo_sort(deps)
    all_chunk_refs = list(gen.chunk_refs(max_ring=2))
    
    # Generate chunks in order
    chunks = []
    for chunk_idx in process_order[:5]:
        chunk = gen.generate_chunk(all_chunk_refs[chunk_idx])
        for prev_chunk in chunks:
            chunk.copy_from(prev_chunk)
        chunks.append(chunk)
    
    chunk_set = ChunkSet(chunks, coarse, gen.scale)
    
    # Convert to terrain
    terrain = chunk_set.to_terrain(fill_gaps=True)
    terrain.hexGrid.adjustRadius(15)
    terrain.colorLevels = coarse.colorLevels
    terrain.seaLevel = coarse.seaLevel
    terrain.elevationDelta = coarse.elevationDelta
    
    for style in terrain.colorLevels:
        terrain.hexGrid.builder.add_style(style)
    terrain.hexGrid.builder.add_style(terrain.seaLevel)
    
    # === MARK INVALID HEXES ===
    # Get bounds from chunk_set positions
    positions = list(chunk_set.pos_to_chunk.keys())
    min_q = min(p.q for p in positions)
    min_r = min(p.r for p in positions)
    origin_world = HexPosition(min_q, min_r, -min_q - min_r)
    
    # Mark hexes that aren't covered by any chunk
    covered_indices = set()
    for world_pos in positions:
        local_pos = world_pos - origin_world
        grid_idx = terrain.hexGrid.hexposition_to_index(local_pos, origin_index=0)
        if 0 <= grid_idx < len(terrain.hexGrid.hexes):
            covered_indices.add(grid_idx)
    
    # All indices NOT in covered_indices are invalid
    all_indices = set(range(len(terrain.hexGrid.hexes)))
    terrain.hexGrid.invalidRegion = all_indices - covered_indices
    
    print(f"Total hexes: {len(all_indices)}")
    print(f"Covered by chunks: {len(covered_indices)}")
    print(f"Marked invalid: {len(terrain.hexGrid.invalidRegion)}")
    
    terrain.colorMap()
    terrain.hexGrid.update()
    
    # Add flow diagram
    flow_overlay = terrain.flow_diagram()
    terrain.hexGrid.builder.adjust("flow", flow_overlay)
    
    terrain.hexGrid.builder.show()
